# TrimFields dataset generation

This notebook generates the TrimFields dataset: trimmed spline designs on NURBS hosts with isogeometric reference
fields, for neural solvers on trimmed CAD faces. Each design is stored as a trimmed surface (periodic cubic
trimming loops in the parameter square of the host) together with a STEP file whose trimming edges are pcurves on
the host surface, the displacement of the crisp cut-cell evaluation, stress and energy samples along every loop,
and the design fields (crisp indicator and signed distance) on the conditioning grid.

| Split | Families | Designs |
|---|---|---|
| `id` | optimized designs on planar hosts rect (rectangle), sector (annular sector), plate (plate with a hole), arc (curved beam), taper (tapered beam), bolted (bolted plate) (2,000 each; every 10th design is test) | 12,000 |
| `ood_load` | optimized, rect to taper with the load centred in the held-out band of edge v1 (100 each) | 500 |
| `ood_host` | optimized, freeform (freeform patch), ell (L-shaped domain) (250 each) | 500 |
| `shell` | optimized shells cyl (cylinder) 400, saddle 300, dome 300, torus (torus sector) 200 (every 10th design is test) | 1,200 |
| `shell_ood` | optimized, hood (car-hood patch) | 100 |
| `rand` | random admissible loop sets on rect to bolted (1,500 each; every 10th design is test) | 9,000 |
| `rand_host` | random loop sets on freeform and ell (500 each) | 1,000 |
| `rand_shell` | random loop sets on cyl to torus (200 each; every 10th design is test) | 800 |
| `rand_shell_ood` | random loop sets on hood | 200 |

Optimized designs run isogeometric SIMP (Heaviside projection for shells), a crisp cut-cell evaluation of the
thresholded density, loop extraction and periodic B-spline fitting, the CAD-native refinement with exact design
gradients (JAX), an admissibility test and the STEP export, as in TrimTO. Random designs skip the optimization;
loop sets that cut the load path (compliance above 50 times that of the untrimmed host) are rejected, and the host
compliance is stored for every design as `J_host`. All boundary conditions are stated for the continuum (bearing tractions on loaded bolt holes, rollers only on
straight axis-aligned edges), and the designs of the test splits also store the compliance of a finer evaluation
(four times the elements per direction for planar faces, three times for shells). The counts above are the
configuration of the paper (about 280 core-hours on one core per worker, measured on the 570-design sample); `pipeline.CONFIG["counts"]` can be raised later without changing
any existing design, since every seed derives from the index.
Problems without an admissible trimmed design are resampled with a new seed; every choice is seeded, so the
dataset is reproducible.

**How to run.** Copy this notebook to the GPU machine (Linux x86_64, Python 3.10 to 3.12, NVIDIA driver 525 or
newer for the CUDA 12 build of JAX). The full run takes days, so a headless run inside `tmux` is recommended:

```bash
tmux new -s trimfields
jupyter nbconvert --to notebook --execute data_generate.ipynb \
    --output data_generate.run.ipynb --ExecutePreprocessor.timeout=-1
```

Running all cells in Jupyter works as well. The run is resumable: after an interruption, run the notebook again and
finished designs are skipped. Options are read from environment variables:

| Variable | Default | Meaning |
|---|---|---|
| `TRIMFIELDS_MODE` | `full` | `full` generates the dataset; `smoke` runs every stage on one small design per family |
| `TRIMFIELDS_OUT` | `trimfields_data` | output directory |
| `TRIMFIELDS_WORKERS` | `auto` | worker processes (default: CPU cores minus one, limited by memory) |
| `TRIMFIELDS_GPU_WORKERS` | `auto` | workers that run the JAX part on the GPU (default: from the benchmark in Section 6) |
| `TRIMFIELDS_INSTALL` | `1` | `0` skips the pip installation |

**Outputs** (under `TRIMFIELDS_OUT`): `designs/<split>/<family>/<idx>.npz` (problem, metrics, SIMP density, fitted,
refined and final loops, canonical tensor, refinement history, evaluation-mesh displacement `u_eval`, loop
samples `loop_*`, design field), `step/<split>/<family>/<idx>.step`, `consolidated/` (index tables, design
tensors, conditioning and design fields, splits, quality report, gallery), and `manifest.json`; the archives with
their SHA-256 sums go to `<TRIMFIELDS_OUT>_archives/`.

## 1. Parameters

In [ ]:
import copy
import importlib
import json
import os
import pathlib
import platform
import shutil
import subprocess
import sys
import time

MODE = os.environ.get("TRIMFIELDS_MODE", "full").lower()              # "full" or "smoke"
OUT_DIR = os.path.abspath(os.environ.get("TRIMFIELDS_OUT", "trimfields_data"))
WORKERS = os.environ.get("TRIMFIELDS_WORKERS", "auto")                 # total worker processes
GPU_WORKERS = os.environ.get("TRIMFIELDS_GPU_WORKERS", "auto")         # workers running the JAX part on the GPU
INSTALL = os.environ.get("TRIMFIELDS_INSTALL", "1") == "1"
WORK = pathlib.Path.cwd()
assert MODE in ("full", "smoke"), f"TRIMFIELDS_MODE must be 'full' or 'smoke', not {MODE!r}"
T_START = time.monotonic()
print(f"mode {MODE} | output {OUT_DIR} | working directory {WORK} | Python {sys.version.split()[0]}")

## 2. Dependencies

Python packages are installed into the running interpreter. JAX is installed with its CUDA 12 wheels when an
NVIDIA GPU is visible, and PARDISO (`pypardiso`, Intel MKL) replaces SuperLU as the sparse direct solver on x86_64.
The second cell imports every package in a fresh process and reinstalls any compiled package that was built
against another NumPy version (a common problem when a system Python mixes packages from several sources).

In [ ]:
ENV_HELP = """
Use a dedicated Python environment for this notebook, for example
  conda create -n trimfields python=3.11 -y && conda activate trimfields
or (Ubuntu: sudo apt install python3-venv)
  python3 -m venv ~/trimfields-env && source ~/trimfields-env/bin/activate
then
  pip install jupyter nbconvert ipykernel
and start Jupyter (or jupyter nbconvert) from that environment."""


def sh(cmd, check=True, **kw):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, **kw)


def pip_install(*pkgs, check=True):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed", *pkgs]
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-3000:])
    if r.returncode != 0:
        if "externally-managed-environment" in out:
            raise RuntimeError("pip may not install into this system Python." + ENV_HELP)
        if check:
            raise subprocess.CalledProcessError(r.returncode, cmd)
    return r.returncode == 0


def nvidia_gpus():
    if shutil.which("nvidia-smi") is None:
        return []
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                       capture_output=True, text=True)
    return [l.strip() for l in r.stdout.splitlines() if l.strip()] if r.returncode == 0 else []


def dist_versions(names):
    import importlib.metadata as md
    out = {}
    for n in names:
        try:
            out[n] = md.version(n)
        except md.PackageNotFoundError:
            out[n] = None
    return out


GPUS = nvidia_gpus()
# Minimum versions are the first releases built against NumPy 2 (older compiled wheels fail to import under it).
REQUIREMENTS = ["numpy>=1.26", "scipy>=1.13", "matplotlib>=3.9", "contourpy>=1.2.1", "shapely>=2.0.4",
                "scikit-image>=0.24", "pandas>=2.2.2", "tqdm>=4.66", "cadquery-ocp>=7.7"]
TRACKED = ["numpy", "scipy", "matplotlib", "contourpy", "shapely", "scikit-image", "pandas", "jax", "jaxlib"]
BEFORE = dist_versions(TRACKED)
if INSTALL:
    assert sys.version_info[:2] >= (3, 10), "Python 3.10 or newer is required"
    pip_install("--upgrade", "pip")
    pip_install(*REQUIREMENTS)
    pip_install("jax[cuda12]>=0.4.30" if GPUS else "jax>=0.4.30")
    if platform.machine() in ("x86_64", "AMD64"):
        if not pip_install("pypardiso>=0.4.6", check=False):
            print("pypardiso could not be installed: SuperLU is used (about 3 times slower solves)")
print("Python:", sys.executable, sys.version.split()[0], "| GPUs:", GPUS or "none")

In [ ]:
# Every module the pipeline needs must import in a fresh process. A compiled package built against another NumPy
# ABI (for example an old matplotlib in ~/.local next to NumPy 2) is reinstalled from its current wheel.
IMPORTS = {"numpy": "numpy", "scipy": "scipy.sparse.linalg,scipy.spatial", "matplotlib": "matplotlib.path",
           "contourpy": "contourpy", "shapely": "shapely", "scikit-image": "skimage.measure", "pandas": "pandas",
           "tqdm": "tqdm", "jax": "jax.numpy"}
if platform.machine() in ("x86_64", "AMD64"):
    IMPORTS["pypardiso"] = "pypardiso"
PROBE = r"""
import importlib, json, os, sys, traceback
try:
    for m in sys.argv[1].split(","):
        importlib.import_module(m)
    print("IMPORT_OK")
except BaseException as e:
    files = [f.filename for f in traceback.extract_tb(e.__traceback__)]
    pk = []
    for f in files:
        for key in ("site-packages", "dist-packages"):
            if os.sep + key + os.sep in f:
                pk.append(f.split(os.sep + key + os.sep)[-1].split(os.sep)[0])
    print("IMPORT_FAIL " + json.dumps({"error": f"{type(e).__name__}: {e}", "packages": pk}))
"""


def import_failures():
    import re
    fails = {}
    for dist, mods in IMPORTS.items():
        try:
            r = subprocess.run([sys.executable, "-c", PROBE, mods], capture_output=True, text=True,
                               env=dict(os.environ, JAX_PLATFORMS="cpu"), timeout=600)
        except subprocess.TimeoutExpired:
            fails[dist] = {"error": "import timed out", "packages": [], "abi": False}
            continue
        abi = "_ARRAY_API not found" in r.stderr or "compiled using NumPy 1.x" in r.stderr
        line = [l for l in r.stdout.splitlines() if l.startswith("IMPORT_")]
        if line and line[-1].startswith("IMPORT_FAIL"):
            info = json.loads(line[-1][len("IMPORT_FAIL "):])
        elif not line:
            info = {"error": (r.stderr.strip().splitlines() or ["no output"])[-1], "packages": []}
        elif abi:
            info = {"error": "built against another NumPy version", "packages": []}
        else:
            continue
        if abi:                         # NumPy prints the importing stack: its innermost package is the culprit
            frames = re.findall(r"(?:site|dist)-packages[/\\\\]([A-Za-z0-9_.]+)", r.stderr)
            info["packages"] = info["packages"] + frames[-1:]
        info["abi"] = abi
        fails[dist] = info
    return fails


def owners(pkg_dirs):
    """Distributions that own the given top-level package directories."""
    import importlib.metadata as md
    top = md.packages_distributions()
    return {d for p in pkg_dirs for d in top.get(p[:-3] if p.endswith(".py") else p, [])}


for _ in range(3):
    FAILS = import_failures()
    if not FAILS:
        break
    for dist, info in FAILS.items():
        print(f"import of {dist} failed: {info['error'][:300]}")
    if not INSTALL:
        break
    fix = (set(FAILS) | owners(p for info in FAILS.values() for p in info["packages"])) - {"numpy"}
    print("reinstalling from current wheels:", sorted(fix))
    pip_install("--upgrade", "--force-reinstall", "--no-deps", *sorted(fix), check=False)
if FAILS:
    raise RuntimeError(f"these packages cannot be imported: {sorted(FAILS)}." + ENV_HELP)
AFTER = dist_versions(TRACKED)
changed = [d for d in TRACKED if BEFORE[d] != AFTER[d]]
stale = [d for d in changed if d.replace("scikit-image", "skimage") in sys.modules]
if stale:
    raise RuntimeError(f"{stale} changed after this kernel had imported them: restart the kernel and run all cells "
                       "again (finished work is kept).")
print("all imports ok:", ", ".join(f"{d} {v}" for d, v in AFTER.items() if v))

In [ ]:
# OpenCASCADE (cadquery-ocp) links libGL and libX11, which headless servers often lack.
def ocp_error():
    r = subprocess.run([sys.executable, "-c", "import OCP.BRepAlgoAPI, OCP.STEPControl"], capture_output=True,
                       text=True)
    return None if r.returncode == 0 else ((r.stderr.strip().splitlines() or ["unknown error"])[-1])


err = ocp_error()
if err:
    print("OCP import failed:", err)
    APT = ["apt-get", "install", "-y", "--no-install-recommends", "libgl1", "libx11-6", "libxcb1", "libexpat1"]
    if shutil.which("apt-get"):
        if os.geteuid() == 0:
            sh(["apt-get", "update"], check=False)
            sh(APT, check=False)
        elif shutil.which("sudo") and subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
            sh(["sudo", "-n", "apt-get", "update"], check=False)
            sh(["sudo", "-n"] + APT, check=False)
    err = ocp_error()
    if err:
        raise RuntimeError("OpenCASCADE cannot be imported: " + err + "\nInstall the system libraries, e.g.\n"
                           "  sudo apt-get install -y libgl1 libx11-6 libxcb1 libexpat1\nand run this cell again.")
print("OpenCASCADE (OCP) import: ok")

## 3. Machine and environment

In [ ]:
import numpy as np


def mem_gb(key="MemAvailable"):
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith(key + ":"):
                return int(line.split()[1]) / 2**20
    return float("nan")


def versions():
    import importlib.metadata as md
    out = {}
    for p in ("numpy", "scipy", "jax", "jaxlib", "shapely", "scikit-image", "matplotlib", "pandas", "cadquery-ocp",
              "pypardiso", "mkl", "tqdm"):
        try:
            out[p] = md.version(p)
        except md.PackageNotFoundError:
            out[p] = None
    return out


CPUS = len(os.sched_getaffinity(0))
os.makedirs(OUT_DIR, exist_ok=True)
MACHINE = {"host": platform.node(), "platform": platform.platform(), "cpu": platform.processor() or platform.machine(),
           "cores": CPUS, "mem_total_gb": round(mem_gb("MemTotal"), 1), "mem_available_gb": round(mem_gb(), 1),
           "disk_free_gb": round(shutil.disk_usage(OUT_DIR).free / 2**30, 1), "gpus": GPUS, "python": sys.version,
           "versions": versions()}
print(json.dumps(MACHINE, indent=1))


def jax_probe(backend):
    code_ = ("import jax, jax.numpy as jnp; jax.config.update('jax_enable_x64', True); "
             "x = jnp.ones((256, 256)); print(jax.default_backend(), jax.devices()[0], float((x @ x).sum()))")
    env = dict(os.environ, JAX_PLATFORMS=backend, XLA_PYTHON_CLIENT_PREALLOCATE="false")
    try:
        r = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True, env=env, timeout=600)
    except subprocess.TimeoutExpired:
        return False, "no answer within 10 minutes"
    return r.returncode == 0 and r.stdout.strip().endswith("16777216.0"), (r.stdout + r.stderr).strip()[-400:]


JAX_GPU_OK = False
if GPUS:
    JAX_GPU_OK, msg = jax_probe("cuda")
    print("JAX on the GPU:", "ok" if JAX_GPU_OK else "FAILED, the CPU is used\n" + msg)
print("JAX on the CPU:", "ok" if jax_probe("cpu")[0] else "FAILED")
if MACHINE["disk_free_gb"] < 15 and MODE == "full":
    print("WARNING: the full dataset with STEP files and archives needs about 15 GB of free disk space")

## 4. The `trimfields` package

The next cells write the package next to this notebook (`./trimfields/`). Worker processes import it from there.

In [ ]:
os.makedirs("trimfields", exist_ok=True)

In [ ]:
%%writefile trimfields/__init__.py
"""TrimFields: trimmed spline designs with isogeometric reference fields for neural solvers on trimmed CAD faces."""
__version__ = "2.0.1"


In [ ]:
%%writefile trimfields/basis.py
"""Vectorized B-spline / NURBS basis functions with derivatives up to second order (Piegl & Tiller A2.3)."""
import numpy as np


def _sdiv(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def find_span(U, p, u):
    U = np.asarray(U, float)
    n = len(U) - p - 2
    return np.clip(np.searchsorted(U, u, side="right") - 1, p, n)


def ders_basis(U, p, u, nd=1):
    """Non-zero basis functions and derivatives. Returns span (m,), D (nd+1, m, p+1)."""
    U = np.asarray(U, float)
    u = np.atleast_1d(np.asarray(u, float))
    m = u.size
    span = find_span(U, p, u)
    ndu = np.zeros((p + 1, p + 1, m))
    ndu[0, 0] = 1.0
    left = np.zeros((p + 1, m))
    right = np.zeros((p + 1, m))
    for j in range(1, p + 1):
        left[j] = u - U[span + 1 - j]
        right[j] = U[span + j] - u
        saved = np.zeros(m)
        for r in range(j):
            ndu[j, r] = right[r + 1] + left[j - r]
            temp = _sdiv(ndu[r, j - 1], ndu[j, r])
            ndu[r, j] = saved + right[r + 1] * temp
            saved = left[j - r] * temp
        ndu[j, j] = saved
    D = np.zeros((nd + 1, p + 1, m))
    for j in range(p + 1):
        D[0, j] = ndu[j, p]
    for r in range(p + 1):
        a = np.zeros((2, p + 1, m))
        a[0, 0] = 1.0
        s1, s2 = 0, 1
        for k in range(1, nd + 1):
            d = np.zeros(m)
            rk, pk = r - k, p - k
            if r >= k:
                a[s2, 0] = _sdiv(a[s1, 0], ndu[pk + 1, rk])
                d = a[s2, 0] * ndu[rk, pk]
            j1 = 1 if rk >= -1 else -rk
            j2 = k - 1 if r - 1 <= pk else p - r
            for j in range(j1, j2 + 1):
                a[s2, j] = _sdiv(a[s1, j] - a[s1, j - 1], ndu[pk + 1, rk + j])
                d = d + a[s2, j] * ndu[rk + j, pk]
            if r <= pk:
                a[s2, k] = _sdiv(-a[s1, k - 1], ndu[pk + 1, r])
                d = d + a[s2, k] * ndu[r, pk]
            D[k, r] = d
            s1, s2 = s2, s1
    fac = p
    for k in range(1, nd + 1):
        D[k] *= fac
        fac *= p - k
    return span, np.transpose(D, (0, 2, 1))


def nurbs2d(Uu, Uv, W, u, v, nd=1, p=2):
    """Rational tensor-product basis at paired points (u, v). Returns dict with ien (m, nen) and R, Ru, Rv
    (and Ruu, Ruv, Rvv if nd == 2). Control-point index = iu * ncv + iv."""
    u = np.atleast_1d(np.asarray(u, float))
    v = np.atleast_1d(np.asarray(v, float))
    m = u.size
    su, Du = ders_basis(Uu, p, u, max(nd, 1))
    sv, Dv = ders_basis(Uv, p, v, max(nd, 1))
    ncv = W.shape[1]
    iu = su[:, None] - p + np.arange(p + 1)[None, :]
    iv = sv[:, None] - p + np.arange(p + 1)[None, :]
    ien = (iu[:, :, None] * ncv + iv[:, None, :]).reshape(m, -1)
    w = W.ravel()[ien]

    def tp(a, b):
        return (Du[a][:, :, None] * Dv[b][:, None, :]).reshape(m, -1)

    wN, wNu, wNv = w * tp(0, 0), w * tp(1, 0), w * tp(0, 1)
    Wt, Wu, Wv = wN.sum(1)[:, None], wNu.sum(1)[:, None], wNv.sum(1)[:, None]
    R = wN / Wt
    Ru = (wNu - R * Wu) / Wt
    Rv = (wNv - R * Wv) / Wt
    out = {"ien": ien, "R": R, "Ru": Ru, "Rv": Rv}
    if nd >= 2:
        wNuu, wNuv, wNvv = w * tp(2, 0), w * tp(1, 1), w * tp(0, 2)
        Wuu, Wuv, Wvv = wNuu.sum(1)[:, None], wNuv.sum(1)[:, None], wNvv.sum(1)[:, None]
        out["Ruu"] = (wNuu - 2 * Ru * Wu - R * Wuu) / Wt
        out["Ruv"] = (wNuv - Ru * Wv - Rv * Wu - R * Wuv) / Wt
        out["Rvv"] = (wNvv - 2 * Rv * Wv - R * Wvv) / Wt
    return out


def gauss_grid(Uu, Uv, nq):
    """Tensor Gauss points on every knot span. Points of one element are contiguous (element e = i*nelv + j).
    Returns u, v, weight (parametric), element index, nelu, nelv."""
    ku, kv = np.unique(np.asarray(Uu, float)), np.unique(np.asarray(Uv, float))
    x, w = np.polynomial.legendre.leggauss(nq)
    du, dv = np.diff(ku), np.diff(kv)
    mu = 0.5 * (ku[1:] + ku[:-1])
    mv = 0.5 * (kv[1:] + kv[:-1])
    nelu, nelv = len(du), len(dv)
    uu = mu[:, None] + 0.5 * du[:, None] * x[None, :]               # (nelu, nq)
    vv = mv[:, None] + 0.5 * dv[:, None] * x[None, :]               # (nelv, nq)
    wu = 0.5 * du[:, None] * w[None, :]
    wv = 0.5 * dv[:, None] * w[None, :]
    U = np.broadcast_to(uu[:, None, :, None], (nelu, nelv, nq, nq))
    V = np.broadcast_to(vv[None, :, None, :], (nelu, nelv, nq, nq))
    Wq = wu[:, None, :, None] * wv[None, :, None, :]
    e = np.broadcast_to(np.arange(nelu * nelv).reshape(nelu, nelv)[:, :, None, None], (nelu, nelv, nq, nq))
    return U.ravel(), V.ravel(), Wq.ravel(), e.ravel(), nelu, nelv


def greville(U, p=2):
    U = np.asarray(U, float)
    n = len(U) - p - 1
    return np.array([U[i + 1:i + p + 1].mean() for i in range(n)])


def periodic_basis(n, t, der=0):
    """(M, n) matrix: c(t) = B @ C for a uniform periodic cubic B-spline with n control points, t in [0, 1)."""
    t = np.asarray(t, float) % 1.0
    x = t * n
    j = np.minimum(np.floor(x).astype(int), n - 1)
    s = x - j
    if der == 0:
        b = [(1 - s) ** 3 / 6, (3 * s**3 - 6 * s**2 + 4) / 6, (-3 * s**3 + 3 * s**2 + 3 * s + 1) / 6, s**3 / 6]
    else:
        b = [-n * (1 - s) ** 2 / 2, n * (9 * s**2 - 12 * s) / 6, n * (-9 * s**2 + 6 * s + 3) / 6, n * s**2 / 2]
    B = np.zeros((len(t), n))
    rows = np.arange(len(t))
    for k in range(4):
        np.add.at(B, (rows, (j + k) % n), b[k])
    return B


In [ ]:
%%writefile trimfields/hosts.py
"""Spline host patches: exact coarse control nets for every host family, knot refinement and geometric queries.

All hosts are biquadratic NURBS patches on the parameter square. Control-point index = iu * ncv + iv.
Planar families (rect-ell) map to R^2, shell families (cyl-hood) to R^3.
"""
import math

import numpy as np

from .basis import gauss_grid, greville, nurbs2d

PLANAR = ("rect", "sector", "plate", "arc", "taper", "bolted", "freeform", "ell")
SHELL = ("cyl", "saddle", "dome", "torus", "hood")
FAMILY_NAMES = {
    "rect": "rectangle", "sector": "annular sector", "plate": "plate with a hole", "arc": "curved beam",
    "taper": "tapered patch", "bolted": "rectangle with bolt holes", "freeform": "freeform patch", "ell": "L-shaped host",
    "cyl": "cylindrical panel", "saddle": "saddle", "dome": "dome", "torus": "torus sector", "hood": "freeform hood",
}


# ------------------------------------------------------------------------------------------------ knot insertion
def insert_knot(U, Pw, t, axis):
    """Boehm insertion of one knot t into knot vector U along axis (0: u, 1: v) of the homogeneous net Pw."""
    p = 2
    U = np.asarray(U, float)
    k = int(np.searchsorted(U, t, side="right") - 1)
    Q = np.moveaxis(Pw, axis, 0)
    n = Q.shape[0]
    new = np.empty((n + 1,) + Q.shape[1:])
    new[: k - p + 1] = Q[: k - p + 1]
    for i in range(k - p + 1, k + 1):
        a = (t - U[i]) / (U[i + p] - U[i])
        new[i] = a * Q[i] + (1 - a) * Q[i - 1]
    new[k + 1:] = Q[k:]
    return np.insert(U, k + 1, t), np.moveaxis(new, 0, axis)


def refine(Uu, Uv, Pw, nelu, nelv):
    """Uniform h-refinement to nelu x nelv elements; existing knots are kept (single insertion keeps C1)."""
    for axis, (U, n) in enumerate(((Uu, nelu), (Uv, nelv))):
        have = {round(x, 12) for x in U}
        Uc = np.asarray(U, float)
        for k in range(1, n):
            t = k / n
            if round(t, 12) in have:
                continue
            Uc, Pw = insert_knot(Uc, Pw, t, axis)
        if axis == 0:
            Uu = Uc
        else:
            Uv = Uc
    return Uu, Uv, Pw


def arc3(angle, radius=1.0, start=0.0):
    """Rational quadratic arc (single span, angle < pi): 3 control points (2D) and weights."""
    h = angle / 2
    a0, a2 = start, start + angle
    P0 = radius * np.array([math.cos(a0), math.sin(a0)])
    P2 = radius * np.array([math.cos(a2), math.sin(a2)])
    P1 = radius / math.cos(h) * np.array([math.cos(start + h), math.sin(start + h)])
    return np.array([P0, P1, P2]), np.array([1.0, math.cos(h), 1.0])


# ------------------------------------------------------------------------------------------------ Host
class Host:
    """Biquadratic NURBS host. `spec` holds the coarse exact net; `build(nelu, nelv)` refines it."""

    def __init__(self, family, params, Uu, Uv, P, W, knot_div=(1, 1)):
        self.family, self.params = family, dict(params)
        self.Uu0, self.Uv0 = np.asarray(Uu, float), np.asarray(Uv, float)
        self.P0, self.W0 = np.asarray(P, float), np.asarray(W, float)
        self.dim = self.P0.shape[-1]
        self.knot_div = knot_div
        self.build(2, 2)
        self.Lu, self.Lv, self.area = self._lengths()

    # -- construction ---------------------------------------------------------------------------------------------
    def build(self, nelu, nelv):
        du, dv = self.knot_div
        nelu = max(du, int(round(nelu / du)) * du)
        nelv = max(dv, int(round(nelv / dv)) * dv)
        Pw = np.concatenate([self.P0 * self.W0[..., None], self.W0[..., None]], -1)
        Uu, Uv, Pw = refine(self.Uu0, self.Uv0, Pw, nelu, nelv)
        self.Uu, self.Uv = Uu, Uv
        self.W = Pw[..., -1]
        self.P = (Pw[..., :-1] / self.W[..., None]).reshape(-1, self.dim)
        self.ncu, self.ncv = self.W.shape
        self.nelu, self.nelv = len(np.unique(Uu)) - 1, len(np.unique(Uv)) - 1
        return self

    def mesh_for(self, n_elements):
        """Element counts with about n_elements elements and elements of roughly unit aspect ratio."""
        r = self.Lu / self.Lv
        nelu = max(4, int(round(math.sqrt(n_elements * r))))
        nelv = max(4, int(round(n_elements / nelu)))
        return nelu, nelv

    # -- evaluation -----------------------------------------------------------------------------------------------
    def basis(self, u, v, nd=1):
        return nurbs2d(self.Uu, self.Uv, self.W, np.clip(u, 0, 1), np.clip(v, 0, 1), nd=nd)

    def eval(self, u, v):
        b = self.basis(u, v, nd=0)
        return np.einsum("mk,mkd->md", b["R"], self.P[b["ien"]])

    def jac(self, u, v):
        """Points X (m, dim) and Jacobian J (m, dim, 2)."""
        b = self.basis(u, v, nd=1)
        Pe = self.P[b["ien"]]
        X = np.einsum("mk,mkd->md", b["R"], Pe)
        J = np.stack([np.einsum("mk,mkd->md", b["Ru"], Pe), np.einsum("mk,mkd->md", b["Rv"], Pe)], -1)
        return X, J

    def metric(self, u, v):
        _, J = self.jac(u, v)
        return np.einsum("mdi,mdj->mij", J, J)

    def area_element(self, u, v):
        return np.sqrt(np.maximum(np.linalg.det(self.metric(u, v)), 0.0))

    def curvature(self, u, v):
        """Second fundamental form b_ab of a shell host (m, 2, 2); zeros for planar hosts."""
        m = np.size(u)
        if self.dim == 2:
            return np.zeros((m, 2, 2))
        b = self.basis(u, v, nd=2)
        Pe = self.P[b["ien"]]
        d = {k: np.einsum("mk,mkd->md", b[k], Pe) for k in ("Ru", "Rv", "Ruu", "Ruv", "Rvv")}
        n = np.cross(d["Ru"], d["Rv"])
        n /= np.linalg.norm(n, axis=1, keepdims=True)
        B = np.zeros((m, 2, 2))
        B[:, 0, 0] = np.sum(d["Ruu"] * n, 1)
        B[:, 0, 1] = B[:, 1, 0] = np.sum(d["Ruv"] * n, 1)
        B[:, 1, 1] = np.sum(d["Rvv"] * n, 1)
        return B

    def _lengths(self, n=48):
        s = (np.arange(n) + 0.5) / n
        U, V = np.meshgrid(s, s, indexing="ij")
        _, J = self.jac(U.ravel(), V.ravel())
        Lu = float(np.mean(np.linalg.norm(J[:, :, 0], axis=1)))
        Lv = float(np.mean(np.linalg.norm(J[:, :, 1], axis=1)))
        u, v, w, _, _, _ = gauss_grid(self.Uu, self.Uv, 3)
        area = float(np.sum(self.area_element(u, v) * w))
        return Lu, Lv, area

    def min_jacobian_ratio(self, n=64):
        s = (np.arange(n) + 0.5) / n
        U, V = np.meshgrid(s, s, indexing="ij")
        dA = self.area_element(U.ravel(), V.ravel())
        return float(dA.min() / dA.mean())

    def greville_uv(self):
        return greville(self.Uu), greville(self.Uv)

    # -- serialization ----------------------------------------------------------------------------------------------
    def to_dict(self):
        return {"family": self.family, "params": self.params, "Uu": self.Uu0.tolist(), "Uv": self.Uv0.tolist(),
                "P": self.P0.tolist(), "W": self.W0.tolist(), "knot_div": list(self.knot_div)}

    @staticmethod
    def from_dict(d):
        return Host(d["family"], d["params"], d["Uu"], d["Uv"], d["P"], d["W"], tuple(d["knot_div"]))


BEZ = [0, 0, 0, 1, 1, 1]


def _net(points, weights=None):
    P = np.asarray(points, float)
    W = np.ones(P.shape[:2]) if weights is None else np.asarray(weights, float)
    return P, W


# ------------------------------------------------------------------------------------------------ planar families
def rectangle(L, H):
    P = [[(x, y) for y in (0.0, H / 2, H)] for x in (0.0, L / 2, L)]
    return _net(P)


def make_host(family, rng):
    """Sample a host of the given family. Returns Host (parameters drawn from the family ranges)."""
    if family in ("rect", "bolted", "ell"):
        if family == "ell":
            L = float(rng.uniform(1.0, 1.5))
            H = L
        else:
            H = 1.0
            L = float(rng.uniform(1.0, 4.0) if family == "rect" else rng.uniform(1.5, 3.0))
        P, W = rectangle(L, H)
        return Host(family, {"L": L, "H": H}, BEZ, BEZ, P, W)
    if family in ("sector", "arc"):
        if family == "sector":
            ang = math.radians(rng.uniform(45, 150))
            r_in = 1.0
            r_out = float(rng.uniform(1.5, 3.0))
        else:
            kappa = float(rng.uniform(0.2, 1.0))
            width = float(rng.uniform(0.2, 0.5))
            rc = 1.0 / kappa
            r_in, r_out = rc - width / 2, rc + width / 2
            ang = min(math.radians(150), 3.0 / rc)
        A, w = arc3(ang)
        radii = (r_in, 0.5 * (r_in + r_out), r_out)
        P = [[tuple(r * A[i]) for r in radii] for i in range(3)]
        W = [[w[i]] * 3 for i in range(3)]
        return Host(family, {"angle_deg": math.degrees(ang), "r_in": r_in, "r_out": r_out}, BEZ, BEZ, P, W)
    if family == "plate":
        R = 1.0
        L = float(R / rng.uniform(0.2, 0.5))
        a, w = math.sqrt(2) - 1, 0.5 * (1 + math.sqrt(2) / 2)
        hole = [(R, 0), (R, a * R), (a * R, R), (0, R)]
        outer = [(L, 0), (L, L), (L, L), (0, L)]
        mid = [tuple(0.5 * (np.array(h) + np.array(o))) for h, o in zip(hole, outer)]
        P = [[hole[i], mid[i], outer[i]] for i in range(4)]
        W = [[(1.0, w, w, 1.0)[i], 1.0, 1.0] for i in range(4)]
        return Host(family, {"R_hole": R, "L": L}, [0, 0, 0, 0.5, 1, 1, 1], BEZ, P, W, knot_div=(2, 1))
    if family == "taper":
        L, H = float(rng.uniform(1.5, 3.0)), 1.0
        taper = float(rng.uniform(0.3, 1.0))
        top = lambda x: H * (1 - (1 - taper) * x / L)  # noqa: E731
        off = lambda x: 0.5 * (H - top(x))              # noqa: E731
        P = [[(x, off(x) + t * top(x)) for t in (0.0, 0.5, 1.0)] for x in (0.0, L / 2, L)]
        return Host(family, {"L": L, "H": H, "taper": taper}, BEZ, BEZ, *_net(P))
    if family == "freeform":
        for _ in range(50):
            xs, ts = np.linspace(0.0, 3.0, 5), np.linspace(-1.0, 1.0, 4)
            arch, taper = rng.uniform(0.0, 0.5), rng.uniform(0.0, 0.35)
            P = []
            for i, x in enumerate(xs):
                half, centre = 0.75 - taper * x / 3.0, arch * math.sin(math.pi * x / 3.0)
                row = []
                for j, t in enumerate(ts):
                    dx, dy = rng.normal(0.0, 0.08, 2) if (0 < i < 4 and 0 < j < 3) else (0.0, 0.0)
                    row.append((x + dx, centre + half * t + dy))
                P.append(row)
            h = Host(family, {"arch": float(arch), "taper": float(taper)}, [0, 0, 0, 1 / 3, 2 / 3, 1, 1, 1],
                     [0, 0, 0, 0.5, 1, 1, 1], *_net(P), knot_div=(3, 2))
            if h.min_jacobian_ratio() > 0.2:
                return h
        raise RuntimeError("no valid freeform host")
    return make_shell_host(family, rng)


# ------------------------------------------------------------------------------------------------ shell families
def make_shell_host(family, rng):
    if family == "cyl":
        R, ang, Lc = float(rng.uniform(0.8, 2.0)), math.radians(rng.uniform(45, 150)), float(rng.uniform(1.0, 3.0))
        A, w = arc3(ang, R, start=math.pi / 2 - ang / 2)
        P = [[(A[i, 0], y, A[i, 1]) for y in (0.0, Lc / 2, Lc)] for i in range(3)]
        W = [[w[i]] * 3 for i in range(3)]
        return Host(family, {"R": R, "angle_deg": math.degrees(ang), "length": Lc}, BEZ, BEZ, P, W)
    if family in ("saddle", "dome"):
        a = float(rng.uniform(0.75, 1.5))
        c = float(rng.uniform(0.2, 1.0))
        b = [a * a, -a * a, a * a]
        xs = (-a, 0.0, a)
        if family == "saddle":
            P = [[(xs[i], xs[j], 0.5 * c / a * (b[i] - b[j])) for j in range(3)] for i in range(3)]
        else:
            P = [[(xs[i], xs[j], 0.5 * c / a * (2 * a * a - b[i] - b[j])) for j in range(3)] for i in range(3)]
        return Host(family, {"a": a, "c": c}, BEZ, BEZ, *_net(P))
    if family == "torus":
        Rm, r = float(rng.uniform(1.5, 3.0)), float(rng.uniform(0.4, 0.8))
        au, av = math.radians(rng.uniform(45, 120)), math.radians(rng.uniform(60, 150))
        C, wu = arc3(au)
        D, wv = arc3(av, start=-av / 2)
        P = [[((Rm + r * D[j, 0]) * C[i, 0], (Rm + r * D[j, 0]) * C[i, 1], r * D[j, 1]) for j in range(3)]
             for i in range(3)]
        W = [[wu[i] * wv[j] for j in range(3)] for i in range(3)]
        return Host(family, {"R_major": Rm, "r_minor": r, "angle_u_deg": math.degrees(au),
                             "angle_v_deg": math.degrees(av)}, BEZ, BEZ, P, W)
    if family == "hood":
        for _ in range(50):
            xs, ys = np.linspace(0, 2.0, 6), np.linspace(0, 1.5, 5)
            X, Y = np.meshgrid(xs, ys, indexing="ij")
            Z = 0.25 * np.exp(-((X - rng.uniform(0.6, 1.4)) ** 2 + (Y - 0.75) ** 2) / rng.uniform(0.3, 0.8))
            Z += 0.15 * (X / 2.0) ** 2 * rng.uniform(-1, 1) + rng.normal(0, 0.03, Z.shape)
            X[1:-1, 1:-1] += rng.normal(0, 0.03, (4, 3))
            P = np.stack([X, Y, Z], -1)
            h = Host(family, {"seed_bump": float(Z.max())}, [0, 0, 0, 1 / 4, 1 / 2, 3 / 4, 1, 1, 1],
                     [0, 0, 0, 1 / 3, 2 / 3, 1, 1, 1], *_net(P), knot_div=(4, 3))
            if h.min_jacobian_ratio() > 0.2:
                return h
        raise RuntimeError("no valid hood host")
    raise ValueError(family)


In [ ]:
%%writefile trimfields/mechanics.py
"""Isogeometric linear elasticity on a host patch: plane stress (planar hosts) and Kirchhoff-Love shells.

A Model is built on a point set (u, v, weight, element) that is either a tensor Gauss grid (optimization) or an
adaptive cut-cell set (evaluation). The Young's modulus is given per point, which is all the design influences.
"""
import os

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu

from .basis import gauss_grid

try:  # optional direct solver (Intel MKL PARDISO); TRIMFIELDS_SOLVER = auto | pardiso | superlu
    import pypardiso  # noqa: F401
    HAVE_PARDISO = True
except Exception:  # noqa: BLE001
    HAVE_PARDISO = False
SOLVER = os.environ.get("TRIMFIELDS_SOLVER", "auto").lower()
USE_PARDISO = HAVE_PARDISO and SOLVER in ("auto", "pardiso")

NU = 0.3


def plane_stress_D(nu=NU):
    f = 1.0 / (1 - nu**2)
    return f * np.array([[1, nu, 0], [nu, 1, 0], [0, 0, (1 - nu) / 2]])


class Model:
    def __init__(self, host, u, v, w, elem, thickness=None, nu=NU):
        self.host = host
        self.shell = host.dim == 3
        self.ndpc = 3 if self.shell else 2
        self.thickness = thickness
        self.D0 = plane_stress_D(nu)
        order = np.argsort(elem, kind="stable")
        self.u, self.v, self.w, self.elem = u[order], v[order], w[order], elem[order]
        self.ncp = host.ncu * host.ncv
        self.ndof = self.ncp * self.ndpc
        self._element_maps()
        self._point_data()
        self.F = np.zeros(self.ndof)
        self.fixed = np.zeros(0, int)
        self.set_bc(self.fixed, self.F)

    @staticmethod
    def on_grid(host, nq, thickness=None):
        u, v, w, e, _, _ = gauss_grid(host.Uu, host.Uv, nq)
        m = Model(host, u, v, w, e, thickness)
        m.gpe = nq * nq
        return m

    # -- element connectivity and sparsity pattern --------------------------------------------------------------------
    def _element_maps(self):
        nel = self.host.nelu * self.host.nelv
        self.nel = nel
        first = np.r_[0, np.flatnonzero(np.diff(self.elem)) + 1]
        self.starts = first
        assert len(first) == nel, "every element needs at least one point"
        b = self.host.basis(self.u[first], self.v[first], nd=0)
        self.ien = b["ien"]                                   # (nel, 9)
        k = self.ndpc
        self.edof = (self.ien[:, :, None] * k + np.arange(k)[None, None, :]).reshape(nel, -1)
        nd = self.edof.shape[1]
        iK = np.repeat(self.edof, nd, axis=1).ravel()
        jK = np.tile(self.edof, (1, nd)).ravel()
        key = iK.astype(np.int64) * self.ndof + jK
        ukey, self.inv = np.unique(key, return_inverse=True)
        self.rows, self.cols = ukey // self.ndof, ukey % self.ndof
        self.nnz = len(ukey)
        self.indptr = np.r_[0, np.cumsum(np.bincount(self.rows, minlength=self.ndof))]
        self.upper = self.rows <= self.cols                    # upper triangle (CSR order kept) for PARDISO SPD
        self.uindptr = np.r_[0, np.cumsum(np.bincount(self.rows[self.upper], minlength=self.ndof))]
        self._ps = None

    # -- strain-displacement data at the points ------------------------------------------------------------------------
    def _point_data(self, chunk=40000):
        m = self.u.size
        nd = self.edof.shape[1]
        self.terms = []                                        # list of (B (m,3,nd), factor)
        self.dA = np.empty(m)
        Bs = [np.empty((m, 3, nd)) for _ in range(2 if self.shell else 1)]
        for s in range(0, m, chunk):
            sl = slice(s, s + chunk)
            if self.shell:
                Bm, Bb, dA = self._shell_B(self.u[sl], self.v[sl], self.elem[sl])
                Bs[0][sl], Bs[1][sl] = Bm, Bb
            else:
                B, dA = self._plane_B(self.u[sl], self.v[sl], self.elem[sl])
                Bs[0][sl] = B
            self.dA[sl] = dA * self.w[sl]
        if self.shell:
            d = self.thickness
            self.terms = [(Bs[0], d), (Bs[1], d**3 / 12.0)]
        else:
            self.terms = [(Bs[0], 1.0)]
        self.DB = [np.einsum("ij,mjk->mik", self.D0, B) for B, _ in self.terms]
        self.area = float(self.dA.sum())

    def _basis_on_elements(self, u, v, elem, nd):
        b = self.host.basis(u, v, nd=nd)
        assert np.array_equal(b["ien"], self.ien[elem]), "point outside its element"
        return b

    def _plane_B(self, u, v, elem):
        b = self._basis_on_elements(u, v, elem, 1)
        Pe = self.host.P[b["ien"]]
        xu, yu = np.sum(b["Ru"] * Pe[..., 0], 1), np.sum(b["Ru"] * Pe[..., 1], 1)
        xv, yv = np.sum(b["Rv"] * Pe[..., 0], 1), np.sum(b["Rv"] * Pe[..., 1], 1)
        detJ = xu * yv - xv * yu
        Rx = (yv[:, None] * b["Ru"] - yu[:, None] * b["Rv"]) / detJ[:, None]
        Ry = (-xv[:, None] * b["Ru"] + xu[:, None] * b["Rv"]) / detJ[:, None]
        m, nen = Rx.shape
        B = np.zeros((m, 3, 2 * nen))
        B[:, 0, 0::2], B[:, 1, 1::2] = Rx, Ry
        B[:, 2, 0::2], B[:, 2, 1::2] = Ry, Rx
        return B, np.abs(detJ)

    def _shell_B(self, u, v, elem):
        """Linear Kirchhoff-Love membrane and bending B matrices in a local Cartesian frame (Kiendl et al. 2009)."""
        b = self._basis_on_elements(u, v, elem, 2)
        Pe = self.host.P[b["ien"]]
        g = {k: np.einsum("mk,mkd->md", b[k], Pe) for k in ("Ru", "Rv", "Ruu", "Ruv", "Rvv")}
        a1, a2 = g["Ru"], g["Rv"]
        a3t = np.cross(a1, a2)
        jac = np.linalg.norm(a3t, axis=1)
        a3 = a3t / jac[:, None]
        m, nen = b["R"].shape
        # membrane (covariant Voigt: e11, e22, 2 e12)
        Bm = np.zeros((m, 3, 3 * nen))
        for d in range(3):
            Bm[:, 0, d::3] = b["Ru"] * a1[:, d:d + 1]
            Bm[:, 1, d::3] = b["Rv"] * a2[:, d:d + 1]
            Bm[:, 2, d::3] = b["Rv"] * a1[:, d:d + 1] + b["Ru"] * a2[:, d:d + 1]
        # bending (linearized change of the second fundamental form)
        Bb = np.zeros((m, 3, 3 * nen))
        a2xa3, a3xa1 = np.cross(a2, a3), np.cross(a3, a1)
        for row, (Rab, aab, fac) in enumerate(((b["Ruu"], g["Ruu"], 1.0), (b["Rvv"], g["Rvv"], 1.0),
                                               (b["Ruv"], g["Ruv"], 2.0))):
            c1 = np.cross(a2, aab) / jac[:, None]
            c2 = np.cross(aab, a1) / jac[:, None]
            s = np.sum(a3 * aab, 1)[:, None] / jac[:, None]
            for d in range(3):
                Bb[:, row, d::3] = fac * (Rab * a3[:, d:d + 1]
                                          + b["Ru"] * (c1[:, d:d + 1] - s * a2xa3[:, d:d + 1])
                                          + b["Rv"] * (c2[:, d:d + 1] - s * a3xa1[:, d:d + 1]))
        # transformation of covariant Voigt strains to a local orthonormal frame
        G = np.stack([np.stack([np.sum(a1 * a1, 1), np.sum(a1 * a2, 1)], 1),
                      np.stack([np.sum(a2 * a1, 1), np.sum(a2 * a2, 1)], 1)], 1)
        Gi = np.linalg.inv(G)
        con1 = Gi[:, 0, 0:1] * a1 + Gi[:, 0, 1:2] * a2
        con2 = Gi[:, 1, 0:1] * a1 + Gi[:, 1, 1:2] * a2
        e1 = a1 / np.linalg.norm(a1, axis=1, keepdims=True)
        e2 = np.cross(a3, e1)
        g11, g12 = np.sum(e1 * con1, 1), np.sum(e1 * con2, 1)
        g21, g22 = np.sum(e2 * con1, 1), np.sum(e2 * con2, 1)
        T = np.stack([np.stack([g11**2, g12**2, g11 * g12], 1),
                      np.stack([g21**2, g22**2, g21 * g22], 1),
                      np.stack([2 * g11 * g21, 2 * g12 * g22, g11 * g22 + g12 * g21], 1)], 1)
        return np.einsum("mij,mjk->mik", T, Bm), np.einsum("mij,mjk->mik", T, Bb), jac

    # -- assembly and solution -----------------------------------------------------------------------------------------
    def element_matrices(self, E):
        c = E * self.dA
        nd = self.edof.shape[1]
        Ke = np.zeros((self.nel, nd, nd))
        if hasattr(self, "gpe"):
            q = self.gpe
            for (B, f), DB in zip(self.terms, self.DB):
                A = (f * c[:, None, None] * DB).reshape(self.nel, q * 3, nd)
                Ke += np.matmul(B.reshape(self.nel, q * 3, nd).transpose(0, 2, 1), A)
        else:
            for (B, f), DB in zip(self.terms, self.DB):
                for s in range(0, len(c), 20000):
                    sl = slice(s, s + 20000)
                    contrib = np.einsum("mik,mil->mkl", B[sl], f * c[sl, None, None] * DB[sl])
                    np.add.at(Ke, self.elem[sl], contrib)
        return Ke

    def _stiffness_data(self, E):
        data = np.bincount(self.inv, weights=self.element_matrices(E).ravel(), minlength=self.nnz)
        data[self.mask_fixed] = 0.0
        data[self.diag_fixed] = 1.0
        return data

    def stiffness(self, E):
        return sp.csr_matrix((self._stiffness_data(E), self.cols, self.indptr), shape=(self.ndof, self.ndof))

    def set_bc(self, fixed_dofs, F):
        self.fixed = np.unique(np.asarray(fixed_dofs, int))
        self.F = np.asarray(F, float).copy()
        self.F[self.fixed] = 0.0
        fx = np.zeros(self.ndof, bool)
        fx[self.fixed] = True
        self.mask_fixed = fx[self.rows] | fx[self.cols]
        self.diag_fixed = np.where(self.mask_fixed & (self.rows == self.cols))[0]

    def solve(self, E):
        data = self._stiffness_data(E)
        U = None
        if USE_PARDISO:                                        # symmetric positive definite: upper triangle only
            import pypardiso
            if self._ps is None:
                self._ps = pypardiso.PyPardisoSolver(mtype=2)
            Ku = sp.csr_matrix((data[self.upper], self.cols[self.upper], self.uindptr), shape=(self.ndof, self.ndof))
            try:
                self._ps.factorize(Ku)
                U = self._ps.solve(Ku, self.F)
            except Exception:  # noqa: BLE001  (e.g. a zero pivot of a singular matrix: fall back to SuperLU)
                U = None
        if U is None:
            K = sp.csr_matrix((data, self.cols, self.indptr), shape=(self.ndof, self.ndof))
            lu = splu(K.tocsc(), permc_spec="MMD_AT_PLUS_A", diag_pivot_thresh=0.0,
                      options=dict(SymmetricMode=True))
            U = lu.solve(self.F)
        return float(self.F @ U), U

    def __del__(self):
        ps = getattr(self, "_ps", None)
        if ps is not None:
            try:
                ps.free_memory(everything=True)
            except Exception:  # noqa: BLE001
                pass

    def energy_density(self, U):
        """Strain energy per point for unit modulus, including the area weight: dJ/dE_q = -energy_q."""
        ue = U[self.edof][self.elem]                            # (m, nd)
        out = np.zeros(self.u.size)
        for (B, f), DB in zip(self.terms, self.DB):
            eps = np.einsum("mik,mk->mi", B, ue)
            sig = np.einsum("mik,mk->mi", DB, ue)
            out += f * np.sum(eps * sig, 1)
        return out * self.dA

    def stress(self, U):
        """Plane-stress Cauchy stress (unit modulus) at the points: (m, 3) = [s11, s22, s12]."""
        ue = U[self.edof][self.elem]
        return np.einsum("mik,mk->mi", self.DB[0], ue)

    def density_matrix(self):
        """Sparse (m, ncp) matrix mapping control-point values to point values."""
        b = self.host.basis(self.u, self.v, nd=0)
        m = self.u.size
        return sp.csr_matrix((b["R"].ravel(), (np.repeat(np.arange(m), b["R"].shape[1]), b["ien"].ravel())),
                             shape=(m, self.ncp))


# ------------------------------------------------------------------------------------------------ loads and supports
EDGES = {"u0": (0, 0.0), "u1": (0, 1.0), "v0": (1, 0.0), "v1": (1, 1.0)}


def _edge_points(edge, s):
    fixed_axis, val = EDGES[edge]
    return (np.full_like(s, val), s) if fixed_axis == 0 else (s, np.full_like(s, val))


def edge_load(model, edge, s0, s1, direction, magnitude, nsub=64, ngl=4):
    """Consistent nodal forces of a uniform traction on a parametric edge interval (total force = magnitude)."""
    host = model.host
    xg, wg = np.polynomial.legendre.leggauss(ngl)
    a = np.linspace(s0, s1, nsub + 1)
    h = np.diff(a) / 2
    s = ((a[:-1] + h)[:, None] + h[:, None] * xg[None]).ravel()
    ws = (h[:, None] * wg[None]).ravel()
    uu, vv = _edge_points(edge, s)
    b = host.basis(uu, vv, nd=1)
    Rt = b["Rv"] if EDGES[edge][0] == 0 else b["Ru"]
    tangent = np.einsum("mk,mkd->md", Rt, host.P[b["ien"]])
    ds = np.linalg.norm(tangent, axis=1) * ws
    dvec = np.asarray(direction, float)
    dvec = dvec / np.linalg.norm(dvec)
    F = np.zeros(model.ndof)
    k = model.ndpc
    for d in range(k):
        np.add.at(F, k * b["ien"].ravel() + d, (b["R"] * (ds * magnitude / ds.sum() * dvec[d])[:, None]).ravel())
    return F


def surface_load(model, direction, magnitude, centre=None, radius=None, nq=4):
    """Shell surface load: 'normal' or a vector direction, over the whole surface or a parametric disk."""
    host = model.host
    u, v, w, _, _, _ = gauss_grid(host.Uu, host.Uv, nq)
    if centre is not None:
        keep = (u - centre[0]) ** 2 + (v - centre[1]) ** 2 <= radius**2
        u, v, w = u[keep], v[keep], w[keep]
    b = host.basis(u, v, nd=1)
    Pe = host.P[b["ien"]]
    a1 = np.einsum("mk,mkd->md", b["Ru"], Pe)
    a2 = np.einsum("mk,mkd->md", b["Rv"], Pe)
    n = np.cross(a1, a2)
    dA = np.linalg.norm(n, axis=1) * w
    if isinstance(direction, str):
        dirs = n / np.linalg.norm(n, axis=1, keepdims=True)
    else:
        dirs = np.broadcast_to(np.asarray(direction, float) / np.linalg.norm(direction), (u.size, 3))
    F = np.zeros(model.ndof)
    for d in range(3):
        np.add.at(F, 3 * b["ien"].ravel() + d, (b["R"] * (dA * magnitude / dA.sum() * dirs[:, d])[:, None]).ravel())
    return F


def edge_cps(host, edge, s0, s1, row=0):
    """Control points of an edge (row 0) or of the next row inward (row 1) whose Greville abscissa is in [s0, s1]."""
    gu, gv = host.greville_uv()
    axis, val = EDGES[edge]
    if axis == 0:
        iu = row if val == 0.0 else host.ncu - 1 - row
        g = gv
        sel = np.where((g >= s0 - 1e-12) & (g <= s1 + 1e-12))[0]
        if len(sel) < 2:
            sel = np.argsort(np.abs(g - 0.5 * (s0 + s1)))[:2]
        return iu * host.ncv + sel
    iv = row if val == 0.0 else host.ncv - 1 - row
    g = gu
    sel = np.where((g >= s0 - 1e-12) & (g <= s1 + 1e-12))[0]
    if len(sel) < 2:
        sel = np.argsort(np.abs(g - 0.5 * (s0 + s1)))[:2]
    return sel * host.ncv + iv


In [ ]:
%%writefile trimfields/problems.py
"""Random design problems: host, supports, loads, passive regions and target volume fraction.

Planar loads are edge tractions (or bolt bearing loads for bolted); shell loads are surface loads.
OOD-load split: in-distribution problems never centre a load in the band s in (0.42, 0.58) of edge v1;
OOD-load problems always do.

TrimFields states every boundary condition for the continuum, so that mesh-free solvers and the isogeometric
reference solve the same problem: a roller is u . n = 0 on an edge whose physical normal is constant and aligned
with a coordinate axis (fixed_component), a bolt support is u = 0 on the ring annulus around the hole, and a bolt
load is a bearing traction on the hole boundary with the cosine law of a rigid pin (resultant = mag * dir).
The random designs of the `rand` splits are admissible loop sets without optimization (random_loops).
"""
import math

import numpy as np

from .hosts import SHELL, make_host
from .mechanics import EDGES, edge_cps, edge_load, surface_load

OOD_BAND = (0.42, 0.58)
EDGE_LIST = ("u0", "u1", "v0", "v1")


def _unit(v):
    v = np.asarray(v, float)
    return (v / np.linalg.norm(v)).tolist()


def _edge_segment(rng, length_range, avoid=(), band=None, force_band=False, edge=None):
    for _ in range(200):
        e = edge or EDGE_LIST[rng.integers(4)]
        ln = rng.uniform(*length_range)
        if force_band:
            c = rng.uniform(*band)
        else:
            c = rng.uniform(ln / 2, 1 - ln / 2)
            if band is not None and e == "v1" and band[0] < c < band[1]:
                continue
        s0, s1 = c - ln / 2, c + ln / 2
        if all(not (a_e == e and s0 < a_s1 + 0.08 and s1 > a_s0 - 0.08) for a_e, a_s0, a_s1 in avoid):
            return e, float(max(0.0, s0)), float(min(1.0, s1))
    raise RuntimeError("no admissible edge segment")


def sample_problem(family, rng, split="id"):
    host = make_host(family, rng)
    prob = {"family": family, "split": split, "host": host.to_dict(), "vstar": float(rng.uniform(0.25, 0.6)),
            "supports": [], "loads": [], "passive": []}
    if family in SHELL:
        _shell_bc(prob, host, rng)
    elif family == "bolted":
        _bolt_hole_bc(prob, host, rng)
    elif family == "ell":
        _l_shape_bc(prob, rng)
    else:
        _planar_bc(prob, host, rng, split)
    return prob


def roller_edges(host, n=16, tol=1e-9):
    """Edges whose physical outward normal is constant along the edge and aligned with a coordinate axis, with the
    index of the displacement component that a roller on that edge fixes. Only such edges carry rollers, so that
    u . n = 0 is a condition on one Cartesian component, exact for the isogeometric and for continuum solvers."""
    out = {}
    s = (np.arange(n) + 0.5) / n
    for e, (axis, val) in EDGES.items():
        u, v = (np.full(n, val), s) if axis == 0 else (s, np.full(n, val))
        _, J = host.jac(u, v)
        t = J[:, :2, 1 - axis]
        nrm = np.stack([t[:, 1], -t[:, 0]], 1)
        nrm /= np.linalg.norm(nrm, axis=1, keepdims=True)
        if np.abs(nrm - nrm[0]).max() > tol:
            continue
        comp = int(np.argmax(np.abs(nrm[0])))
        if abs(nrm[0, 1 - comp]) > tol:
            continue
        out[e] = comp
    return out


def _planar_bc(prob, host, rng, split):
    r = rng.uniform()
    avoid = []
    rollers = roller_edges(host) if host.dim == 2 else {}
    if r < 0.45:                                           # cantilever: one edge clamped
        e = EDGE_LIST[rng.integers(4)]
        prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "all"})
        avoid.append((e, 0.0, 1.0))
    elif r < 0.75 or not rollers:                          # two pinned segments (also where no edge admits a roller)
        for _ in range(2):
            e, s0, s1 = _edge_segment(rng, (0.05, 0.15), avoid)
            prob["supports"].append({"kind": "edge", "edge": e, "s0": s0, "s1": s1, "dofs": "all"})
            avoid.append((e, s0, s1))
    else:                                                  # roller on a full straight edge + pinned segment
        e = sorted(rollers)[rng.integers(len(rollers))]
        prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "normal",
                                 "fixed_component": rollers[e]})
        avoid.append((e, 0.0, 1.0))
        e2, s0, s1 = _edge_segment(rng, (0.05, 0.15), avoid)
        prob["supports"].append({"kind": "edge", "edge": e2, "s0": s0, "s1": s1, "dofs": "all"})
        avoid.append((e2, s0, s1))
    n_loads = 1 if rng.uniform() < 0.7 else 2
    w = [1.0] if n_loads == 1 else [float(x) for x in (lambda a: (a, 1 - a))(rng.uniform(0.3, 0.7))]
    for k in range(n_loads):
        force = split == "ood_load" and k == 0
        e, s0, s1 = _edge_segment(rng, (0.03, 0.15), avoid, band=OOD_BAND, force_band=force,
                                  edge="v1" if force else None)
        ang = rng.uniform(0, 2 * math.pi)
        prob["loads"].append({"kind": "edge", "edge": e, "s0": s0, "s1": s1,
                              "dir": [math.cos(ang), math.sin(ang)], "mag": w[k]})
        avoid.append((e, s0, s1))


def _bolt_hole_bc(prob, host, rng):
    L, H = host.params["L"], host.params["H"]
    n = int(rng.integers(1, 4))
    holes = []
    for _ in range(500):
        if len(holes) == n:
            break
        r = float(rng.uniform(0.06, 0.12) * H)
        ring = 0.08 * H
        m = r + ring + 0.05 * H
        c = (float(rng.uniform(m, L - m)), float(rng.uniform(m, H - m)))
        if all(math.hypot(c[0] - h["centre"][0], c[1] - h["centre"][1]) > r + h["radius"] + 2 * ring + 0.1 * H
               for h in holes):
            holes.append({"kind": "circle", "centre": list(c), "radius": r, "ring": ring})
    prob["passive"] = holes
    ang = rng.uniform(0, 2 * math.pi)
    prob["loads"].append({"kind": "ring", "passive": len(holes) - 1, "dir": [math.cos(ang), math.sin(ang)],
                          "mag": 1.0})
    if len(holes) == 1:
        e = EDGE_LIST[rng.integers(4)]
        prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "all"})
    else:                                       # every other hole is a bolted support (no floating rings)
        for k in range(len(holes) - 1):
            prob["supports"].append({"kind": "ring", "passive": k})


def _l_shape_bc(prob, rng):
    a = float(rng.uniform(0.35, 0.6))
    prob["passive"] = [{"kind": "polygon", "uv": [[a, a], [1.2, a], [1.2, 1.2], [a, 1.2]]}]
    prob["vstar"] *= 1.0 - (1.0 - a) ** 2          # target fraction of the L-shaped design domain, not of the square
    prob["supports"].append({"kind": "edge", "edge": "v1", "s0": 0.0, "s1": a, "dofs": "all"})
    ln = float(rng.uniform(0.04, 0.12))
    c = float(rng.uniform(ln / 2 + 0.02, a - ln / 2 - 0.02))
    ang = math.radians(rng.uniform(-135, -45))
    prob["loads"].append({"kind": "edge", "edge": "u1", "s0": c - ln / 2, "s1": c + ln / 2,
                          "dir": [math.cos(ang), math.sin(ang)], "mag": 1.0})


def _shell_bc(prob, host, rng):
    r = rng.uniform()
    if r < 0.35:                                           # two opposite edges clamped
        pair = ("u0", "u1") if rng.uniform() < 0.5 else ("v0", "v1")
        for e in pair:
            prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "clamped"})
    elif r < 0.6:                                          # four corners pinned
        for e in ("u0", "u1"):
            for s0, s1 in ((0.0, 0.08), (0.92, 1.0)):
                prob["supports"].append({"kind": "edge", "edge": e, "s0": s0, "s1": s1, "dofs": "pinned"})
    elif r < 0.8:                                          # all edges pinned
        for e in EDGE_LIST:
            prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "pinned"})
    else:                                                  # one edge clamped (cantilever shell)
        e = EDGE_LIST[rng.integers(4)]
        prob["supports"].append({"kind": "edge", "edge": e, "s0": 0.0, "s1": 1.0, "dofs": "clamped"})
    # concentrated loads: perforated shells carry them through rib-like load paths (distributed pressure or
    # self-weight favours continuous membranes, for which holes are not an efficient design)
    n_loads = 1 if rng.uniform() < 0.6 else 2
    w = [1.0] if n_loads == 1 else [float(x) for x in (lambda a: (a, 1 - a))(rng.uniform(0.3, 0.7))]
    for k in range(n_loads):
        c = [float(x) for x in rng.uniform(0.2, 0.8, 2)]
        d = "normal" if rng.uniform() < 0.5 else [0.0, 0.0, -1.0]
        prob["loads"].append({"kind": "surface", "dir": d, "centre": c, "radius": float(rng.uniform(0.04, 0.08)),
                              "mag": w[k]})
    prob["vstar"] = float(rng.uniform(0.25, 0.5))


# ------------------------------------------------------------------------------------------------ passive regions
def passive_masks_cp(prob, host):
    """Control points inside passive voids (rho = 0) and passive solid rings (rho = 1)."""
    P = host.P
    void = np.zeros(len(P), bool)
    solid = np.zeros(len(P), bool)
    gu, gv = host.greville_uv()
    GU, GV = np.meshgrid(gu, gv, indexing="ij")
    uv = np.stack([GU.ravel(), GV.ravel()], 1)
    for ps in prob["passive"]:
        if ps["kind"] == "circle":
            d = np.hypot(P[:, 0] - ps["centre"][0], P[:, 1] - ps["centre"][1])
            void |= d < ps["radius"]
            solid |= (d >= ps["radius"]) & (d < ps["radius"] + ps["ring"])
        else:
            from matplotlib.path import Path
            void |= Path(np.asarray(ps["uv"])).contains_points(uv)
    return void, solid


def ring_cps(prob, host, k):
    ps = prob["passive"][k]
    d = np.hypot(host.P[:, 0] - ps["centre"][0], host.P[:, 1] - ps["centre"][1])
    return np.where((d >= ps["radius"]) & (d < ps["radius"] + ps["ring"]))[0]


# ------------------------------------------------------------------------------------------------ non-design regions
def _boundary_items(prob):
    """Edge supports/loads and patch loads; ring supports/loads are already passive solid rings."""
    edges = [x for x in prob["supports"] + prob["loads"] if x["kind"] == "edge"]
    patches = [x for x in prob["loads"] if x["kind"] == "surface" and x.get("centre") is not None]
    return edges, patches


def keep_cps(prob, host):
    """Control points kept solid in SIMP: two edge rows under every support and edge load, and the control points
    whose Greville abscissae lie in a load patch (standard non-design regions of density-based TO)."""
    keep = np.zeros(host.ncu * host.ncv, bool)
    edges, patches = _boundary_items(prob)
    for x in edges:                             # two rows: a solid strip of about 1.5 elements
        for row in (0, 1):
            keep[edge_cps(host, x["edge"], x["s0"], x["s1"], row=row)] = True
    if patches:
        gu, gv = host.greville_uv()
        GU, GV = np.meshgrid(gu, gv, indexing="ij")
        for x in patches:
            r = max(x["radius"], 0.75 / min(host.nelu, host.nelv))
            keep |= ((GU - x["centre"][0]) ** 2 + (GV - x["centre"][1]) ** 2 <= r**2).ravel()
    return keep


def keep_geometry(prob, n=96):
    """Exact protected regions in the parameter square (shapely): supported and loaded boundary segments, load
    patches and bolt-hole rings. An admissible loop interior must not overlap any of them."""
    from shapely.geometry import LineString, Point, Polygon
    edges, patches = _boundary_items(prob)
    out = []
    for x in edges:
        axis, val = EDGES[x["edge"]]
        a, b = (val, x["s0"]), (val, x["s1"])
        out.append(LineString([a, b] if axis == 0 else [a[::-1], b[::-1]]))
    for x in patches:
        out.append(Point(x["centre"]).buffer(x["radius"], quad_segs=n // 4))
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    for ps in prob["passive"]:
        if ps["kind"] == "circle":
            L, H = prob["host"]["params"]["L"], prob["host"]["params"]["H"]
            ring = [np.stack([(ps["centre"][0] + r * np.cos(t)) / L, (ps["centre"][1] + r * np.sin(t)) / H], 1)
                    for r in (ps["radius"] + ps["ring"], ps["radius"])]
            out.append(Polygon(ring[0], [ring[1]]))
    return out


def keep_samples(keep_geoms, spacing=0.01):
    """Parametric sample points of the non-design set: points along protected boundary segments and on the
    boundary and interior of protected patches and rings (used by the keep-region penalty of the refinement)."""
    pts = []
    for g in keep_geoms:
        if g.geom_type == "LineString":
            n = max(2, int(np.ceil(g.length / spacing)) + 1)
            pts.append(np.array([g.interpolate(t, normalized=True).coords[0] for t in np.linspace(0, 1, n)]))
            continue
        for ring in [g.exterior] + list(g.interiors):
            n = max(8, int(np.ceil(ring.length / spacing)))
            pts.append(np.array([ring.interpolate(t, normalized=True).coords[0] for t in np.linspace(0, 1, n,
                                                                                                  endpoint=False)]))
        x0, y0, x1, y1 = g.bounds
        X, Y = np.meshgrid(np.arange(x0, x1, spacing), np.arange(y0, y1, spacing), indexing="ij")
        cand = np.stack([X.ravel(), Y.ravel()], 1)
        if len(cand):
            import shapely
            pts.append(cand[shapely.contains_xy(g, cand[:, 0], cand[:, 1])])
    return np.concatenate(pts) if pts else np.zeros((0, 2))


def covers_keep(loop_poly, keep_geoms, tol=1e-7):
    """True if a loop interior (shapely polygon) overlaps a protected region by more than touching."""
    for g in keep_geoms:
        if not loop_poly.intersects(g):
            continue
        inter = loop_poly.intersection(g)
        if (inter.length if g.geom_type == "LineString" else inter.area) > tol:
            return True
    return False


def passive_discs_uv(prob, scale=1.0, n=64):
    """Hole-plus-ring discs of the bolt holes as parametric polygons (rectangle host)."""
    out = []
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    for ps in prob["passive"]:
        if ps["kind"] == "circle":
            L, H = prob["host"]["params"]["L"], prob["host"]["params"]["H"]
            r = scale * (ps["radius"] + ps["ring"])
            out.append(np.stack([(ps["centre"][0] + r * np.cos(t)) / L, (ps["centre"][1] + r * np.sin(t)) / H], 1))
    return out


# ------------------------------------------------------------------------------------------------ random designs
def random_loops(prob, host, nel, rng, n_cp=16, max_loops=8, w_min_h=1.5, keep_geoms=(), tries=400):
    """Admissible loop set without optimization (the `rand` splits): 1 to max_loops closed loops, each an ellipse
    with a low-order radial perturbation (a blob), with radii of 1.5 to 8 elements, pairwise clearance of w_min and
    no overlap with the non-design set. Loops near the patch boundary may protrude beyond the square (notches)."""
    from shapely.geometry import Polygon, box

    from .loops import fit_periodic, sample_loop
    L = np.array([host.Lu, host.Lv])
    h = math.sqrt(host.area / (nel[0] * nel[1]))
    w_min = w_min_h * h
    el_area = 1.0 / (nel[0] * nel[1])
    K = int(rng.integers(1, max_loops + 1))
    loops, polys = [], []
    t = np.linspace(0, 2 * np.pi, 200, endpoint=False)
    for _ in range(tries):
        if len(loops) == K:
            break
        c = rng.uniform(0.04, 0.96, 2) * L
        r0 = rng.uniform(1.5, 8.0) * h
        asp = math.exp(rng.uniform(math.log(0.5), math.log(2.0)))
        a, b = r0 * math.sqrt(asp), r0 / math.sqrt(asp)
        amp, ph = rng.uniform(0.0, 0.25, 3), rng.uniform(0.0, 2 * np.pi, 3)
        rad = 1.0 + sum(amp[i] * np.cos((i + 2) * t + ph[i]) for i in range(3))
        th = rng.uniform(0.0, 2 * np.pi)
        R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]])
        pts = c + (R @ np.stack([a * rad * np.cos(t), b * rad * np.sin(t)])).T
        C = fit_periodic(pts, n_cp) / L
        poly_uv = Polygon(sample_loop(C, 20)).buffer(0)
        if poly_uv.is_empty or poly_uv.intersection(box(0, 0, 1, 1)).area < 2 * el_area:
            continue
        if keep_geoms and covers_keep(poly_uv, keep_geoms):
            continue
        poly_z = Polygon(sample_loop(C, 20) * L).buffer(0)
        if any(poly_z.distance(p) < w_min for p in polys):
            continue
        loops.append(C)
        polys.append(poly_z)
    return loops


# ------------------------------------------------------------------------------------------------ boundary conditions
def bearing_load(model, prob, k, direction, magnitude, n=256):
    """Consistent nodal forces of the bearing traction of a rigid pin in bolt hole k (rectangle host): the radial
    traction p0 cos(phi) e_r on the half of the hole boundary facing the load direction, with p0 = 2 F / (pi r), so
    that the resultant is magnitude * direction; the line integral runs over the exact ellipse of the hole in the
    parameter square."""
    host = model.host
    ps = prob["passive"][k]
    L, H = prob["host"]["params"]["L"], prob["host"]["params"]["H"]
    d = np.asarray(direction, float)
    d = d / np.linalg.norm(d)
    phi = (np.arange(n) + 0.5) * 2 * np.pi / n
    er = np.stack([np.cos(phi), np.sin(phi)], 1)
    cosw = np.maximum(er @ d, 0.0)
    r = ps["radius"]
    ds = 2 * np.pi * r / n
    traction = (2 * magnitude / (np.pi * r)) * cosw[:, None] * er
    x = ps["centre"][0] + r * np.cos(phi)
    y = ps["centre"][1] + r * np.sin(phi)
    b = host.basis(x / L, y / H, nd=0)
    F = np.zeros(model.ndof)
    kd = model.ndpc
    for c in range(kd):
        np.add.at(F, kd * b["ien"].ravel() + c, (b["R"] * (traction[:, c] * ds)[:, None]).ravel())
    res = np.array([F[c::kd].sum() for c in range(kd)])            # quadrature of the cosine law: rescale exactly
    F *= magnitude / np.linalg.norm(res)
    return F


def apply_bc(prob, model):
    host, k = model.host, model.ndpc
    fixed = []
    for s in prob["supports"]:
        if s["kind"] == "ring":                           # bolt support: u = 0 on the ring annulus
            cps = ring_cps(prob, host, s["passive"])
            fixed += [k * cps + d for d in range(k)]
            continue
        cps = edge_cps(host, s["edge"], s["s0"], s["s1"])
        if s["dofs"] == "all" or s["dofs"] == "pinned":
            fixed += [k * cps + d for d in range(k)]
        elif s["dofs"] == "clamped":
            cps1 = edge_cps(host, s["edge"], s["s0"], s["s1"], row=1)
            fixed += [k * c + d for c in (cps, cps1) for d in range(k)]
        elif s["dofs"] == "normal":                      # roller: u . n = 0 on an axis-aligned straight edge
            comp = s.get("fixed_component")
            if comp is None:
                comp = roller_edges(host)[s["edge"]]
            fixed.append(k * cps + int(comp))
    F = np.zeros(model.ndof)
    for ld in prob["loads"]:
        if ld["kind"] == "edge":
            F += edge_load(model, ld["edge"], ld["s0"], ld["s1"], ld["dir"], ld["mag"])
        elif ld["kind"] == "ring":
            F += bearing_load(model, prob, ld["passive"], ld["dir"], ld["mag"])
        else:
            F += surface_load(model, ld["dir"] if isinstance(ld["dir"], str) else ld["dir"], ld["mag"],
                              ld["centre"], ld["radius"])
    model.set_bc(np.concatenate([np.atleast_1d(f) for f in fixed]) if fixed else np.zeros(0, int), F)
    return model


In [ ]:
%%writefile trimfields/simp.py
"""Isogeometric SIMP with control-point densities, optimality-criteria update and a control-net filter."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.spatial import cKDTree


def _check_feasible(dvol, vstar, void, solid, margin=0.02):
    """The volume target must lie between the fixed solid volume and the volume of the design domain."""
    v_min, v_max = float(dvol[solid].sum()), float(dvol[~void].sum())
    if not v_min + margin < vstar < v_max - margin:
        raise ValueError(f"infeasible volume target {vstar:.3f} (fixed solid {v_min:.3f}, domain {v_max:.3f})")


def run_simp(model, vstar, rmin, void=None, solid=None, penal=3.0, maxit=200, tol=0.01, move=0.2, emin=1e-9,
             continuation=False, min_iters=0):
    """OC-based SIMP. With continuation, the penalty rises as 1 + 0.1 it up to `penal` (used for shells, whose
    grey designs otherwise keep membrane continuity), and the loop stops only after reaching the final penalty."""
    ncp = model.ncp
    Nmat = model.density_matrix()
    dvol = Nmat.T @ (model.dA / model.area)
    tree = cKDTree(model.host.P)
    Hf = tree.sparse_distance_matrix(tree, rmin, output_type="coo_matrix")
    Hf = sp.coo_matrix((rmin - Hf.data, (Hf.row, Hf.col)), shape=(ncp, ncp)).tocsr() + sp.eye(ncp) * rmin
    Hs = np.asarray(Hf.sum(1)).ravel()
    void = np.zeros(ncp, bool) if void is None else void
    solid = np.zeros(ncp, bool) if solid is None else solid
    _check_feasible(dvol, vstar, void, solid)
    rho = np.full(ncp, vstar)
    rho[void], rho[solid] = 0.0, 1.0
    t0 = time.perf_counter()
    change, it, J = 1.0, 0, np.nan
    p_it = penal
    while it < maxit and (change > tol or it < min_iters or p_it < penal):
        p_it = min(penal, 1.0 + 0.1 * it) if continuation else penal
        rq = Nmat @ rho
        J, U = model.solve(emin + rq**p_it * (1 - emin))
        dJdE = -model.energy_density(U)
        dc = Nmat.T @ (p_it * rq ** (p_it - 1) * (1 - emin) * dJdE)
        dc = (Hf @ (rho * dc)) / (Hs * np.maximum(1e-3, rho))
        l1, l2 = 0.0, 1e9
        while (l2 - l1) / (l1 + l2) > 1e-4 and l2 > 1e-40:
            lm = 0.5 * (l1 + l2)
            Be = np.maximum(-dc, 0) / np.maximum(dvol * lm, 1e-300)
            rnew = np.clip(rho * np.sqrt(np.minimum(Be, 1e300)), np.maximum(0, rho - move), np.minimum(1, rho + move))
            rnew[void], rnew[solid] = 0.0, 1.0
            l1, l2 = (lm, l2) if dvol @ rnew > vstar else (l1, lm)
        change = float(np.abs(rnew - rho).max())
        rho = rnew
        it += 1
    rq = Nmat @ rho
    J, _ = model.solve(emin + rq**penal * (1 - emin))
    return rho, {"J_simp": float(J), "iterations": it, "final_change": change, "vol": float(dvol @ rho),
                 "grey_percent": float(400 * np.mean(rq * (1 - rq))), "t_s": time.perf_counter() - t0}


def _proj(x, beta, eta=0.5):
    a = np.tanh(beta * eta)
    return (a + np.tanh(beta * (x - eta))) / (a + np.tanh(beta * (1 - eta)))


def _dproj(x, beta, eta=0.5):
    a = np.tanh(beta * eta)
    return beta * (1 - np.tanh(beta * (x - eta)) ** 2) / (a + np.tanh(beta * (1 - eta)))


def run_simp_projected(model, vstar, rmin, penal=3.0, maxit=250, tol=0.005, move=0.2, emin=1e-6, beta_max=16.0,
                       beta_every=25, solid=None):
    """SIMP with a density filter and Heaviside projection (beta doubled every `beta_every` iterations up to
    beta_max). Used for shells, where plain SIMP stays grey because bending stiffness scales like thickness cubed.
    Control points in `solid` are held at 1 after projection."""
    ncp = model.ncp
    Nmat = model.density_matrix()
    dvol = Nmat.T @ (model.dA / model.area)
    tree = cKDTree(model.host.P)
    Hf = tree.sparse_distance_matrix(tree, rmin, output_type="coo_matrix")
    Hf = sp.coo_matrix((rmin - Hf.data, (Hf.row, Hf.col)), shape=(ncp, ncp)).tocsr() + sp.eye(ncp) * rmin
    Hs = np.asarray(Hf.sum(1)).ravel()
    solid = np.zeros(ncp, bool) if solid is None else solid

    def phys(x_, b_):
        xp_ = _proj((Hf @ x_) / Hs, b_)
        xp_[solid] = 1.0
        return xp_

    _check_feasible(dvol, vstar, np.zeros(ncp, bool), solid)
    x = np.full(ncp, vstar)
    x[solid] = 1.0
    beta, it, change = 1.0, 0, 1.0
    t0 = time.perf_counter()
    while it < maxit and (change > tol or beta < beta_max):
        xt = (Hf @ x) / Hs
        xp = phys(x, beta)
        rq = Nmat @ xp
        J, U = model.solve(emin + rq**penal * (1 - emin))
        dc_xp = Nmat.T @ (penal * rq ** (penal - 1) * (1 - emin) * (-model.energy_density(U)))
        dp = _dproj(xt, beta)
        dp[solid] = 0.0
        dc = Hf @ (dc_xp * dp / Hs)
        dv = Hf @ (dvol * dp / Hs)
        l1, l2 = 0.0, 1e9
        while (l2 - l1) / (l1 + l2) > 1e-4 and l2 > 1e-40:
            lm = 0.5 * (l1 + l2)
            Be = np.maximum(-dc, 0) / np.maximum(np.maximum(dv, 1e-30) * lm, 1e-300)
            xnew = np.clip(x * np.sqrt(np.minimum(Be, 1e300)), np.maximum(0, x - move), np.minimum(1, x + move))
            xnew[solid] = 1.0
            l1, l2 = (lm, l2) if dvol @ phys(xnew, beta) > vstar else (l1, lm)
        change = float(np.abs(xnew - x).max())
        x = xnew
        it += 1
        if it % beta_every == 0 and beta < beta_max:
            beta, change = 2 * beta, 1.0
    rho = phys(x, beta)
    rq = Nmat @ rho
    J, _ = model.solve(emin + rq**penal * (1 - emin))
    return rho, {"J_simp": float(J), "iterations": it, "final_change": change, "vol": float(dvol @ rho),
                 "grey_percent": float(400 * np.mean(rq * (1 - rq))), "beta": beta, "t_s": time.perf_counter() - t0}


def density_at(host, rho, u, v):
    b = host.basis(np.clip(u, 0, 1), np.clip(v, 0, 1), nd=0)
    return np.sum(b["R"] * rho[b["ien"]], 1)


In [ ]:
%%writefile trimfields/loops.py
"""Trimming loops: extraction from density fields, periodic cubic fitting, validity checks, canonical form.

Loops are stored in parameter coordinates (u, v); fitting works in scaled coordinates z = (Lu u, Lv v), which are
close to physical lengths, so that control points spread evenly along the physical boundary.
"""
import numpy as np
from shapely.geometry import LineString, Polygon, box
from skimage import measure

from .basis import periodic_basis
from .simp import density_at


def sample_loop(C, per_span=20):
    n = len(C)
    return periodic_basis(n, np.arange(n * per_span) / (n * per_span)) @ C


def signed_area(P):
    x, y = P[:, 0], P[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)


def resample_closed(P, m):
    seg = np.linalg.norm(np.diff(np.vstack([P, P[:1]]), axis=0), axis=1)
    s = np.r_[0, np.cumsum(seg)]
    t = np.linspace(0, s[-1], m, endpoint=False)
    Pc = np.vstack([P, P[:1]])
    return np.stack([np.interp(t, s, Pc[:, 0]), np.interp(t, s, Pc[:, 1])], 1)


def fit_periodic(P, n, iters=8, lam=1e-3):
    """Least-squares closed periodic cubic B-spline with n control points, foot-point parameter correction."""
    m = max(8 * n, len(P))
    X = resample_closed(P, m)
    t = np.arange(m) / m
    D = np.eye(n) - np.roll(np.eye(n), 1, axis=1)
    reg = lam * (D.T @ D) * m / n
    for _ in range(iters):
        B = periodic_basis(n, t)
        C = np.linalg.solve(B.T @ B + reg, B.T @ X)
        dense_t = np.arange(40 * n) / (40 * n)
        S = periodic_basis(n, dense_t) @ C
        d = ((X[:, None, :] - S[None, :, :]) ** 2).sum(-1)
        t = dense_t[np.argmin(d, 1)]
    return C


def extract_void_loops(host, rho, level=0.5, spe=6, min_area_frac=2.0):
    """Closed rho = level contours enclosing voids in (u, v). A solid ring around an extended grid closes every
    contour; loops touching the patch boundary therefore protrude beyond the parameter square."""
    nu, nv = spe * host.nelu, spe * host.nelv
    pad = 3
    us = (np.arange(-pad, nu + pad) + 0.5) / nu
    vs = (np.arange(-pad, nv + pad) + 0.5) / nv
    U, V = np.meshgrid(us, vs, indexing="ij")
    Z = density_at(host, rho, U.ravel(), V.ravel()).reshape(U.shape)
    Z = np.pad(Z, 1, constant_values=1.0)
    us = np.r_[us[0] - 1 / nu, us, us[-1] + 1 / nu]
    vs = np.r_[vs[0] - 1 / nv, vs, vs[-1] + 1 / nv]
    loops = []
    for c in measure.find_contours(Z, level):
        if np.linalg.norm(c[0] - c[-1]) > 1e-9:
            continue
        uv = np.stack([np.interp(c[:-1, 0], np.arange(len(us)), us), np.interp(c[:-1, 1], np.arange(len(vs)), vs)], 1)
        if len(uv) < 8 or signed_area(uv) == 0:
            continue
        loops.append(uv)
    # keep void loops (density below the level just inside the curve) that are not nested in another void loop
    el_area = 1.0 / (host.nelu * host.nelv)
    keep = []
    for i, a in enumerate(loops):
        poly = Polygon(a).buffer(0)
        if poly.is_empty:
            continue
        pt = poly.representative_point()
        if density_at(host, rho, np.array([np.clip(pt.x, 0, 1)]), np.array([np.clip(pt.y, 0, 1)]))[0] >= level:
            continue                                     # material island, not a void
        if any(j != i and Polygon(b).buffer(0).contains(pt) and Polygon(b).buffer(0).area > poly.area
               for j, b in enumerate(loops)):
            continue
        if poly.intersection(box(0, 0, 1, 1)).area < min_area_frac * el_area:
            continue
        keep.append(a)
    return keep


def fit_loops(host, loops_uv, n_cp=16):
    L = np.array([host.Lu, host.Lv])
    out = []
    for P in loops_uv:
        C = fit_periodic(P * L, n_cp) / L
        out.append(C)
    return out


def in_square_area(C):
    poly = Polygon(sample_loop(C, 12)).buffer(0)
    return poly.intersection(box(0, 0, 1, 1)).area


def self_intersections(C, per_span=12):
    ls = LineString(np.vstack([sample_loop(C, per_span), sample_loop(C, per_span)[:1]]))
    return 0 if ls.is_simple else 1


def loop_collisions(Cs):
    polys = [Polygon(sample_loop(C, 12)).buffer(0).intersection(box(0, 0, 1, 1)) for C in Cs]
    return sum(1 for i in range(len(polys)) for j in range(i + 1, len(polys)) if polys[i].intersects(polys[j]))


def canonical(Cs, k_max=16):
    """Canonical slot tensor (k_max, 2 n + 1): loops ordered by centroid (u, then v), clockwise, starting at the
    control point of largest u; the last column is the existence score (+1 / -1)."""
    items = []
    for C in Cs:
        C = np.asarray(C, float)
        if signed_area(C) > 0:
            C = C[::-1]
        C = np.roll(C, -int(np.argmax(C[:, 0])), axis=0)
        c = C.mean(0)
        items.append((round(c[0], 3), round(c[1], 3), C))
    items.sort(key=lambda x: (x[0], x[1]))
    n = len(Cs[0]) if len(Cs) else 16
    X = np.zeros((k_max, 2 * n + 1))
    X[:, -1] = -1.0
    for k, (_, _, C) in enumerate(items[:k_max]):
        X[k, :-1] = C.ravel()
        X[k, -1] = 1.0
    return X


In [ ]:
%%writefile trimfields/indicator.py
"""Differentiable material indicator of a loop set (JAX, float64), on CPU or GPU.

Shapes are fixed (active slots bucketed to 1, 2, 4, 8 or 16 loops x n_cp control points; quadrature points padded
to a bucket) and all design data enter as arguments, so a few compiled functions serve every design of a worker.
The all-pairs closest-segment search and the inside test run once per design state without gradient
(`closest`); the indicator is then a function of the distance to the selected segment only (`chi_sel`), whose
reverse-mode derivative costs O(points x loops). This gives the exact gradient almost everywhere.
"""
import math

import jax
import jax.numpy as jnp
import numpy as np
import shapely
from shapely.geometry import Polygon

from .basis import periodic_basis

jax.config.update("jax_enable_x64", True)
EMIN = 1e-6
MAX_CIRCLES = 3
SLOT_BUCKETS = (1, 2, 4, 8, 16)


def heaviside(s, delta):
    x = jnp.clip(s / delta, -1.0, 1.0)
    return 0.5 + 0.75 * x - 0.25 * x**3


def _seg_d2(dx, dy, ex, ey, g11, g12, g22):
    gex, gey = g11 * ex + g12 * ey, g12 * ex + g22 * ey
    t = jnp.clip(-(dx * gex + dy * gey) / (ex * gex + ey * gey + jnp.finfo(dx.dtype).tiny), 0.0, 1.0)
    px, py = dx + t * ex, dy + t * ey
    return g11 * px**2 + 2 * g12 * px * py + g22 * py**2


def _closest(poly, c):
    """Closest segment of a polygon and inside flag for every point (metric distance; no gradient). Runs in float32:
    it only selects a segment and a side, and the distance itself is recomputed in float64 by _dist_sel."""
    f = jnp.float32
    q, g11, g12, g22 = c["q"].astype(f), c["g11"].astype(f)[:, None], c["g12"].astype(f)[:, None], \
        c["g22"].astype(f)[:, None]
    poly = poly.astype(f)
    a, b = poly, jnp.roll(poly, -1, axis=0)
    qx, qy = q[:, 0:1], q[:, 1:2]
    ax, ay, bx, by = a[:, 0][None], a[:, 1][None], b[:, 0][None], b[:, 1][None]
    d2 = _seg_d2(ax - qx, ay - qy, bx - ax, by - ay, g11, g12, g22)
    j = jnp.argmin(d2, axis=1).astype(jnp.int32)
    cond = (ay > qy) != (by > qy)
    xint = ax + (qy - ay) * (bx - ax) / jnp.where(by == ay, f(1.0), by - ay)
    inside = jnp.sum(cond & (qx < xint), axis=1) % 2 == 1
    return j, inside


def _dist_sel(poly, j, inside, c):
    """Signed metric distance to the selected segment (differentiable in the polygon)."""
    q = c["q"]
    M = poly.shape[0]
    A, B = poly[j], poly[(j + 1) % M]
    d2 = _seg_d2(A[:, 0] - q[:, 0], A[:, 1] - q[:, 1], B[:, 0] - A[:, 0], B[:, 1] - A[:, 1],
                 c["g11"], c["g12"], c["g22"])
    dist = jnp.sqrt(d2 + 1e-24)
    return jnp.where(inside, -dist, dist)


def closest(Cz, c):
    polys = jnp.einsum("mn,knd->kmd", c["S"], jax.lax.stop_gradient(Cz))
    return jax.lax.map(lambda p: _closest(p, c), polys)          # sequential over loops: memory O(points x M)


def chi_sel(Cz, act, c, J, INS):
    polys = jnp.einsum("mn,knd->kmd", c["S"], Cz)
    d = jax.vmap(_dist_sel, in_axes=(0, 0, 0, None))(polys, J, INS, c)
    v = (1.0 - heaviside(d, c["delta"])) * act[:, None]
    H = jnp.prod(1.0 - v, axis=0) * c["p_void"]
    return 1.0 - (1.0 - H) * (1.0 - c["p_solid"])


def chi(Cz, act, c):
    J, INS = closest(Cz, c)
    return chi_sel(Cz, act, c, J, INS)


def modulus(Cz, act, c):
    return EMIN + chi(Cz, act, c) * (1.0 - EMIN)


def volume(Cz, act, c):
    return jnp.sum(chi(Cz, act, c) * c["dA"]) / c["area"]


def penalty(Cz, act, c, w):
    """w = (alpha, beta, gamma, w_min) as an array."""
    alpha, beta, gamma, w_min = w[0], w[1], w[2], w[3]
    d1 = jnp.roll(Cz, -1, 1) - Cz
    d2 = jnp.roll(Cz, -1, 1) - 2 * Cz + jnp.roll(Cz, 1, 1)
    Lseg = jnp.sqrt(jnp.sum(d1**2, 2) + 1e-30)
    reg = jnp.sum(d2**2, (1, 2)) / jnp.sum(d1**2, (1, 2)) + jnp.var(Lseg, 1) / jnp.mean(Lseg, 1) ** 2
    cosang = jnp.sum(jnp.roll(d1, 1, 1) * d1, 2) / (jnp.roll(Lseg, 1, 1) * Lseg)
    fold = jnp.sum(jnp.maximum(0.0, math.cos(math.radians(150)) - cosang) ** 2, 1) \
        + jnp.sum(jnp.maximum(0.0, 0.2 - Lseg / jnp.mean(Lseg, 1, keepdims=True)) ** 2, 1)
    P = jnp.einsum("mn,knd->kmd", c["S"][::2], Cz)
    k = Cz.shape[0]
    diff = P[:, None, :, None, :] - P[None, :, None, :, :]
    dist = jnp.sqrt(jnp.sum(diff**2, -1) + 1e-30)
    pair = act[:, None] * act[None, :] * (1.0 - jnp.eye(k))
    clear = 0.5 * jnp.sum(pair[:, :, None, None] * jnp.maximum(0.0, w_min - dist) ** 2) / w_min**2
    dc = jnp.sqrt(jnp.sum((P[:, :, None, :] - c["circ_c"][None, None]) ** 2, -1))        # (k, M, n_circ)
    circ = jnp.sum(act[:, None, None] * c["circ_on"][None, None]
                   * jnp.maximum(0.0, c["circ_r"][None, None] + 0.5 * w_min - dc) ** 2) / w_min**2
    return jnp.sum(act * (alpha * reg + beta * fold)) + gamma * (clear + circ)


closest_jit = jax.jit(closest)
chi_sel_jit = jax.jit(chi_sel)
chi_sel_vjp = jax.jit(lambda Cz, act, c, J, INS, ct: jax.vjp(lambda C: chi_sel(C, act, c, J, INS), Cz)[1](ct)[0])
penalty_vg = jax.jit(jax.value_and_grad(penalty))


def keep_penalty(Cz, act, ck, margin):
    """Loops must stay at least `margin` away from the sample points of the non-design set (signed metric distance,
    so the penalty also pulls back loops that already cover such points deeply)."""
    polys = jnp.einsum("mn,knd->kmd", ck["S"], Cz)
    J, INS = jax.lax.map(lambda p: _closest(p, ck), jax.lax.stop_gradient(polys))
    d = jax.vmap(_dist_sel, in_axes=(0, 0, 0, None))(polys, J, INS, ck)
    return jnp.sum(act[:, None] * jnp.maximum(0.0, margin - d) ** 2) / margin**2


keep_penalty_vg = jax.jit(jax.value_and_grad(keep_penalty))


class Indicator:
    """Builds the constant arrays of one design (points, metric, passive regions) for the jitted functions."""

    def __init__(self, model, prob, n_cp=16, k_max=16, delta_factor=1.25, samples_per_cp=5, bucket=8192,
                 keep_uv=None):
        host = model.host
        self.L = np.array([host.Lu, host.Lv])
        self.m = model.u.size
        self.m_pad = int(math.ceil(self.m / bucket) * bucket)
        G = host.metric(model.u, model.v) / (self.L[None, :, None] * self.L[None, None, :])
        q = np.full((self.m_pad, 2), -1e3)
        q[: self.m] = np.stack([model.u, model.v], 1) * self.L
        g = np.tile(np.eye(2), (self.m_pad, 1, 1))
        g[: self.m] = G
        dA = np.zeros(self.m_pad)
        dA[: self.m] = model.dA
        nq = int(round(math.sqrt(getattr(model, "gpe", 16))))
        self.h = math.sqrt(model.area / model.nel)
        self.delta = delta_factor * self.h / nq
        self.n_cp, self.k_max = n_cp, k_max
        self.dA_np, self.area, self._cache = dA, float(model.area), None
        pv, ps = self._passive(model, prob)
        circ_c, circ_r, circ_on = np.zeros((MAX_CIRCLES, 2)), np.zeros(MAX_CIRCLES), np.zeros(MAX_CIRCLES)
        for i, p in enumerate([p for p in prob["passive"] if p["kind"] == "circle"][:MAX_CIRCLES]):
            circ_c[i], circ_r[i], circ_on[i] = p["centre"], p["radius"] + p["ring"], 1.0
        S = periodic_basis(n_cp, np.arange(samples_per_cp * n_cp) / (samples_per_cp * n_cp))
        self.ck = self._keep_constants(host, keep_uv, S) if keep_uv is not None and len(keep_uv) else None
        self.keep_margin = 0.5 * self.h
        self.c = {k: jnp.asarray(v) for k, v in dict(
            q=q, g11=g[:, 0, 0], g12=g[:, 0, 1], g22=g[:, 1, 1], dA=dA, area=np.float64(model.area),
            delta=np.float64(self.delta), p_void=pv, p_solid=ps, S=S, circ_c=circ_c, circ_r=circ_r,
            circ_on=circ_on).items()}

    def _keep_constants(self, host, keep_uv, S, bucket=256):
        n = len(keep_uv)
        n_pad = int(math.ceil(n / bucket) * bucket)
        q = np.full((n_pad, 2), -1e3)
        q[:n] = np.asarray(keep_uv) * self.L
        g = np.tile(np.eye(2), (n_pad, 1, 1))
        g[:n] = host.metric(np.clip(keep_uv[:, 0], 0, 1), np.clip(keep_uv[:, 1], 0, 1)) / (
            self.L[None, :, None] * self.L[None, None, :])
        return {k: jnp.asarray(v) for k, v in dict(q=q, g11=g[:, 0, 0], g12=g[:, 0, 1], g22=g[:, 1, 1], S=S).items()}

    def keep_penalty(self, Cz, act):
        """Value and gradient of the keep-region penalty (zero without a non-design set)."""
        if self.ck is None:
            return 0.0, np.zeros_like(np.asarray(Cz))
        v, g = keep_penalty_vg(jnp.asarray(Cz), jnp.asarray(act), self.ck, self.keep_margin)
        return float(v), np.asarray(g)

    def _passive(self, model, prob):
        pv, ps = np.ones(self.m_pad), np.zeros(self.m_pad)
        if not prob["passive"]:
            return pv, ps
        X = model.host.eval(model.u, model.v)
        uvL = np.stack([model.u, model.v], 1) * self.L
        for p in prob["passive"]:
            if p["kind"] == "circle":
                d = np.hypot(X[:, 0] - p["centre"][0], X[:, 1] - p["centre"][1])
                pv[: self.m] *= np.asarray(heaviside(d - p["radius"], self.delta))
                ps[: self.m] = np.maximum(ps[: self.m], ((d >= p["radius"]) & (d < p["radius"] + p["ring"])) * 1.0)
            else:
                poly = Polygon(np.asarray(p["uv"]) * self.L)
                s = shapely.distance(poly.exterior, shapely.points(uvL))
                inside = shapely.contains_xy(poly, uvL[:, 0], uvL[:, 1])
                pv[: self.m] *= np.asarray(heaviside(np.where(inside, -s, s), self.delta))
        return pv, ps

    # -- evaluation with active-slot bucketing and a one-state cache ---------------------------------------------
    def state(self, Cz, act):
        """Indicator of a loop set, with the data needed for gradients. Only active slots are evaluated; they are
        padded with parked loops to the next bucket size so that compiled functions are reused."""
        Cz, act = np.asarray(Cz, float), np.asarray(act, float)
        key = hash((Cz.tobytes(), act.tobytes()))
        if self._cache is not None and self._cache["key"] == key:
            return self._cache
        idx = np.flatnonzero(act > 0)
        kb = next(b for b in SLOT_BUCKETS if b >= max(1, len(idx))) if len(idx) <= SLOT_BUCKETS[-1] else len(idx)
        Czb = np.empty((kb, Cz.shape[1], 2))
        Czb[:] = -50.0 + 1e-3 * np.stack([np.cos(np.arange(Cz.shape[1])), np.sin(np.arange(Cz.shape[1]))], 1)
        Czb[: len(idx)] = Cz[idx]
        actb = np.zeros(kb)
        actb[: len(idx)] = 1.0
        Czb, actb = jnp.asarray(Czb), jnp.asarray(actb)
        J, INS = closest_jit(Czb, self.c)
        chi_ = chi_sel_jit(Czb, actb, self.c, J, INS)
        self._cache = {"key": key, "idx": idx, "Czb": Czb, "actb": actb, "J": J, "INS": INS,
                       "chi": np.asarray(chi_), "shape": Cz.shape}
        return self._cache

    def chi_vjp(self, st, ct):
        """Cotangent ct of chi (length m_pad) -> gradient with respect to all k_max x n_cp x 2 loop coordinates."""
        g = np.asarray(chi_sel_vjp(st["Czb"], st["actb"], self.c, st["J"], st["INS"], jnp.asarray(ct)))
        out = np.zeros(st["shape"])
        out[st["idx"]] = g[: len(st["idx"])]
        return out

    def chi(self, Cz, act):
        return self.state(Cz, act)["chi"]

    def modulus(self, Cz, act):
        return EMIN + self.chi(Cz, act) * (1.0 - EMIN)

    def volume(self, Cz, act):
        """Volume fraction and its gradient."""
        st = self.state(Cz, act)
        V = float(np.sum(st["chi"] * self.dA_np) / self.area)
        return V, self.chi_vjp(st, self.dA_np / self.area)


In [ ]:
%%writefile trimfields/refine.py
"""CAD-native refinement R_m: gradient steps on the loop control points plus discrete topology moves."""
import math
import time

import jax.numpy as jnp
import numpy as np
from matplotlib.path import Path
from shapely.geometry import Polygon
from shapely.ops import unary_union

from . import indicator as I
from .loops import fit_periodic, resample_closed, sample_loop
from .problems import covers_keep


class Refiner:
    """Refines a loop set on the model an Indicator was built on; compiled functions are shared module-wide."""

    def __init__(self, model, ind, cfg, keep_geoms=()):
        self.model, self.ind, self.cfg = model, ind, cfg
        self.keep_geoms = list(keep_geoms)            # protected regions (problems.keep_geometry)
        self.w = jnp.asarray([cfg["alpha"], cfg["beta"], cfg["gamma"], cfg["w_min_h"] * ind.h])

    def E(self, Cz, act):
        return self.ind.modulus(Cz, act)

    def vol(self, Cz, act):
        return self.ind.volume(Cz, act)

    def pen(self, Cz, act):
        return I.penalty_vg(jnp.asarray(Cz), jnp.asarray(act), self.ind.c, self.w)

    # -- helpers --------------------------------------------------------------------------------------------------
    def _slots(self, loops_uv):
        k, n = self.ind.k_max, self.ind.n_cp
        Cz = np.zeros((k, n, 2))
        act = np.zeros(k)
        for i, C in enumerate(loops_uv[:k]):
            Cz[i], act[i] = np.asarray(C) * self.ind.L, 1.0
        # inactive slots: tiny loops far outside the domain (no influence on chi)
        for i in range(len(loops_uv), k):
            Cz[i] = -50.0 + 1e-3 * np.stack([np.cos(np.arange(n)), np.sin(np.arange(n))], 1)
        return Cz, act

    def _J_and_grad(self, Cz, act):
        st = self.ind.state(Cz, act)
        E = I.EMIN + st["chi"][: self.ind.m] * (1.0 - I.EMIN)
        J, U = self.model.solve(E)
        ct = np.zeros(self.ind.m_pad)
        ct[: self.ind.m] = -self.model.energy_density(U) * (1.0 - I.EMIN)
        return J, U, self.ind.chi_vjp(st, ct)

    def _loop_area(self, Cz_k):
        poly = sample_loop(Cz_k / self.ind.L, 10)
        inside = Path(poly).contains_points(np.stack([self.model.u, self.model.v], 1))
        return float(self.model.dA[inside].sum())

    # -- topology moves -------------------------------------------------------------------------------------------
    def _topology(self, Cz, act, U, it, maxit, vol, vstar, events):
        el_area = self.model.area / self.model.nel
        n = self.ind.n_cp
        for k in np.flatnonzero(act > 0):                            # deletion
            if self._loop_area(Cz[k]) < 0.5 * el_area:
                act[k] = 0.0
                Cz[k] = -50.0 + 1e-3 * np.stack([np.cos(np.arange(n)), np.sin(np.arange(n))], 1)
                events.append({"it": it, "move": "delete", "slot": int(k)})
        w_min = self.cfg["w_min_h"] * self.ind.h
        ids = list(np.flatnonzero(act > 0))
        polys = {k: Polygon(sample_loop(Cz[k], 10)).buffer(0) for k in ids}
        for a in ids:                                                 # merging of touching loops
            for b in ids:
                if b <= a or act[a] == 0 or act[b] == 0:
                    continue
                if polys[a].distance(polys[b]) < 0.25 * w_min:
                    u = unary_union([polys[a].buffer(0.1 * w_min), polys[b].buffer(0.1 * w_min)])
                    if u.geom_type != "Polygon":
                        continue
                    ring = np.asarray(u.exterior.coords)[:-1]
                    Cz[a] = fit_periodic(resample_closed(ring, 200), n)
                    act[b] = 0.0
                    Cz[b] = -50.0 + 1e-3 * np.stack([np.cos(np.arange(n)), np.sin(np.arange(n))], 1)
                    polys[a] = Polygon(sample_loop(Cz[a], 10)).buffer(0)
                    events.append({"it": it, "move": "merge", "slots": [int(a), int(b)]})
        free = np.flatnonzero(act == 0)
        if (self.cfg["nucleation"] and len(free) and it <= 0.6 * maxit and vol >= vstar - 0.005):
            self._nucleate(Cz, act, U, free[0], it, events)

    def _nucleate(self, Cz, act, U, slot, it, events):
        m = self.model
        E = np.asarray(self.E(Cz, act))[: self.ind.m]
        if m.shell:
            field = m.energy_density(U) / m.dA                        # energy density per unit area
        else:
            s = m.stress(U)
            field = math.pi * (4 * (s[:, 0] ** 2 + s[:, 1] ** 2 + 2 * s[:, 2] ** 2) - (s[:, 0] + s[:, 1]) ** 2)
        uv = np.stack([m.u, m.v], 1)
        ok = (E > 0.99) & np.all((uv > 0.06) & (uv < 0.94), 1)
        zq = uv * self.ind.L
        for k in np.flatnonzero(act > 0):
            poly = sample_loop(Cz[k], 6)
            d = np.min(np.linalg.norm(zq[:, None, :] - poly[None, ::2, :], axis=2), axis=1)
            ok &= d > 3 * self.ind.h
        if ok.sum() < 10:
            return
        ref = np.median(field[E > 0.99])
        i = np.flatnonzero(ok)[np.argmin(field[ok])]
        if field[i] > self.cfg["nucleation_tau"] * ref:
            return
        G = m.host.metric(m.u[i:i + 1], m.v[i:i + 1])[0] / np.outer(self.ind.L, self.ind.L)
        Lc = np.linalg.cholesky(G)                                     # physical length = |L^T dz|
        th = np.linspace(0, 2 * np.pi, 64, endpoint=False)
        r0 = self.cfg["nucleation_radius_h"] * self.ind.h
        circ = zq[i] + (np.linalg.solve(Lc.T, r0 * np.stack([np.cos(th), np.sin(th)]))).T
        Cz[slot] = fit_periodic(circ, self.ind.n_cp)
        act[slot] = 1.0
        events.append({"it": it, "move": "nucleate", "slot": int(slot), "uv": [float(m.u[i]), float(m.v[i])]})

    # -- volume projection and validity ------------------------------------------------------------------------------
    def _project_volume(self, Cz, act, vstar, iters=4):
        """Newton steps along the volume gradient until V = vstar (steps capped at 0.5 h per control point)."""
        for _ in range(iters):
            V, gV = self.vol(Cz, act)
            err = float(V) - vstar
            if abs(err) < 2e-4:
                break
            gVn = np.asarray(gV) * act[:, None, None]
            den = float(np.sum(gVn**2))
            if den <= 0:
                break
            step = err / den * gVn
            nrm = np.linalg.norm(step, axis=2, keepdims=True)
            Cz = Cz - step * np.minimum(1.0, 0.5 * self.ind.h / (nrm + 1e-30))
        return Cz

    def _valid(self, Cz, act):
        from shapely.geometry import LineString
        polys = []
        for k in np.flatnonzero(act > 0):
            P = sample_loop(Cz[k] / self.ind.L, 10)
            if not LineString(np.vstack([P, P[:1]])).is_simple:
                return False
            poly = Polygon(P).buffer(0)
            if self.keep_geoms and covers_keep(poly, self.keep_geoms):
                return False
            polys.append(poly)
        return not any(polys[i].intersects(polys[j]) for i in range(len(polys)) for j in range(i + 1, len(polys)))

    # -- main loop ------------------------------------------------------------------------------------------------
    def run(self, loops_uv, vstar, maxit):
        """Projected gradient: steps tangent to the volume constraint, volume restoration, topology moves, and
        return of the best valid iterate (lowest compliance among admissible designs at V = vstar)."""
        cfg = self.cfg
        h = self.ind.h
        Cz, act = self._slots(loops_uv)
        Cz = self._project_volume(Cz, act, vstar)
        J0, U, _ = self._J_and_grad(Cz, act)
        mom = np.zeros_like(Cz)
        vrms, b1, b2 = 0.0, 0.8, 0.95
        hist, events = [], []
        rejected = 0
        best = None
        t0 = time.perf_counter()
        it = 0
        for it in range(1, maxit + 1):
            lr = h * (cfg["lr1"] + (cfg["lr0"] - cfg["lr1"]) * 0.5 * (1 + math.cos(math.pi * (it - 1) / maxit)))
            J, U, gJ = self._J_and_grad(Cz, act)
            V, gV = self.vol(Cz, act)
            V = float(V)
            if (it % 5 == 1) and abs(V - vstar) < 2e-3 and (best is None or J < best[0]) and self._valid(Cz, act):
                best = (J, Cz.copy(), act.copy(), it, V)
            R, gR = self.pen(Cz, act)
            K, gK = self.ind.keep_penalty(Cz, act)
            g = (gJ / J0 + np.asarray(gR) + cfg.get("gamma_keep", 1.0) * gK) * act[:, None, None]
            gVn = np.asarray(gV) * act[:, None, None]
            den = float(np.sum(gVn**2))
            if den > 0:
                g = g - float(np.sum(g * gVn)) / den * gVn
            mom = b1 * mom + (1 - b1) * g
            gn2 = np.sum(g**2, 2)[act > 0]
            if gn2.size:
                vrms = b2 * vrms + (1 - b2) * float(gn2.mean())
            step = lr * (mom / (1 - b1**it)) / (math.sqrt(vrms / (1 - b2**it)) + 1e-30)
            nrm = np.linalg.norm(step, axis=2, keepdims=True)
            Cz = Cz - step * np.minimum(1.0, 2 * lr / (nrm + 1e-30))
            Cz = self._project_volume(Cz, act, vstar)
            hist.append((it, J, V, float(R)))
            if it % cfg["n_top"] == 0 and it < 0.8 * maxit:
                n_ev = len(events)
                before = (Cz.copy(), act.copy())
                self._topology(Cz, act, U, it, maxit, V, vstar, events)
                if len(events) > n_ev:
                    # keep the moves only if the compliance after volume restoration does not rise
                    J_pre = self._J_and_grad(*before)[0]
                    Cz = self._project_volume(Cz, act, vstar)
                    J_new = self._J_and_grad(Cz, act)[0]
                    if J_new > J_pre * (1.0 + cfg.get("move_accept_tol", 0.0)):
                        Cz, act = before
                        rejected += len(events) - n_ev
                        del events[n_ev:]
                    else:
                        mom[:] = 0.0
            if it >= cfg["min_iters"] and it % 10 == 0 and len(hist) > 20:
                Jold = hist[-21][1]
                if abs(J - Jold) / abs(Jold) < cfg["stop_rel"] and abs(V - vstar) < 2e-3:
                    break
        Jf, _, _ = self._J_and_grad(Cz, act)
        Vf = float(self.vol(Cz, act)[0])
        if abs(Vf - vstar) < 2e-3 and (best is None or Jf < best[0]) and self._valid(Cz, act):
            best = (Jf, Cz.copy(), act.copy(), it, Vf)
        if best is None:                                  # no admissible iterate: keep the final state
            best = (Jf, Cz, act, it, Vf)
        Jb, Cb, ab, it_b, Vb = best
        loops = [Cb[k] / self.ind.L for k in np.flatnonzero(ab > 0)]
        return loops, {"iterations": it, "best_iteration": it_b, "J_smooth_final": float(Jb),
                       "V_smooth_final": float(Vb), "J_smooth_init": float(J0), "events": events,
                       "rejected_moves": rejected,
                       "t_s": time.perf_counter() - t0, "history": np.array(hist, float)}


In [ ]:
%%writefile trimfields/evaluate.py
"""Crisp evaluation with adaptive quadrature on cut elements (finite-cell style), used for every reported number.

Uncut elements: 3 x 3 Gauss points with the element's material state. Elements crossed by the boundary:
4 x 4 sub-cells with 2 x 2 Gauss points each and the crisp material state at every point.
"""
import numpy as np
from matplotlib.path import Path

from .hosts import Host
from .loops import sample_loop
from .mechanics import Model
from .problems import apply_bc
from .simp import density_at

EMIN = 1e-6


def _element_points(Uu, Uv, nsub, nq):
    """Points of every element on an nsub x nsub sub-cell grid with nq x nq Gauss points per sub-cell."""
    ku, kv = np.unique(Uu), np.unique(Uv)
    x, w = np.polynomial.legendre.leggauss(nq)
    loc = (np.arange(nsub)[:, None] + 0.5 * (x[None, :] + 1)) / nsub          # (nsub, nq) in [0, 1]
    wl = (np.ones(nsub)[:, None] * 0.5 * w[None, :]) / nsub
    lu, lv = loc.ravel(), loc.ravel()
    LU, LV = np.meshgrid(lu, lv, indexing="ij")
    WW = np.outer(wl.ravel(), wl.ravel())
    nelu, nelv = len(ku) - 1, len(kv) - 1
    du, dv = np.diff(ku), np.diff(kv)
    u = ku[:-1, None, None, None] + du[:, None, None, None] * LU[None, None]
    v = kv[None, :-1, None, None] + dv[None, :, None, None] * LV[None, None]
    wq = (du[:, None, None, None] * dv[None, :, None, None]) * WW[None, None]
    full = (nelu, nelv) + LU.shape
    e = np.broadcast_to(np.arange(nelu * nelv).reshape(nelu, nelv)[:, :, None, None], full)
    shape = (nelu * nelv, -1)
    return (np.broadcast_to(u, full).reshape(shape), np.broadcast_to(v, full).reshape(shape),
            np.broadcast_to(wq, full).reshape(shape), e.reshape(shape))


class Evaluator:
    def __init__(self, prob, nel, thickness=None, factor=2):
        self.prob = prob
        self.host = Host.from_dict(prob["host"]).build(factor * nel[0], factor * nel[1])
        self.thickness = thickness
        self.base = _element_points(self.host.Uu, self.host.Uv, 1, 3)
        self.fine = _element_points(self.host.Uu, self.host.Uv, 4, 2)
        self.probe = _element_points(self.host.Uu, self.host.Uv, 5, 1)          # 5 x 5 classification samples
        self._fine_dA = None

    # -- material tests -------------------------------------------------------------------------------------------
    def passive_sets(self, u, v):
        """Exact passive geometry at the points: the void of the bolt holes and polygon cut-outs, and the solid of
        the bolt rings (fields.passive_sets, the one rule shared by every material test). Every material state
        becomes (state & ~void) | solid, for loop sets and for thresholded densities alike, so that a bearing load
        always acts on material."""
        from .fields import passive_sets
        return passive_sets(self.prob, self.host, u, v)

    def loop_material(self, loops_uv):
        polys = [Path(sample_loop(np.asarray(C), 20)) for C in loops_uv]

        def fn(u, v):
            pts = np.stack([u, v], 1)
            mat = np.ones(len(u), bool)
            for p in polys:
                mat &= ~p.contains_points(pts)
            void, solid = self.passive_sets(u, v)
            return (mat & ~void) | solid
        return fn

    def density_material(self, rho, th):
        host_opt = self.host_opt                      # bound at creation (threshold_for_volume sets it)

        def fn(u, v):
            void, solid = self.passive_sets(u, v)
            return ((density_at(host_opt, rho, u, v) >= th) & ~void) | solid
        return fn

    # -- evaluation -----------------------------------------------------------------------------------------------
    def evaluate(self, fn, keep_model=False):
        """Crisp compliance and volume fraction of a material test. With keep_model, the result also holds the
        displacement vector and the Model, from which fields at arbitrary points follow (fields.py)."""
        pu, pv, _, _ = self.probe
        state = fn(pu.ravel(), pv.ravel()).reshape(pu.shape)
        full, empty = state.all(1), (~state).all(1)
        cut = ~(full | empty)
        bu, bv, bw, be = self.base
        fu, fv, fw, fe = self.fine
        u = np.concatenate([bu[~cut].ravel(), fu[cut].ravel()])
        v = np.concatenate([bv[~cut].ravel(), fv[cut].ravel()])
        w = np.concatenate([bw[~cut].ravel(), fw[cut].ravel()])
        e = np.concatenate([be[~cut].ravel(), fe[cut].ravel()])
        # no cut element (an untrimmed host whose passive edges fall between the probe samples): skip the test
        mat_cut = fn(fu[cut].ravel(), fv[cut].ravel()) if cut.any() else np.zeros(0, bool)
        mat = np.concatenate([np.repeat(full[~cut], bu.shape[1]), mat_cut])
        model = Model(self.host, u, v, w, e, self.thickness)
        order = np.argsort(e, kind="stable")                  # Model sorts its points by element
        chi = mat[order].astype(float)
        apply_bc(self.prob, model)
        J, U = model.solve(EMIN + chi * (1 - EMIN))
        res = {"J": float(J), "V": float(np.sum(chi * model.dA) / model.area), "cut_elements": int(cut.sum())}
        if keep_model:
            res["U"], res["model"] = U, model
        return res

    def volume(self, fn):
        """Crisp volume fraction of a material test on the fine sub-cell points (no analysis)."""
        fu, fv, fw, _ = self.fine
        if self._fine_dA is None:
            self._fine_dA = self.host.area_element(fu.ravel(), fv.ravel()) * fw.ravel()
        return float(np.sum(self._fine_dA * fn(fu.ravel(), fv.ravel())) / self._fine_dA.sum())

    def threshold_for_volume(self, host_opt, rho, vstar):
        """Density level whose crisp material set has volume fraction vstar (on the fine sub-cell points)."""
        self.host_opt = host_opt
        fu, fv, fw, _ = self.fine
        if self._fine_dA is None:
            self._fine_dA = self.host.area_element(fu.ravel(), fv.ravel()) * fw.ravel()
        r = density_at(host_opt, rho, fu.ravel(), fv.ravel())
        void, solid = self.passive_sets(fu.ravel(), fv.ravel())
        lo, hi = 0.0, 1.0
        for _ in range(40):
            th = 0.5 * (lo + hi)
            V = np.sum(self._fine_dA * (((r >= th) & ~void) | solid)) / self._fine_dA.sum()
            lo, hi = (th, hi) if V > vstar else (lo, th)
        return 0.5 * (lo + hi)


In [ ]:
%%writefile trimfields/conditioning.py
"""Conditioning fields on a 64 x 64 grid of the parameter square (independent of the analysis mesh).

Planar (9 channels): G11, G12, G22, log det J, t_u, t_v, support mask, passive mask, V*.
Shell (13 channels): G11, G12, G22, log sqrt(det G), b11, b12, b22, t_u, t_v, t_n, support mask, passive mask, V*.
Metric channels are divided by the host area and curvature channels by sqrt(area), so a uniform scaling
of the host leaves them unchanged.
"""
import numpy as np
from matplotlib.path import Path

from .hosts import Host
from .mechanics import EDGES

GRID = 64
PLANAR_CHANNELS = ("G11", "G12", "G22", "logdetJ", "t_u", "t_v", "support", "passive", "vstar")
SHELL_CHANNELS = ("G11", "G12", "G22", "logdetG", "b11", "b12", "b22", "t_u", "t_v", "t_n", "support", "passive",
                  "vstar")


def _edge_cells(edge, s0, s1, n=GRID):
    axis, val = EDGES[edge]
    idx = np.arange(n)
    c = (idx + 0.5) / n
    sel = idx[(c >= s0 - 0.5 / n) & (c <= s1 + 0.5 / n)]
    if sel.size == 0:
        sel = np.array([int(np.clip(0.5 * (s0 + s1) * n, 0, n - 1))])
    fixed = 0 if val == 0.0 else n - 1
    return (np.full_like(sel, fixed), sel) if axis == 0 else (sel, np.full_like(sel, fixed))


def conditioning(prob, n=GRID):
    host = Host.from_dict(prob["host"]).build(8, 8)
    c = (np.arange(n) + 0.5) / n
    U, V = np.meshgrid(c, c, indexing="ij")
    u, v = U.ravel(), V.ravel()
    X, J = host.jac(u, v)
    G = np.einsum("mdi,mdj->mij", J, J)
    A = host.area
    shell = host.dim == 3
    ch = {"G11": G[:, 0, 0] / A, "G12": G[:, 0, 1] / A, "G22": G[:, 1, 1] / A}
    detG = np.linalg.det(G)
    if shell:
        ch["logdetG"] = 0.5 * np.log(detG / A**2)
        B = host.curvature(u, v) / np.sqrt(A)                         # b_ab has units of length
        ch["b11"], ch["b12"], ch["b22"] = B[:, 0, 0], B[:, 0, 1], B[:, 1, 1]
    else:
        ch["logdetJ"] = 0.5 * np.log(detG / A**2)
    shape = (n, n)
    t = np.zeros((n, n, 3))
    sup = np.zeros(shape)
    pas = np.zeros(shape)
    for ld in prob["loads"]:
        if ld["kind"] == "edge":
            iu, iv = _edge_cells(ld["edge"], ld["s0"], ld["s1"], n)
            k = iu * n + iv
            d = np.asarray(ld["dir"], float)
            tp = np.linalg.solve(J[k][:, :2, :], d[None, :, None])[..., 0]           # J^{-1} t (planar)
            tp /= np.linalg.norm(tp, axis=1, keepdims=True)
            t[iu, iv, :2] += ld["mag"] * tp
        elif ld["kind"] == "ring":                                          # bolt bearing load (problems.bearing_load)
            ps = prob["passive"][ld["passive"]]
            dx, dy = X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1]
            dd = np.hypot(dx, dy).reshape(shape)
            ring = (dd >= ps["radius"]) & (dd < ps["radius"] + ps["ring"])
            k = np.flatnonzero(ring.ravel())
            if k.size:
                er = np.stack([dx[k], dy[k]], 1) / np.maximum(dd.ravel()[k], 1e-300)[:, None]
                cosw = np.maximum(er @ np.asarray(ld["dir"], float), 0.0)       # cosine law on the loaded half
                tp = np.linalg.solve(J[k], er[..., None])[..., 0]
                tp /= np.linalg.norm(tp, axis=1, keepdims=True)
                t.reshape(-1, 3)[k, :2] += ld["mag"] * cosw[:, None] * tp
        else:                                                               # shell surface load
            mask = np.ones(n * n, bool)
            if ld["centre"] is not None:
                mask = (u - ld["centre"][0]) ** 2 + (v - ld["centre"][1]) ** 2 <= ld["radius"] ** 2
                if not mask.any():
                    mask[np.argmin((u - ld["centre"][0]) ** 2 + (v - ld["centre"][1]) ** 2)] = True
            nrm = np.cross(J[:, :, 0], J[:, :, 1])
            nrm /= np.linalg.norm(nrm, axis=1, keepdims=True)
            f = nrm if isinstance(ld["dir"], str) else np.broadcast_to(np.asarray(ld["dir"], float), (n * n, 3))
            tn = np.sum(f * nrm, 1)
            ft = f - tn[:, None] * nrm
            tt = np.linalg.solve(G, np.einsum("mdi,md->mi", J, ft)[..., None])[..., 0]
            vals = np.stack([tt[:, 0], tt[:, 1], tn], 1) * mask[:, None]
            t.reshape(-1, 3)[:] += ld["mag"] * vals
    for s in prob["supports"]:
        if s["kind"] == "ring":
            ps = prob["passive"][s["passive"]]
            dd = np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1]).reshape(shape)
            sup[(dd >= ps["radius"]) & (dd < ps["radius"] + ps["ring"])] = 1.0
        else:
            iu, iv = _edge_cells(s["edge"], s["s0"], s["s1"], n)
            sup[iu, iv] = {"all": 1.0, "clamped": 1.0, "pinned": 0.5, "normal": 0.5}[s["dofs"]]
    for ps in prob["passive"]:
        if ps["kind"] == "circle":
            dd = np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1]).reshape(shape)
            pas[dd < ps["radius"]] = -1.0
            pas[(dd >= ps["radius"]) & (dd < ps["radius"] + ps["ring"])] = 1.0
        else:
            pas.ravel()[Path(np.asarray(ps["uv"])).contains_points(np.stack([u, v], 1))] = -1.0
    ch["t_u"], ch["t_v"] = t[..., 0].ravel(), t[..., 1].ravel()
    if shell:
        ch["t_n"] = t[..., 2].ravel()
    ch["support"], ch["passive"] = sup.ravel(), pas.ravel()
    ch["vstar"] = np.full(n * n, prob["vstar"])
    names = SHELL_CHANNELS if shell else PLANAR_CHANNELS
    return np.stack([ch[k].reshape(shape) for k in names]).astype(np.float32)


In [ ]:
%%writefile trimfields/fields.py
"""Reference fields stored with every design (new in TrimFields).

- stress and energy density of the isogeometric solution at arbitrary parameter points, in particular along the
  trimming loops (the cut-cell boundary stress that body-fitted and neural solvers are compared with);
- the design fields on the conditioning grid: the crisp material indicator and the signed distance to the
  trimming curves, which are the inputs of a design-conditioned neural solver.
"""
import numpy as np
from matplotlib.path import Path

from .basis import periodic_basis
from .hosts import Host
from .loops import sample_loop, signed_area

GRID = 64


# ------------------------------------------------------------------------------------------------ fields at points
def element_index(host, u, v):
    """Element index (e = iu * nelv + iv, as in basis.gauss_grid) of parameter points."""
    ku, kv = np.unique(host.Uu), np.unique(host.Uv)
    iu = np.clip(np.searchsorted(ku, u, side="right") - 1, 0, len(ku) - 2)
    iv = np.clip(np.searchsorted(kv, v, side="right") - 1, 0, len(kv) - 2)
    return iu * (len(kv) - 1) + iv


def _von_mises(s):
    return np.sqrt(np.maximum(s[:, 0] ** 2 - s[:, 0] * s[:, 1] + s[:, 1] ** 2 + 3.0 * s[:, 2] ** 2, 0.0))


def stress_energy_at(model, U, u, v):
    """Von Mises stress (unit modulus; for shells the larger value of the two outer surfaces) and the energy density
    w = eps:D:eps per unit area (membrane plus bending for shells) of the solution U at parameter points."""
    u = np.clip(np.asarray(u, float), 1e-12, 1 - 1e-12)
    v = np.clip(np.asarray(v, float), 1e-12, 1 - 1e-12)
    e = element_index(model.host, u, v)
    if model.shell:
        Bm, Bb, _ = model._shell_B(u, v, e)
        terms = [(Bm, model.thickness), (Bb, model.thickness**3 / 12.0)]
    else:
        B, _ = model._plane_B(u, v, e)
        terms = [(B, 1.0)]
    ue = U[model.edof][e]
    eps = [np.einsum("mik,mk->mi", B, ue) for B, _ in terms]
    sig = [np.einsum("ij,mj->mi", model.D0, ep) for ep in eps]
    w = sum(f * np.sum(ep * sg, 1) for (_, f), ep, sg in zip(terms, eps, sig))
    if model.shell:
        z = 0.5 * model.thickness
        vm = np.maximum(_von_mises(sig[0] + z * sig[1]), _von_mises(sig[0] - z * sig[1]))
    else:
        vm = _von_mises(sig[0])
    return vm, w


def loop_samples(model, U, loops_uv, n=64, prob=None):
    """n samples per loop: parameter point, physical point, unit normal pointing into the hole, arc-length weight,
    von Mises stress, energy density and a flag for samples on a free boundary of the face: inside the parameter
    square and, when `prob` is given, outside every passive region (loop parts beyond the square or inside a bolt
    hole, a ring or a cut-out bound no material). Arrays of shape (K, n, ...)."""
    host = model.host
    K, dim = len(loops_uv), host.dim
    out = {"uv": np.zeros((K, n, 2)), "x": np.zeros((K, n, dim)), "normal": np.zeros((K, n, dim)),
           "ds": np.zeros((K, n)), "vm": np.zeros((K, n)), "w": np.zeros((K, n)), "inside": np.zeros((K, n), bool)}
    r = np.arange(n) / n
    for k, C in enumerate(loops_uv):
        C = np.asarray(C, float)
        Bp, Bd = periodic_basis(len(C), r), periodic_basis(len(C), r, der=1)
        uv, tuv = Bp @ C, Bd @ C
        inside = np.all((uv >= 0.0) & (uv <= 1.0), 1)
        uvc = np.clip(uv, 1e-9, 1 - 1e-9)
        if prob is not None:
            inside &= ~in_passive(prob, host, uvc[:, 0], uvc[:, 1])
        _, J = host.jac(uvc[:, 0], uvc[:, 1])
        X = host.eval(uvc[:, 0], uvc[:, 1])
        t = np.einsum("mdi,mi->md", J, tuv)
        tn = np.linalg.norm(t, axis=1, keepdims=True)
        t = t / np.maximum(tn, 1e-300)
        clockwise = signed_area(sample_loop(C, 10)) < 0           # interior on the right of the direction of travel
        n_uv = np.stack([tuv[:, 1], -tuv[:, 0]], 1) if clockwise else np.stack([-tuv[:, 1], tuv[:, 0]], 1)
        nrm = np.einsum("mdi,mi->md", J, n_uv)
        nrm = nrm - np.sum(nrm * t, 1, keepdims=True) * t          # in the tangent plane, orthogonal to the curve
        nrm = nrm / np.maximum(np.linalg.norm(nrm, axis=1, keepdims=True), 1e-300)
        vm, w = stress_energy_at(model, U, uvc[:, 0], uvc[:, 1])
        out["uv"][k], out["x"][k], out["normal"][k] = uv, X, nrm
        out["ds"][k], out["vm"][k], out["w"][k], out["inside"][k] = tn[:, 0] / n, vm, w, inside
    return out


# ------------------------------------------------------------------------------------------------ design fields
def _segment_distance(P, Q):
    """Distance from points P (M, 2) to the closed polyline Q (S, 2)."""
    A, B = Q, np.roll(Q, -1, axis=0)
    d = np.full(len(P), np.inf)
    for s in range(0, len(A), 64):
        a, b = A[s:s + 64], B[s:s + 64]
        ab = b - a
        ap = P[:, None, :] - a[None, :, :]
        t = np.clip(np.sum(ap * ab[None], 2) / np.maximum(np.sum(ab * ab, 1)[None], 1e-300), 0.0, 1.0)
        foot = a[None] + t[..., None] * ab[None]
        d = np.minimum(d, np.min(np.linalg.norm(P[:, None, :] - foot, axis=2), 1))
    return d


def passive_sets(prob, host, u, v):
    """Exact passive geometry at parameter points: the void of the bolt holes and polygon cut-outs, and the solid
    of the bolt rings. Every material state is composed as (state & ~void) | solid; evaluate.Evaluator.passive_sets
    delegates here, so that all material tests share one rule."""
    void, solid = np.zeros(len(u), bool), np.zeros(len(u), bool)
    if prob["passive"] and len(u):                           # the host basis cannot be evaluated at no points
        pts = np.stack([u, v], 1)
        X = host.eval(u, v)
        for ps in prob["passive"]:
            if ps["kind"] == "circle":
                d = np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1])
                void |= d < ps["radius"]
                solid |= (d >= ps["radius"]) & (d < ps["radius"] + ps["ring"])
            else:
                void |= Path(np.asarray(ps["uv"])).contains_points(pts)
    return void, solid


def in_passive(prob, host, u, v):
    """Points inside a passive region (hole, ring or cut-out): a loop there bounds no free boundary of the face."""
    void, solid = passive_sets(prob, host, u, v)
    return void | solid


def material_mask(prob, host, u, v, loops_uv):
    """Crisp material test on parameter points, including the passive regions (as in evaluate.loop_material)."""
    pts = np.stack([u, v], 1)
    mat = np.ones(len(u), bool)
    for C in loops_uv:
        mat &= ~Path(sample_loop(np.asarray(C), 20)).contains_points(pts)
    void, solid = passive_sets(prob, host, u, v)
    return (mat & ~void) | solid


def design_field(prob, loops_uv, n=GRID):
    """(2, n, n) float32: crisp indicator (+1 material, -1 void) at the cell centres of the conditioning grid, and
    the signed distance to the nearest trimming curve or passive boundary, measured in the scaled coordinates
    z = (Lu u, Lv v) of the host (physical lengths to first order), divided by sqrt(area), positive in material."""
    host = Host.from_dict(prob["host"]).build(8, 8)
    c = (np.arange(n) + 0.5) / n
    U, V = np.meshgrid(c, c, indexing="ij")
    u, v = U.ravel(), V.ravel()
    L = np.array([host.Lu, host.Lv])
    mat = material_mask(prob, host, u, v, loops_uv)
    P = np.stack([u, v], 1) * L
    d = np.full(len(u), np.inf)
    for C in loops_uv:
        d = np.minimum(d, _segment_distance(P, sample_loop(np.asarray(C), 20) * L))
    for ps in prob["passive"]:
        if ps["kind"] == "circle":                               # rectangle host: z equals the physical point
            r = np.hypot(P[:, 0] - ps["centre"][0], P[:, 1] - ps["centre"][1])
            d = np.minimum(d, np.abs(r - ps["radius"]))
        else:
            d = np.minimum(d, _segment_distance(P, np.asarray(ps["uv"], float) * L))
    d = np.where(np.isfinite(d), d, np.sqrt(host.area))
    sdf = np.where(mat, d, -d) / np.sqrt(host.area)
    return np.stack([np.where(mat, 1.0, -1.0).reshape(n, n), sdf.reshape(n, n)]).astype(np.float32)


In [ ]:
%%writefile trimfields/export_step.py
"""STEP export of a trimmed design: the host NURBS surface with the loops as pcurves (no approximation of loops).

Holes are cut in the parameter square by Booleans; every resulting edge is then rebuilt as a curve on the host
surface. Planar hosts are written as faces in the plane z = 0, shells as faces on their 3D surface.
"""
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot  # noqa: E402,F401  (import before OCP: OCP's libexpat otherwise breaks pyexpat)
import numpy as np  # noqa: E402

try:
    from OCP.BRep import BRep_Builder
    from OCP.BRepAdaptor import BRepAdaptor_Curve
    from OCP.BRepAlgoAPI import BRepAlgoAPI_Cut
    from OCP.BRepBuilderAPI import (BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeFace, BRepBuilderAPI_MakePolygon,
                                    BRepBuilderAPI_MakeWire)
    from OCP.BRepCheck import BRepCheck_Analyzer
    from OCP.BRepGProp import BRepGProp
    from OCP.BRepLib import BRepLib
    from OCP.BRepTools import BRepTools, BRepTools_WireExplorer
    from OCP.GC import GC_MakeEllipse
    from OCP.Geom import Geom_BSplineCurve, Geom_BSplineSurface
    from OCP.GeomAPI import GeomAPI
    from OCP.GProp import GProp_GProps
    from OCP.IFSelect import IFSelect_RetDone
    from OCP.ShapeFix import ShapeFix_Shape
    from OCP.STEPControl import STEPControl_AsIs, STEPControl_Reader, STEPControl_Writer
    try:                                                              # OCP 7.x
        from OCP.TColgp import TColgp_Array1OfPnt, TColgp_Array2OfPnt
        from OCP.TColStd import TColStd_Array1OfInteger, TColStd_Array1OfReal, TColStd_Array2OfReal
    except ImportError:                                               # OCP 8.x
        from OCP.collections import Array1_double as TColStd_Array1OfReal
        from OCP.collections import Array1_gp_Pnt as TColgp_Array1OfPnt
        from OCP.collections import Array1_int as TColStd_Array1OfInteger
        from OCP.collections import Array2_double as TColStd_Array2OfReal
        from OCP.collections import Array2_gp_Pnt as TColgp_Array2OfPnt
    from OCP.TopAbs import TopAbs_FACE, TopAbs_REVERSED, TopAbs_WIRE
    from OCP.TopExp import TopExp_Explorer
    from OCP.TopoDS import TopoDS, TopoDS_Compound
    from OCP.gp import gp_Ax2, gp_Ax3, gp_Dir, gp_Elips, gp_Pln, gp_Pnt
    HAVE_OCP = True
except Exception:  # noqa: BLE001
    HAVE_OCP = False


def _knots(U):
    U = np.asarray(U, float)
    uk, mult = np.unique(np.round(U, 12), return_counts=True)
    k, m = TColStd_Array1OfReal(1, len(uk)), TColStd_Array1OfInteger(1, len(uk))
    for i, (a, b) in enumerate(zip(uk, mult)):
        k.SetValue(i + 1, float(a))
        m.SetValue(i + 1, int(b))
    return k, m


def host_surface(hd):
    P, W = np.asarray(hd["P"], float), np.asarray(hd["W"], float)
    ncu, ncv = W.shape
    poles, wts = TColgp_Array2OfPnt(1, ncu, 1, ncv), TColStd_Array2OfReal(1, ncu, 1, ncv)
    for i in range(ncu):
        for j in range(ncv):
            p = P[i, j]
            poles.SetValue(i + 1, j + 1, gp_Pnt(float(p[0]), float(p[1]), float(p[2]) if len(p) > 2 else 0.0))
            wts.SetValue(i + 1, j + 1, float(W[i, j]))
    ku, mu = _knots(hd["Uu"])
    kv, mv = _knots(hd["Uv"])
    return Geom_BSplineSurface(poles, wts, ku, kv, mu, mv, 2, 2)


def _periodic_curve(C):
    n = len(C)
    poles = TColgp_Array1OfPnt(1, n)
    for i, p in enumerate(C):
        poles.SetValue(i + 1, gp_Pnt(float(p[0]), float(p[1]), 0.0))
    knots, mults = TColStd_Array1OfReal(1, n + 1), TColStd_Array1OfInteger(1, n + 1)
    for i in range(n + 1):
        knots.SetValue(i + 1, i / n)
        mults.SetValue(i + 1, 1)
    return Geom_BSplineCurve(poles, knots, mults, 3, True)


def _face_of_edge(edge):
    return BRepBuilderAPI_MakeFace(BRepBuilderAPI_MakeWire(edge).Wire(), True).Face()


def _passive_face(ps, hd):
    if ps["kind"] == "polygon":
        poly = BRepBuilderAPI_MakePolygon()
        for p in ps["uv"]:
            poly.Add(gp_Pnt(float(p[0]), float(p[1]), 0.0))
        poly.Close()
        return BRepBuilderAPI_MakeFace(poly.Wire(), True).Face()
    L, H = hd["params"]["L"], hd["params"]["H"]                    # rectangle host: circle -> exact ellipse in (u, v)
    cu, cv = ps["centre"][0] / L, ps["centre"][1] / H
    ru, rv = ps["radius"] / L, ps["radius"] / H
    xdir = gp_Dir(1, 0, 0) if ru >= rv else gp_Dir(0, 1, 0)
    el = gp_Elips(gp_Ax2(gp_Pnt(cu, cv, 0), gp_Dir(0, 0, 1), xdir), max(ru, rv), min(ru, rv))
    return _face_of_edge(BRepBuilderAPI_MakeEdge(GC_MakeEllipse(el).Value()).Edge())


def _faces(sh):
    ex, out = TopExp_Explorer(sh, TopAbs_FACE), []
    while ex.More():
        out.append(TopoDS.Face_s(ex.Current()) if hasattr(TopoDS, "Face_s") else TopoDS.Face(ex.Current()))
        ex.Next()
    return out


def _as_wire(x):
    return TopoDS.Wire_s(x) if hasattr(TopoDS, "Wire_s") else TopoDS.Wire(x)


def _area(sh):
    pr = GProp_GProps()
    BRepGProp.SurfaceProperties_s(sh, pr, 1e-9)
    return abs(pr.Mass())


class _Quiet:
    """Redirect the C-level stdout (OpenCASCADE transfer statistics) to /dev/null."""

    def __enter__(self):
        import sys
        sys.stdout.flush()
        self.fd = os.dup(1)
        self.null = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self.null, 1)

    def __exit__(self, *exc):
        try:
            import ctypes
            ctypes.CDLL(None).fflush(None)                              # flush C stdio while redirected
        except Exception:  # noqa: BLE001
            pass
        os.dup2(self.fd, 1)
        os.close(self.fd)
        os.close(self.null)


def export_design(prob, loops_uv, path):
    """Write the trimmed face(s) to STEP, read them back and check validity and area."""
    with _Quiet():
        return _export(prob, loops_uv, path)


def _export(prob, loops_uv, path):
    hd = prob["host"]
    pln = gp_Pln(gp_Ax3(gp_Pnt(0, 0, 0), gp_Dir(0, 0, 1), gp_Dir(1, 0, 0)))
    sq = BRepBuilderAPI_MakeFace(pln, 0.0, 1.0, 0.0, 1.0).Face()
    for C in loops_uv:
        sq = BRepAlgoAPI_Cut(sq, _face_of_edge(BRepBuilderAPI_MakeEdge(_periodic_curve(C)).Edge())).Shape()
    for ps in prob["passive"]:
        sq = BRepAlgoAPI_Cut(sq, _passive_face(ps, hd)).Shape()
    S = host_surface(hd)
    comp, bb = TopoDS_Compound(), BRep_Builder()
    bb.MakeCompound(comp)
    n_faces = n_edges = 0
    for f in _faces(sq):
        outer = BRepTools.OuterWire_s(f)
        wires, exw = [], TopExp_Explorer(f, TopAbs_WIRE)
        while exw.More():
            wires.append(_as_wire(exw.Current()))
            exw.Next()
        wires.sort(key=lambda w_: 0 if w_.IsSame(outer) else 1)
        built = []
        for wr in wires:
            mw, we = BRepBuilderAPI_MakeWire(), BRepTools_WireExplorer(wr, f)
            while we.More():
                e = we.Current()
                ad = BRepAdaptor_Curve(e)
                c2d = GeomAPI.To2d_s(ad.Curve().Curve(), pln)
                ne = BRepBuilderAPI_MakeEdge(c2d, S, ad.FirstParameter(), ad.LastParameter()).Edge()
                if e.Orientation() == TopAbs_REVERSED:
                    ne.Reverse()
                mw.Add(ne)
                n_edges += 1
                we.Next()
            built.append(mw.Wire())
        mf = BRepBuilderAPI_MakeFace(S, built[0], True)
        for wr in built[1:]:
            mf.Add(wr)
        face = mf.Face()
        BRepLib.BuildCurves3d_s(face)
        fix = ShapeFix_Shape(face)
        fix.Perform()
        bb.Add(comp, fix.Shape())
        n_faces += 1
    os.makedirs(os.path.dirname(path), exist_ok=True)
    w = STEPControl_Writer()
    w.Transfer(comp, STEPControl_AsIs)
    w.Write(path)
    r = STEPControl_Reader()
    ok = r.ReadFile(path) == IFSelect_RetDone
    r.TransferRoots()
    back = r.OneShape()
    return {"faces": n_faces, "pcurve_edges": n_edges, "valid": bool(ok and BRepCheck_Analyzer(back).IsValid()),
            "area": _area(back), "bytes": os.path.getsize(path)}


In [ ]:
%%writefile trimfields/bodyfit.py
"""Body-fitted finite-element reference for planar designs (optional; requires gmsh).

The exported STEP face is meshed with quadratic triangles that conform to the trimming curves, and the plane-stress
problem is solved with the continuum boundary conditions of problems.py. This solution does not share the cut-cell
and soft-void errors of the immersed evaluator, so it serves as the reference of the boundary-stress comparison.

    python -m trimfields.bodyfit <design.npz> <design.step> [--size h] [--out reference.npz]

The result holds the mesh, the displacement, the compliance, the volume fraction and the von Mises stress at the
loop samples stored in the design (loop_x), interpolated along the conforming boundary.
"""
import argparse
import json

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu

from .hosts import Host
from .mechanics import EDGES, plane_stress_D

# 6-point Gauss rule on the reference triangle (degree 4); weights sum to 1/2
_A, _B = 0.445948490915965, 0.091576213509771
_WA, _WB = 0.223381589678011 / 2, 0.109951743655322 / 2
GAUSS = np.array([[_A, _A, _WA], [1 - 2 * _A, _A, _WA], [_A, 1 - 2 * _A, _WA],
                  [_B, _B, _WB], [1 - 2 * _B, _B, _WB], [_B, 1 - 2 * _B, _WB]])
G3, W3 = np.array([-np.sqrt(0.6), 0.0, np.sqrt(0.6)]), np.array([5.0, 8.0, 5.0]) / 9.0


def shape_p2(xi, eta):
    """Quadratic triangle (gmsh node order: corners 0-2, mid-sides 3: 0-1, 4: 1-2, 5: 2-0): N (6,), dN (6, 2)."""
    l1, l2, l3 = 1 - xi - eta, xi, eta
    N = np.array([l1 * (2 * l1 - 1), l2 * (2 * l2 - 1), l3 * (2 * l3 - 1), 4 * l1 * l2, 4 * l2 * l3, 4 * l3 * l1])
    dN = np.array([[1 - 4 * l1, 1 - 4 * l1], [4 * l2 - 1, 0.0], [0.0, 4 * l3 - 1],
                   [4 * (l1 - l2), -4 * l2], [4 * l3, 4 * l2], [-4 * l3, 4 * (l1 - l3)]])
    return N, dN


def shape_edge(s):
    """Quadratic edge (nodes: end 0, end 1, middle): N (3,), dN/ds (3,) for s in [-1, 1]."""
    return (np.array([0.5 * s * (s - 1), 0.5 * s * (s + 1), 1 - s * s]),
            np.array([s - 0.5, s + 0.5, -2 * s]))


def mesh_face(step_path, size, order=2):
    """Conforming quadratic triangulation of the STEP face(s) with gmsh: nodes (n, 2), triangles (m, 6)."""
    import gmsh
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 0)
        gmsh.model.occ.importShapes(step_path)
        gmsh.model.occ.synchronize()
        gmsh.option.setNumber("Mesh.MeshSizeMin", 0.5 * size)
        gmsh.option.setNumber("Mesh.MeshSizeMax", size)
        gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 24)
        gmsh.option.setNumber("Mesh.ElementOrder", order)
        gmsh.option.setNumber("Mesh.Algorithm", 6)
        gmsh.model.mesh.generate(2)
        tags, coords, _ = gmsh.model.mesh.getNodes()
        nodes = coords.reshape(-1, 3)[:, :2]
        renum = {int(t): i for i, t in enumerate(tags)}
        etypes, _, enodes = gmsh.model.mesh.getElements(2)
        tris = []
        for et, en in zip(etypes, enodes):
            if et == 9:                                           # 6-node triangles
                tris.append(np.vectorize(renum.get)(np.asarray(en).reshape(-1, 6)))
        return nodes, np.concatenate(tris)
    finally:
        gmsh.finalize()


def _edge_curve(host, edge, n=800):
    axis, val = EDGES[edge]
    s = np.linspace(0, 1, n)
    u, v = (np.full(n, val), s) if axis == 0 else (s, np.full(n, val))
    return s, host.eval(u, v)


def _nearest_on_curve(P, s, Xc):
    """Nearest point of a polyline (s, Xc) for every point of P: distance and curve parameter."""
    d = np.full(len(P), np.inf)
    sp_ = np.zeros(len(P))
    A, B, sa, sb = Xc[:-1], Xc[1:], s[:-1], s[1:]
    for k in range(0, len(A), 100):
        a, b = A[k:k + 100], B[k:k + 100]
        ab = b - a
        t = np.clip(np.sum((P[:, None, :] - a[None]) * ab[None], 2) / np.maximum(np.sum(ab * ab, 1)[None], 1e-300),
                    0, 1)
        foot = a[None] + t[..., None] * ab[None]
        dist = np.linalg.norm(P[:, None, :] - foot, axis=2)
        j = np.argmin(dist, 1)
        better = dist[np.arange(len(P)), j] < d
        d[better] = dist[np.arange(len(P)), j][better]
        sp_[better] = (sa[k:k + 100][j] + t[np.arange(len(P)), j] * (sb[k:k + 100] - sa[k:k + 100])[j])[better]
    return d, sp_


def boundary_edges(tris):
    """Boundary edges of the quadratic triangulation as (end, end, middle) node triples."""
    edges = {}
    for tri in tris:
        for a, b, m in ((0, 1, 3), (1, 2, 4), (2, 0, 5)):
            key = (min(tri[a], tri[b]), max(tri[a], tri[b]))
            edges[key] = edges.get(key, 0) + 1
            edges.setdefault(("mid",) + key, (tri[a], tri[b], tri[m]))
    return [edges[("mid",) + k] for k, c in edges.items() if k[0] != "mid" and c == 1]


def solve_face(prob, nodes, tris, tol=None):
    """Plane-stress solution on the conforming mesh with the continuum boundary conditions of the problem."""
    host = Host.from_dict(prob["host"]).build(8, 8)
    n = len(nodes)
    diag = np.linalg.norm(nodes.max(0) - nodes.min(0))
    tol = 1e-5 * diag if tol is None else tol
    D = plane_stress_D()
    # assembly
    rows, cols, vals = [], [], []
    area = 0.0
    for tri in tris:
        X = nodes[tri]
        Ke = np.zeros((12, 12))
        for xi, eta, wq in GAUSS:
            N, dN = shape_p2(xi, eta)
            Jm = dN.T @ X                                        # (2, 2): Jm[i, j] = d x_j / d xi_i
            detJ = abs(np.linalg.det(Jm))                        # either orientation of the imported face
            dNx = dN @ np.linalg.inv(Jm).T                       # (6, 2): dN / dx_j = dN / dxi_i * dxi_i / dx_j
            B = np.zeros((3, 12))
            B[0, 0::2], B[1, 1::2] = dNx[:, 0], dNx[:, 1]
            B[2, 0::2], B[2, 1::2] = dNx[:, 1], dNx[:, 0]
            Ke += B.T @ D @ B * detJ * wq
            area += detJ * wq
        dof = np.stack([2 * tri, 2 * tri + 1], 1).ravel()
        rows.append(np.repeat(dof, 12))
        cols.append(np.tile(dof, 12))
        vals.append(Ke.ravel())
    K = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(2 * n, 2 * n))
    # boundary conditions
    bedges = boundary_edges(tris)
    F = np.zeros(2 * n)
    fixed = set()
    edge_data = {e: _edge_curve(host, e) for e in EDGES}
    for s_ in prob["supports"]:
        if s_["kind"] == "ring":
            ps = prob["passive"][s_["passive"]]
            r = np.hypot(nodes[:, 0] - ps["centre"][0], nodes[:, 1] - ps["centre"][1])
            for i in np.flatnonzero(r <= ps["radius"] + ps["ring"] + tol):
                fixed.update((2 * i, 2 * i + 1))
            continue
        d, sv = _nearest_on_curve(nodes, *edge_data[s_["edge"]])
        on = (d < tol) & (sv >= s_["s0"] - 1e-9) & (sv <= s_["s1"] + 1e-9)
        comps = (int(s_["fixed_component"]),) if s_["dofs"] == "normal" else (0, 1)
        for i in np.flatnonzero(on):
            fixed.update(2 * i + c for c in comps)
    for ld in prob["loads"]:
        Fl = np.zeros(2 * n)
        if ld["kind"] == "edge":
            s, Xc = edge_data[ld["edge"]]
            dvec = np.asarray(ld["dir"], float) / np.linalg.norm(ld["dir"])
            d, sv = _nearest_on_curve(nodes, s, Xc)
            for a, b, m in bedges:                           # mesh edges on the host edge, integrated only over
                if all(d[i] < tol for i in (a, b, m)):       # the part of the segment [s0, s1] (Gauss points)
                    se = sv[[a, b, m]]

                    def uniform(x, g, se=se, dvec=dvec, ld=ld):
                        sg = shape_edge(g)[0] @ se                    # host-edge parameter of the Gauss point
                        return dvec if ld["s0"] <= sg <= ld["s1"] else np.zeros(2)
                    _edge_force(Fl, nodes, (a, b, m), uniform)
        elif ld["kind"] == "ring":
            ps = prob["passive"][ld["passive"]]
            c, r = np.asarray(ps["centre"], float), ps["radius"]
            dvec = np.asarray(ld["dir"], float) / np.linalg.norm(ld["dir"])
            p0 = 2 * ld["mag"] / (np.pi * r)

            def bearing(x, g, c=c, r=r, dvec=dvec, p0=p0):
                er = (x - c) / max(np.linalg.norm(x - c), 1e-300)
                return p0 * max(er @ dvec, 0.0) * er
            rr = np.hypot(nodes[:, 0] - c[0], nodes[:, 1] - c[1])
            for a, b, m in bedges:
                if all(abs(rr[i] - r) < tol for i in (a, b, m)):
                    _edge_force(Fl, nodes, (a, b, m), bearing)
        res = np.linalg.norm([Fl[0::2].sum(), Fl[1::2].sum()])
        if res <= 0:
            raise RuntimeError(f"no boundary edge found for the load {ld}")
        F += Fl * (ld["mag"] / res)                            # exact resultant (quadrature of the segment ends)
    # floating islands: mesh components without a supported node (slivers of material cut off by a loop that
    # protrudes beyond the square) carry rigid-body modes; their nodes are fixed and reported so that samples on
    # them can be excluded. An island that carries load has no body-fitted solution (broken load path).
    from scipy.sparse.csgraph import connected_components
    adj = sp.coo_matrix((np.ones(6 * len(tris)), (np.repeat(tris[:, 0], 6), tris.ravel())), shape=(n, n))
    _, label = connected_components(adj, directed=False)
    supported = np.zeros(n, bool)
    supported[np.array([d // 2 for d in fixed], int)] = True
    island_nodes = ~np.isin(label, np.unique(label[supported]))
    if island_nodes.any():
        f_isl = np.linalg.norm(F.reshape(n, 2)[island_nodes])
        if f_isl > 1e-9 * np.linalg.norm(F):
            raise RuntimeError(f"load on a floating island ({island_nodes.sum()} nodes): broken load path")
        fixed.update(2 * i + c for i in np.flatnonzero(island_nodes) for c in (0, 1))
    fixed = np.array(sorted(fixed), int)
    free = np.setdiff1d(np.arange(2 * n), fixed)
    U = np.zeros(2 * n)
    lu = splu(K[free][:, free].tocsc())
    U[free] = lu.solve(F[free])
    return {"U": U, "J": float(F @ U), "V": float(area / host.area), "fixed_dofs": len(fixed),
            "boundary_edges": len(bedges), "islands": int(len(np.unique(label[island_nodes]))),
            "island_nodes": island_nodes}


def _edge_force(F, nodes, edge, traction_fn):
    """Consistent nodal forces of a traction on a quadratic boundary edge (3-point Gauss); traction_fn receives the
    physical point and the edge coordinate g in [-1, 1] of every Gauss point."""
    Xe = nodes[list(edge)]
    for g, w in zip(G3, W3):
        N, dN = shape_edge(g)
        x = N @ Xe
        ds = np.linalg.norm(dN @ Xe)
        t = traction_fn(x, g)
        for k, i in enumerate(edge):
            F[2 * i] += N[k] * t[0] * ds * w
            F[2 * i + 1] += N[k] * t[1] * ds * w


def structured_p2(L, H, nx, ny):
    """Structured quadratic triangulation of the rectangle [0, L] x [0, H] (gmsh node order), for tests."""
    xs, ys = np.linspace(0, L, 2 * nx + 1), np.linspace(0, H, 2 * ny + 1)
    X, Y = np.meshgrid(xs, ys, indexing="ij")
    nodes = np.stack([X.ravel(), Y.ravel()], 1)
    nid = lambda i, j: i * (2 * ny + 1) + j  # noqa: E731
    tris = []
    for i in range(nx):
        for j in range(ny):
            a, b = 2 * i, 2 * j
            c00, c20, c22, c02 = nid(a, b), nid(a + 2, b), nid(a + 2, b + 2), nid(a, b + 2)
            tris.append([c00, c20, c22, nid(a + 1, b), nid(a + 2, b + 1), nid(a + 1, b + 1)])
            tris.append([c00, c22, c02, nid(a + 1, b + 1), nid(a + 1, b + 2), nid(a, b + 1)])
    return nodes, np.array(tris)


def nodal_stress(nodes, tris, U):
    """Von Mises stress at the nodes, averaged over the adjacent elements (stress evaluated at the nodes)."""
    D = plane_stress_D()
    ref = np.array([[0, 0], [1, 0], [0, 1], [0.5, 0], [0.5, 0.5], [0, 0.5]], float)
    acc, cnt = np.zeros((len(nodes), 3)), np.zeros(len(nodes))
    for tri in tris:
        X = nodes[tri]
        ue = np.stack([U[2 * tri], U[2 * tri + 1]], 1).ravel()
        for k, (xi, eta) in enumerate(ref):
            _, dN = shape_p2(xi, eta)
            dNx = dN @ np.linalg.inv(dN.T @ X).T
            B = np.zeros((3, 12))
            B[0, 0::2], B[1, 1::2] = dNx[:, 0], dNx[:, 1]
            B[2, 0::2], B[2, 1::2] = dNx[:, 1], dNx[:, 0]
            acc[tri[k]] += D @ (B @ ue)
            cnt[tri[k]] += 1
    s = acc / np.maximum(cnt, 1)[:, None]
    return np.sqrt(np.maximum(s[:, 0] ** 2 - s[:, 0] * s[:, 1] + s[:, 1] ** 2 + 3 * s[:, 2] ** 2, 0.0))


def boundary_values(nodes, tris, values, points):
    """Values at points on the boundary, interpolated with the quadratic shape functions of the nearest boundary
    edge; returns the values and the distance of every point to the mesh boundary."""
    bedges = boundary_edges(tris)
    best = np.full(len(points), np.inf)
    out = np.zeros(len(points))
    ss = np.linspace(-1, 1, 21)
    for a, b, m in bedges:
        Xe = nodes[[a, b, m]]
        Ns = np.stack([shape_edge(s)[0] for s in ss])              # (21, 3)
        Xs = Ns @ Xe
        d = np.linalg.norm(points[:, None, :] - Xs[None], axis=2)
        j = np.argmin(d, 1)
        dj = d[np.arange(len(points)), j]
        better = dj < best
        best[better] = dj[better]
        out[better] = (Ns[j] @ values[[a, b, m]])[better]
    return out, best


def reference(npz_path, step_path, size=None):
    d = np.load(npz_path, allow_pickle=False)
    prob, met = json.loads(str(d["problem"])), json.loads(str(d["metrics"]))
    if len(prob["host"]["P"][0][0]) != 2:
        raise ValueError("body-fitted references are implemented for planar designs only")
    host = Host.from_dict(prob["host"]).build(8, 8)
    h = np.sqrt(host.area / (met["nel"][0] * met["nel"][1]))
    size = 0.5 * h if size is None else size
    nodes, tris = mesh_face(step_path, size)
    sol = solve_face(prob, nodes, tris)
    vm = nodal_stress(nodes, tris, sol["U"])
    out = {"nodes": nodes, "tris": tris, "U": sol["U"], "J": sol["J"], "V": sol["V"], "mesh_size": size,
           "n_nodes": len(nodes), "n_elements": len(tris), "fixed_dofs": sol["fixed_dofs"], "vm_nodes": vm,
           "J_eval": met["J_final"], "V_eval": met["V_final"]}
    if "loop_x" in d.files and d["loop_x"].size:
        X = d["loop_x"].reshape(-1, 2)
        vals, dist = boundary_values(nodes, tris, vm, X)
        out["loop_vm_bodyfit"] = vals.reshape(d["loop_x"].shape[:2])
        out["loop_dist_to_mesh"] = dist.reshape(d["loop_x"].shape[:2])
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("npz")
    ap.add_argument("step")
    ap.add_argument("--size", type=float, default=None, help="mesh size (default: half the optimization element)")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    res = reference(args.npz, args.step, args.size)
    print(json.dumps({k: v for k, v in res.items() if np.isscalar(v)}, indent=1, default=float))
    if args.out:
        np.savez_compressed(args.out, **res)


In [ ]:
%%writefile trimfields/pipeline.py
"""Dataset generation: one design end to end, a resumable process pool, and consolidation into arrays.

Two kinds of designs are generated. Optimized designs follow the TrimTO pipeline (isogeometric SIMP, loop fit,
CAD-native refinement, admissibility test, STEP export). Random designs (the `rand` splits) are admissible loop
sets without optimization, analysed and exported in the same way; loop sets that cut the load path are rejected.
Every design stores the reference fields of fields.py and the compliance of the untrimmed host under its loads
(J_host); designs of the test splits also store the compliance of a finer evaluation (ref_factor for planar faces,
ref_factor_shell for shells).
"""
import copy
import hashlib
import json
import os
import time
import traceback

import numpy as np

CONFIG = {
    "out_dir": "trimfields_data",
    "seed": 20260927,
    # designs per split and family: 14,300 optimized (as the TrimTO pipeline, under continuum boundary conditions)
    # and 11,000 random designs; 25,300 in total (about 280 core-hours at the costs measured on the 570-design sample)
    "counts": {
        "id": {"rect": 2000, "sector": 2000, "plate": 2000, "arc": 2000, "taper": 2000, "bolted": 2000},
        "ood_load": {"rect": 100, "sector": 100, "plate": 100, "arc": 100, "taper": 100},
        "ood_host": {"freeform": 250, "ell": 250},
        "shell": {"cyl": 400, "saddle": 300, "dome": 300, "torus": 200},
        "shell_ood": {"hood": 100},
        "rand": {"rect": 1500, "sector": 1500, "plate": 1500, "arc": 1500, "taper": 1500, "bolted": 1500},
        "rand_host": {"freeform": 500, "ell": 500},
        "rand_shell": {"cyl": 200, "saddle": 200, "dome": 200, "torus": 200},
        "rand_shell_ood": {"hood": 200},
    },
    "test_every": 10,                       # every 10th in-distribution design goes to the test split
    "n_el_planar": 2700,
    "n_el_shell": 2304,
    "thickness_rel": 0.05,                  # shell thickness / sqrt(area)
    "simp": {"penal": 3.0, "maxit": 200, "tol": 0.01, "move": 0.2, "rmin_h": 2.0},
    "simp_shell": {"penal": 3.0, "maxit": 250, "tol": 0.005, "move": 0.2, "rmin_h": 1.5, "beta_max": 16.0},
    "n_cp": 16,
    "k_max": 16,
    "refine": {"maxit": 200, "n_top": 25, "lr0": 0.25, "lr1": 0.01, "alpha": 2e-3, "beta": 0.1, "gamma": 1.0,
               "mu": 50.0, "w_min_h": 1.5, "nucleation": True, "nucleation_tau": 0.02,
               "nucleation_radius_h": 1.5, "min_iters": 80, "stop_rel": 2e-4, "gamma_keep": 1.0,
               "move_accept_tol": 0.0},
    # random designs: 1 to max_loops loops, clearance w_min_h elements, volume fraction at least min_volume, and a
    # compliance of at most max_compliance_host times that of the untrimmed host (an intact load path)
    "random": {"max_loops": 8, "w_min_h": 1.5, "min_volume": 0.15, "max_compliance_host": 50.0},
    "eval_factor": 2,                       # production evaluation: f times as many elements per direction
    "ref_factor": 4,                        # finer evaluation stored for the planar designs of the test splits
    "ref_factor_shell": 3,                  # the same for shells (their f = 4 evaluation needs 2.6 GB per worker)
    "loop_samples": 64,                     # boundary samples per loop (fields.loop_samples)
    "export_step": True,
    "bucket": 8192,
    "vol_tol": 5e-3,                        # admissible designs: |V - V*| below this (crisp volume fraction)
    "max_attempts": 5,                      # problems without an admissible trimmed design are resampled
    "attempt_offset": 0,                    # raised by retry rounds so that they draw new problems
    "task_timeout": 3600,                   # seconds per design (all attempts); the worker is killed beyond it
}

RANDOM_SPLITS = ("rand", "rand_host", "rand_shell", "rand_shell_ood")
PLANAR_SPLITS = ("id", "ood_load", "ood_host", "rand", "rand_host")
SHELL_SPLITS = ("shell", "shell_ood", "rand_shell", "rand_shell_ood")
TRAIN_TEST_SPLITS = ("id", "shell", "rand", "rand_shell")   # every test_every-th design is a test design


class Rejected(RuntimeError):
    """The sampled problem gave no admissible trimmed design; the next attempt resamples the problem."""


def design_path(cfg, split, family, idx, ext="npz"):
    sub = "designs" if ext == "npz" else "step"
    return os.path.join(cfg["out_dir"], sub, split, family, f"{idx:05d}.{ext}")


def task_list(cfg):
    tasks = []
    for split, fam_counts in cfg["counts"].items():
        for family, n in fam_counts.items():
            tasks += [(split, family, i) for i in range(n)]
    return tasks


def sub_split(cfg, split, idx):
    """Name of the split in the consolidated index: train/test parts of the in-distribution splits."""
    if split in TRAIN_TEST_SPLITS:
        return f"{split}_test" if idx % cfg["test_every"] == 0 else f"{split}_train"
    return split


def is_test(cfg, split, idx):
    return not sub_split(cfg, split, idx).endswith("_train")


def _seed(cfg, split, family, idx, attempt):
    h = hashlib.sha256(f"{cfg['seed']}/{split}/{family}/{idx}/{attempt}".encode()).digest()
    return int.from_bytes(h[:8], "little")


# ------------------------------------------------------------------------------------------------ one design
def admissible(loops, r, vstar, keep_geoms, vol_tol, j_max=None):
    """'ok' or the reason a loop set is not an admissible trimmed design. j_max (twice the compliance of the
    thresholded density design) rejects designs whose load path is broken."""
    from shapely.geometry import Polygon

    from .loops import loop_collisions, sample_loop, self_intersections
    from .problems import covers_keep
    if not (np.isfinite(r["J"]) and r["J"] > 0):
        return f"invalid compliance {r['J']}"
    if j_max is not None:
        if not (np.isfinite(j_max) and j_max > 0):
            return f"invalid reference compliance for the load-path test ({j_max})"
        if r["J"] > j_max:
            return f"compliance {r['J']:.4g} above {j_max:.4g} (broken load path)"
    if not abs(r["V"] - vstar) <= vol_tol:                  # also rejects NaN
        return f"volume {r['V']:.4f} vs {vstar:.4f}"
    if any(self_intersections(C) for C in loops):
        return "self-intersection"
    if len(loops) > 1 and loop_collisions(loops):
        return "loop collision"
    if any(covers_keep(Polygon(sample_loop(np.asarray(C), 20)).buffer(0), keep_geoms) for C in loops):
        return "loop covers a support or load"
    return "ok"


def _strip_passive(raw, prob, nel):
    """Contours minus the passive regions: an L-shape cut-out, or hole-plus-ring discs grown by one element.
    Pieces below two element areas are dropped (contours of a passive hole alone)."""
    from shapely.geometry import MultiPolygon, Polygon
    from shapely.ops import unary_union

    from .problems import passive_discs_uv
    el = 1.0 / (nel[0] * nel[1])
    pas = [Polygon(ps["uv"]) for ps in prob["passive"] if ps["kind"] == "polygon"]
    pas += [Polygon(d).buffer(1.0 / max(nel)) for d in passive_discs_uv(prob)]
    pas_u = unary_union(pas)
    out = []
    for a in raw:
        g = Polygon(a).buffer(0).difference(pas_u)
        for piece in (g.geoms if isinstance(g, MultiPolygon) else [g]):
            if piece.is_empty or piece.area <= 2 * el:
                continue
            if any(ps["kind"] == "circle" for ps in prob["passive"]):
                out.append(np.asarray(piece.exterior.coords)[:-1])
            else:                                   # L-shape: keep the original contour (union of voids)
                out.append(np.asarray(a))
                break
    return out


def is_done(cfg, task):
    """A design is complete when its npz exists and, with STEP export, its STEP file too."""
    return os.path.exists(design_path(cfg, *task)) and (
        not cfg["export_step"] or os.path.exists(design_path(cfg, *task, ext="step")))


def fit_at_volume(host, rho, level, prob, nel, cfg, ev, iters=10):
    """Loops fitted to the density contours of a level. If the fitted loops miss V* by more than the tolerance
    (for instance where passive regions are cut out of the contours), the level is adjusted by bisection until the
    crisp volume of the fitted loops matches V*. Returns (loops, level)."""
    from .loops import extract_void_loops, fit_loops, in_square_area

    def fit(lev):
        raw = extract_void_loops(host, rho, level=lev)
        if prob["passive"]:
            raw = _strip_passive(raw, prob, nel)
        loops = sorted(fit_loops(host, raw, cfg["n_cp"]), key=in_square_area, reverse=True)[: cfg["k_max"]]
        return loops, ev.volume(ev.loop_material(loops))

    loops, V = fit(level)
    tol = 0.4 * cfg["vol_tol"]
    if abs(V - prob["vstar"]) <= tol:
        return loops, level
    best = (abs(V - prob["vstar"]), loops, level)
    lo, hi = (level, min(0.95, level + 0.4)) if V > prob["vstar"] else (max(0.05, level - 0.4), level)
    for _ in range(iters):                                # more material than V*: raise the level, and vice versa
        mid = 0.5 * (lo + hi)
        L_, V_ = fit(mid)
        if abs(V_ - prob["vstar"]) < best[0]:
            best = (abs(V_ - prob["vstar"]), L_, mid)
        if abs(V_ - prob["vstar"]) <= tol:
            break
        lo, hi = (mid, hi) if V_ > prob["vstar"] else (lo, mid)
    return best[1], best[2]


def _setup(prob, cfg):
    """Host, mesh and thickness of a sampled problem."""
    from .hosts import SHELL, Host
    shell = prob["family"] in SHELL
    host = Host.from_dict(prob["host"])
    nel = host.mesh_for(cfg["n_el_shell"] if shell else cfg["n_el_planar"])
    host.build(*nel)
    nel = (host.nelu, host.nelv)
    thick = cfg["thickness_rel"] * np.sqrt(host.area) if shell else None
    prob["thickness"] = thick
    return host, nel, thick, shell


def _optimized_design(prob, host, nel, thick, shell, cfg, T):
    """SIMP, thresholded evaluation, loop fit and CAD-native refinement; returns the final design and its records."""
    from .evaluate import Evaluator
    from .indicator import Indicator
    from .mechanics import Model
    from .problems import apply_bc, keep_cps, keep_geometry, keep_samples, passive_masks_cp
    from .refine import Refiner
    from .simp import run_simp, run_simp_projected

    # 1. isogeometric SIMP
    t = time.perf_counter()
    m3 = apply_bc(prob, Model.on_grid(host, 3, thick))
    void, solid = passive_masks_cp(prob, host) if prob["passive"] else (None, np.zeros(host.ncu * host.ncv, bool))
    solid = solid | keep_cps(prob, host)    # non-design solid under supports and loads
    keep_g = keep_geometry(prob)
    h_el = np.sqrt(host.area / (nel[0] * nel[1]))
    if shell:
        sc = cfg["simp_shell"]
        rho, sinfo = run_simp_projected(m3, prob["vstar"], sc["rmin_h"] * h_el, sc["penal"], sc["maxit"],
                                        sc["tol"], sc["move"], beta_max=sc["beta_max"], solid=solid)
    else:
        sc = cfg["simp"]
        rho, sinfo = run_simp(m3, prob["vstar"], sc["rmin_h"] * h_el, void, solid, sc["penal"], sc["maxit"],
                              sc["tol"], sc["move"])
    if not np.isfinite(sinfo["J_simp"]) or sinfo["J_simp"] <= 0:
        raise RuntimeError("SIMP failed")
    del m3
    T["simp"] = time.perf_counter() - t

    # 2. crisp evaluation of the thresholded density
    t = time.perf_counter()
    ev = Evaluator(prob, nel, thick, cfg["eval_factor"])
    th = ev.threshold_for_volume(host, rho, prob["vstar"])
    r_thr = ev.evaluate(ev.density_material(rho, th))
    T["eval_thr"] = time.perf_counter() - t
    if not abs(r_thr["V"] - prob["vstar"]) <= 4 * cfg["vol_tol"]:      # the target volume is unattainable by a level
        raise Rejected(f"thresholded density volume {r_thr['V']:.4f} vs {prob['vstar']:.4f}")
    J_host = _host_compliance(ev, T)

    # 3. loops fitted to the density contour
    t = time.perf_counter()
    loops_fit, level_fit = fit_at_volume(host, rho, th, prob, nel, cfg, ev)
    r_fit = ev.evaluate(ev.loop_material(loops_fit))
    T["fit"] = time.perf_counter() - t

    # 4. CAD-native refinement
    t = time.perf_counter()
    m4 = apply_bc(prob, Model.on_grid(host, 4, thick))
    ind = Indicator(m4, prob, cfg["n_cp"], cfg["k_max"], bucket=cfg["bucket"],
                    keep_uv=keep_samples(keep_g, 0.5 / max(nel)))
    ref = Refiner(m4, ind, cfg["refine"], keep_g)
    loops_ref, rinfo = ref.run(loops_fit, prob["vstar"], cfg["refine"]["maxit"])
    del ref, ind, m4
    T["refine"] = time.perf_counter() - t
    t = time.perf_counter()
    r_ref = ev.evaluate(ev.loop_material(loops_ref), keep_model=True)
    T["eval_ref"] = time.perf_counter() - t
    # final design: the admissible one of refined and fitted loops with the lower crisp compliance
    adm = {name: admissible(L, r, prob["vstar"], keep_g, cfg["vol_tol"], 2.0 * r_thr["J"])
           for name, L, r in (("refine", loops_ref, r_ref), ("fit", loops_fit, r_fit))}
    cands = [(r["J"], name, L, r) for name, L, r in (("refine", loops_ref, r_ref), ("fit", loops_fit, r_fit))
             if adm[name] == "ok"]
    if not cands:
        raise Rejected(f"no admissible design (refine: {adm['refine']}, fit: {adm['fit']})")
    _, final_source, loops_fin, r_fin = min(cands, key=lambda c_: c_[0])
    if final_source == "fit":                                 # one evaluation model at a time (memory per worker)
        r_ref.pop("model", None)
        r_ref.pop("U", None)
        t = time.perf_counter()
        r_fin = ev.evaluate(ev.loop_material(loops_fit), keep_model=True)
        T["eval_ref"] += time.perf_counter() - t
    metrics = {"kind": "optimized", "simp": {k: v for k, v in sinfo.items()}, "threshold": th, "J_host": J_host,
               "fit_level": level_fit, "J_thr": r_thr["J"], "V_thr": r_thr["V"], "J_fit": r_fit["J"],
               "V_fit": r_fit["V"], "J_ref": r_ref["J"], "V_ref": r_ref["V"],
               "cut_elements_ref": r_ref["cut_elements"], "final_source": final_source, "admissible": adm,
               "loops_fit": len(loops_fit), "loops_ref": len(loops_ref),
               "refine": {k: v for k, v in rinfo.items() if k != "history"}}
    arrays = {"rho": rho.astype(np.float32), "loops_fit": loops_fit, "loops_ref": loops_ref,
              "history": rinfo["history"].astype(np.float32)}
    return loops_fin, r_fin, ev, keep_g, metrics, arrays


def _host_compliance(ev, T):
    """Crisp compliance of the untrimmed host (passive holes excluded) under the problem's loads: the scale of the
    compliance of every design on that problem, and the reference of the load-path test of random designs."""
    t = time.perf_counter()
    J = ev.evaluate(ev.loop_material([]))["J"]
    T["eval_host"] = time.perf_counter() - t
    if not (np.isfinite(J) and J > 0):
        raise Rejected(f"invalid host compliance {J}")
    return J


def _random_design(prob, host, nel, thick, cfg, rng, T):
    """Random admissible loop set, analysed with the crisp evaluator; V* is set to its volume fraction. Loop sets
    that cut the load path (compliance above max_compliance_host times that of the untrimmed host) are rejected."""
    from .evaluate import Evaluator
    from .problems import keep_geometry, random_loops
    t = time.perf_counter()
    rc = cfg["random"]
    keep_g = keep_geometry(prob)
    loops = random_loops(prob, host, nel, rng, cfg["n_cp"], rc["max_loops"], rc["w_min_h"], keep_g)
    if not loops:
        raise Rejected("no admissible random loop")
    T["random"] = time.perf_counter() - t
    ev = Evaluator(prob, nel, thick, cfg["eval_factor"])
    J_host = _host_compliance(ev, T)
    t = time.perf_counter()
    r = ev.evaluate(ev.loop_material(loops), keep_model=True)
    T["eval_ref"] = time.perf_counter() - t
    if not (np.isfinite(r["V"]) and r["V"] >= rc["min_volume"]):
        raise Rejected(f"random design too light: volume fraction {r['V']:.3f}")
    prob["vstar"] = float(r["V"])
    adm = admissible(loops, r, prob["vstar"], keep_g, cfg["vol_tol"], rc["max_compliance_host"] * J_host)
    if adm != "ok":
        raise Rejected(f"random design not admissible: {adm}")
    metrics = {"kind": "random", "final_source": "random", "admissible": {"random": adm}, "J_host": J_host,
               "cut_elements_ref": r["cut_elements"], "loops_fit": 0, "loops_ref": 0}
    arrays = {"rho": np.zeros(0, np.float32), "loops_fit": [], "loops_ref": [], "history": np.zeros((0, 4), np.float32)}
    return loops, r, ev, keep_g, metrics, arrays


def _reference_fields(prob, host, nel, thick, loops, r_fin, ev, cfg, test, T):
    """Reference fields of the final design: evaluation-mesh displacement, boundary samples along the loops, the
    design fields, and the finer evaluation on the test splits."""
    from .evaluate import Evaluator
    from .fields import design_field, loop_samples
    t = time.perf_counter()
    model, U = r_fin["model"], r_fin["U"]
    ls = loop_samples(model, U, loops, cfg["loop_samples"], prob)
    arrays = {"u_eval": U.astype(np.float32), "design_field": design_field(prob, loops).astype(np.float16)}
    arrays.update({f"loop_{k}": v for k, v in ls.items()})
    metrics = {"eval_factor": cfg["eval_factor"], "eval_nel": [ev.host.nelu, ev.host.nelv],
               "eval_ndof": int(model.ndof),
               "vm_max_loops": float(ls["vm"][ls["inside"]].max()) if ls["inside"].any() else None}
    T["fields"] = time.perf_counter() - t
    if test:
        t = time.perf_counter()
        rf = cfg["ref_factor_shell"] if thick is not None else cfg["ref_factor"]
        ev_fine = Evaluator(prob, nel, thick, rf)
        r_fine = ev_fine.evaluate(ev_fine.loop_material(loops))
        metrics.update({"ref_factor": rf, "J_fine": r_fine["J"], "V_fine": r_fine["V"]})
        T["eval_fine"] = time.perf_counter() - t
    else:
        metrics.update({"ref_factor": None, "J_fine": None, "V_fine": None})
    return arrays, metrics


def generate_one(task, cfg):
    split, family, idx = task
    out = design_path(cfg, split, family, idx)
    if is_done(cfg, task):
        return {"task": task, "status": "exists"}
    from .loops import canonical, loop_collisions, self_intersections
    from .problems import sample_problem

    random_design = split in RANDOM_SPLITS
    last_err, rejections = None, []
    for attempt in range(cfg["max_attempts"]):
        rng = np.random.default_rng(_seed(cfg, split, family, idx, attempt + cfg.get("attempt_offset", 0)))
        T = {}
        try:
            t = time.perf_counter()
            prob = sample_problem(family, rng, "ood_load" if split == "ood_load" else "id")
            prob["split"] = split
            host, nel, thick, shell = _setup(prob, cfg)
            T["setup"] = time.perf_counter() - t
            if random_design:
                loops_fin, r_fin, ev, keep_g, met, arrays = _random_design(prob, host, nel, thick, cfg, rng, T)
            else:
                loops_fin, r_fin, ev, keep_g, met, arrays = _optimized_design(prob, host, nel, thick, shell, cfg, T)

            # validity, reference fields and export
            metrics = {
                "split": split, "family": family, "idx": idx, "attempt": attempt + cfg.get("attempt_offset", 0),
                "test": is_test(cfg, split, idx), "vstar": prob["vstar"],
                "nel": list(nel), "ndof": int(host.ncu * host.ncv * (3 if shell else 2)), "area": host.area,
                "thickness": thick, "J_final": r_fin["J"], "V_final": r_fin["V"], "rejections": rejections,
                "loops_final": len(loops_fin),
                "self_intersections_final": int(sum(self_intersections(C) for C in loops_fin)),
                "collisions_final": int(loop_collisions(loops_fin)) if loops_fin else 0,
            }
            metrics.update(met)
            metrics["valid"] = metrics["self_intersections_final"] == 0 and metrics["collisions_final"] == 0
            f_arrays, f_metrics = _reference_fields(prob, host, nel, thick, loops_fin, r_fin, ev, cfg,
                                                    metrics["test"], T)
            metrics.update(f_metrics)
            r_fin.pop("model", None)
            if cfg["export_step"]:
                t = time.perf_counter()
                from .export_step import HAVE_OCP, export_design
                if not HAVE_OCP:
                    raise RuntimeError("OCP (cadquery-ocp) is not importable; install it or set export_step=False")
                spath = design_path(cfg, split, family, idx, "step")
                stmp = f"{spath[:-5]}.{os.getpid()}.tmp.step"   # one file per writer, renamed when complete
                try:
                    st = export_design(prob, loops_fin, stmp)
                except Exception as ex:  # noqa: BLE001
                    if os.path.exists(stmp):
                        os.remove(stmp)
                    raise Rejected(f"STEP export failed: {type(ex).__name__}: {ex}") from ex
                st["area_rel_err"] = abs(st["area"] - r_fin["V"] * host.area) / (r_fin["V"] * host.area)
                if not st["valid"] or not st["area_rel_err"] <= 0.02:
                    os.remove(stmp)
                    raise Rejected(f"STEP check failed: valid {st['valid']}, area error {st['area_rel_err']:.3g}")
                metrics["step"] = st
                T["step"] = time.perf_counter() - t
            metrics["times"] = T
            metrics["t_total"] = float(sum(T.values()))
            X = canonical(loops_fin, cfg["k_max"])
            pack = lambda Ls: (np.stack(Ls) if len(Ls) else np.zeros((0, cfg["n_cp"], 2))).astype(np.float64)  # noqa: E731
            os.makedirs(os.path.dirname(out), exist_ok=True)
            tmp = f"{out[:-4]}.{os.getpid()}.tmp.npz"
            np.savez_compressed(tmp, problem=json.dumps(prob), metrics=json.dumps(metrics, default=float),
                                rho=arrays["rho"], loops_fit=pack(arrays["loops_fit"]),
                                loops_ref=pack(arrays["loops_ref"]), loops_final=pack(loops_fin),
                                x=X.astype(np.float32), history=arrays["history"], **f_arrays)
            if cfg["export_step"]:
                os.replace(stmp, spath)                 # STEP first: an npz always has its STEP file
            os.replace(tmp, out)
            for suffix in ("", "_killed"):          # records of earlier rounds that failed on this design
                stale = os.path.join(cfg["out_dir"], "failures", f"{split}_{family}_{idx:05d}{suffix}.txt")
                if os.path.exists(stale):
                    os.remove(stale)
            return {"task": task, "status": "ok", "t": metrics["t_total"], "valid": metrics["valid"],
                    "attempt": attempt}
        except Rejected as ex:
            rejections.append(str(ex))
            last_err = f"Rejected: {ex}"
        except Exception as ex:  # noqa: BLE001
            fr = traceback.extract_tb(ex.__traceback__)[-1]
            rejections.append(f"{type(ex).__name__}: {ex} [{os.path.basename(fr.filename)}:{fr.lineno}]")
            last_err = f"{type(ex).__name__}: {ex}\n{traceback.format_exc()}"
    fdir = os.path.join(cfg["out_dir"], "failures")
    os.makedirs(fdir, exist_ok=True)
    with open(os.path.join(fdir, f"{split}_{family}_{idx:05d}.txt"), "w") as f:
        f.write(last_err or "unknown")
    return {"task": task, "status": "failed", "error": (last_err or "")[:300]}


# ------------------------------------------------------------------------------------------------ process pool
THREAD_VARS = ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",
               "VECLIB_MAXIMUM_THREADS")


def _limit_child_threads():
    """Set before the pool starts: spawned workers import NumPy/SciPy (and their BLAS pools) while unpickling the
    initializer, so settings made inside the initializer come too late for those libraries."""
    for k in THREAD_VARS:
        os.environ[k] = "1"


def _worker_init(gpu_slots=None):
    """Single-threaded numerics per worker. The first workers that find a free GPU slot run JAX on the GPU, the
    others on one CPU core (JAX is imported only after this, inside generate_one)."""
    for k in THREAD_VARS:
        os.environ[k] = "1"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
    backend = "cpu"
    if gpu_slots is not None:
        with gpu_slots.get_lock():
            if gpu_slots.value > 0:
                gpu_slots.value -= 1
                backend = "cuda"
                devices = [d for d in os.environ.get("TRIMFIELDS_GPU_DEVICES", "").split(",") if d.strip()]
                if len(devices) > 1:                          # several GPUs: round-robin over the workers
                    os.environ["CUDA_VISIBLE_DEVICES"] = devices[gpu_slots.value % len(devices)]
    os.environ["JAX_PLATFORMS"] = backend
    if backend == "cpu":
        os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"


def _task_name(task):
    split, family, idx = task
    return f"{split}_{family}_{idx:05d}"


def _run_task(args):
    """One design under a watchdog: faulthandler writes the stack and kills the worker when the design exceeds
    task_timeout (this also works when OpenCASCADE hangs inside C++), and records the stack of a crash."""
    import faulthandler
    task, cfg = args
    wd = os.path.join(cfg["out_dir"], "watchdog")
    os.makedirs(wd, exist_ok=True)
    fn = os.path.join(wd, _task_name(task) + ".txt")
    with open(fn, "w") as fh:
        faulthandler.enable(file=fh)
        faulthandler.dump_traceback_later(cfg.get("task_timeout", 3600), exit=True, file=fh)
        try:
            r = generate_one(task, cfg)
        except BaseException as ex:  # noqa: BLE001
            r = {"task": task, "status": "failed", "error": f"{type(ex).__name__}: {ex}"[:300]}
        finally:
            faulthandler.cancel_dump_traceback_later()
            faulthandler.disable()
    try:
        os.remove(fn)
    except FileNotFoundError:
        pass
    return r


def _pending(cfg, tasks, skip):
    todo = [t for t in tasks if t not in skip and not is_done(cfg, t)]
    order = {"shell": 0, "shell_ood": 0, "rand_shell": 1, "rand_shell_ood": 1}   # longest designs first
    return sorted(todo, key=lambda t: (order.get(t[0], 2), t[2], t[0], t[1]))


IDENTITY_EXCLUDE = ("out_dir", "attempt_offset", "task_timeout")


def check_identity(cfg):
    """Resuming is only allowed with the configuration that produced the designs already in out_dir."""
    ident = {k: v for k, v in cfg.items() if k not in IDENTITY_EXCLUDE}
    ident["counts"] = None                                     # more or fewer designs per family are fine
    path = os.path.join(cfg["out_dir"], "config.json")
    if os.path.exists(path):
        with open(path) as f:
            old = json.load(f)
        if old != json.loads(json.dumps(ident)):
            diff = sorted(k for k in set(old) | set(ident) if old.get(k) != json.loads(json.dumps(ident)).get(k))
            raise RuntimeError(f"{cfg['out_dir']} holds designs of another configuration (differences: {diff}); "
                               "use a new output directory or restore the configuration")
    else:
        with open(path, "w") as f:
            json.dump(ident, f, indent=1)


def _progress_bar(total):
    """Plain-text progress bar. Jupyter widget bars (tqdm.auto) break with old ipywidgets and show nothing in
    headless nbconvert runs, whereas text output works everywhere; progress.log holds the full record."""
    import sys
    try:
        from tqdm import tqdm
        return tqdm(total=total, desc="designs", smoothing=0.01, mininterval=60, file=sys.stdout, ascii=True)
    except Exception:  # noqa: BLE001
        return None


def run_all(cfg, tasks, n_workers, progress=True, max_restarts=200, n_gpu_workers=0):
    """Generate every missing design with a spawn-based process pool; returns the list of task results.
    n_gpu_workers of the n_workers processes run the JAX part on the GPU.

    Resumable: designs already on disk are skipped. If a worker dies (watchdog timeout or crash), the pool is
    restarted; the design that caused it is identified from its watchdog file and skipped in this call."""
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor, as_completed
    from concurrent.futures.process import BrokenProcessPool
    tasks = [tuple(t) for t in tasks]
    os.makedirs(cfg["out_dir"], exist_ok=True)
    check_identity(cfg)
    _limit_child_threads()
    wd = os.path.join(cfg["out_dir"], "watchdog")
    os.makedirs(wd, exist_ok=True)
    for f in os.listdir(wd):                                   # stale files of an interrupted earlier run
        os.remove(os.path.join(wd, f))
    skip, suspects, results = set(), {}, []
    todo = _pending(cfg, tasks, skip)
    bar = None
    if progress and todo:
        bar = _progress_bar(len(todo))
    log = open(os.path.join(cfg["out_dir"], "progress.log"), "a")
    ctx = mp.get_context("spawn")
    restarts = 0
    while todo:
        n_before = len(results)
        try:
            slots = ctx.Value("i", int(n_gpu_workers))
            with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx, initializer=_worker_init,
                                     initargs=(slots,)) as pool:
                futs = [pool.submit(_run_task, (t, cfg)) for t in todo]
                for f in as_completed(futs):
                    r = f.result()
                    r["task"] = tuple(r["task"])
                    results.append(r)
                    skip.add(r["task"])
                    log.write(json.dumps({"task": list(r["task"]), "status": r["status"], "t": r.get("t"),
                                          "attempt": r.get("attempt"), "time": time.time()}) + "\n")
                    log.flush()
                    if bar is not None:
                        bar.update(1)
                        bar.set_postfix(failed=sum(1 for x in results if x["status"] != "ok"), refresh=False)
        except BrokenProcessPool:
            restarts += 1
            if len(results) == n_before and not os.listdir(wd):
                raise RuntimeError("worker processes die before running any design (check that the trimfields "
                                   "package is importable and the environment variables are valid)")
            fdir = os.path.join(cfg["out_dir"], "failures")
            os.makedirs(fdir, exist_ok=True)
            by_name = {_task_name(t): t for t in tasks}
            for f in os.listdir(wd):
                p = os.path.join(wd, f)
                t = by_name.get(f[:-4])
                if t is None:
                    os.remove(p)
                    continue
                text = open(p).read()
                suspects[t] = suspects.get(t, 0) + 1
                if text.strip() or suspects[t] >= 3:            # watchdog stack or repeated crash: give up here
                    os.replace(p, os.path.join(fdir, f"{f[:-4]}_killed.txt"))
                    skip.add(t)
                    r = {"task": t, "status": "killed"}
                    results.append(r)
                    log.write(json.dumps({"task": list(t), "status": "killed", "time": time.time()}) + "\n")
                    if bar is not None:
                        bar.update(1)
                else:
                    os.remove(p)
            log.flush()
            if restarts > max_restarts:
                raise RuntimeError("too many worker crashes; see failures/ and progress.log")
        todo = _pending(cfg, tasks, skip)
    if bar is not None:
        bar.close()
    log.close()
    return results


def generate_until_complete(cfg, n_workers, rounds=10, progress=True, n_gpu_workers=0):
    """run_all over all tasks, then retry rounds for designs still missing. Every round draws new problems: the
    attempt offset is stored in out_dir/rounds.json, so a rerun of the notebook continues with unused seeds."""
    tasks = task_list(cfg)
    os.makedirs(cfg["out_dir"], exist_ok=True)
    state_path = os.path.join(cfg["out_dir"], "rounds.json")
    state = {"next_offset": 0, "history": []}
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
    for _ in range(rounds):
        missing = [t for t in tasks if not is_done(cfg, t)]
        if not missing:
            break
        c = copy.deepcopy(cfg)
        c["attempt_offset"] = state["next_offset"]
        state["next_offset"] += cfg["max_attempts"]
        with open(state_path, "w") as f:                       # reserve the seeds before using them
            json.dump(state, f, indent=1)
        res = run_all(c, missing, n_workers, progress=progress, n_gpu_workers=n_gpu_workers)
        left = sum(1 for t in tasks if not is_done(cfg, t))
        state["history"].append({"offset": c["attempt_offset"], "attempted": len(res), "missing": left,
                                 "time": time.strftime("%Y-%m-%d %H:%M:%S")})
        with open(state_path, "w") as f:
            json.dump(state, f, indent=1)
        print(f"round with attempt offset {c['attempt_offset']}: {len(res)} designs attempted, {left} missing",
              flush=True)
    return state["history"]


# ------------------------------------------------------------------------------------------------ consolidation
def _load(path):
    d = np.load(path, allow_pickle=False)
    return json.loads(str(d["problem"])), json.loads(str(d["metrics"])), d["x"], d["design_field"]


def _cond_job(path):
    from .conditioning import conditioning
    prob, _, _, df = _load(path)
    return conditioning(prob).astype(np.float16), df.astype(np.float16)


INDEX_COLUMNS = ("row", "split", "kind", "family", "idx", "test", "vstar", "J_host", "J_simp", "J_thr", "J_fit",
                 "J_ref", "J_final", "V_final", "ref_factor", "J_fine", "V_fine", "final_source", "loops_final",
                 "vm_max_loops", "attempt", "valid", "step_valid", "t_total", "path")


def consolidate(cfg, n_workers=8):
    """Index tables, canonical design tensors, conditioning and design fields, and splits for the planar and shell
    sets (optimized and random designs together; the `kind` and `split` columns tell them apart)."""
    import csv
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor
    out = os.path.join(cfg["out_dir"], "consolidated")
    os.makedirs(out, exist_ok=True)
    groups = {"planar": PLANAR_SPLITS, "shell": SHELL_SPLITS}
    summary = {}
    for name, splits in groups.items():
        paths, rows, X = [], [], []
        for split in splits:
            for family in cfg["counts"].get(split, {}):
                for i in range(cfg["counts"][split][family]):
                    p = design_path(cfg, split, family, i)
                    if not is_done(cfg, (split, family, i)):
                        continue
                    prob, met, x, _ = _load(p)
                    if met.get("J_host") is None:            # a design of an earlier version of the pipeline
                        raise RuntimeError(f"{p} lacks J_host: generated before trimfields 2.0.0; delete and regenerate")
                    rows.append({"row": len(rows), "split": sub_split(cfg, split, i), "kind": met["kind"],
                                 "family": family, "idx": i, "test": met["test"], "vstar": met["vstar"],
                                 "J_host": met.get("J_host"),
                                 "J_simp": met.get("simp", {}).get("J_simp"), "J_thr": met.get("J_thr"),
                                 "J_fit": met.get("J_fit"), "J_ref": met.get("J_ref"), "J_final": met["J_final"],
                                 "V_final": met["V_final"], "ref_factor": met.get("ref_factor"), "J_fine": met.get("J_fine"),
                                 "V_fine": met.get("V_fine"),
                                 "final_source": met["final_source"], "loops_final": met["loops_final"],
                                 "vm_max_loops": met.get("vm_max_loops"), "attempt": met["attempt"],
                                 "valid": met["valid"], "step_valid": met.get("step", {}).get("valid"),
                                 "t_total": met["t_total"], "path": os.path.relpath(p, cfg["out_dir"])})
                    paths.append(p)
                    X.append(x)
        if not rows:
            continue
        with open(os.path.join(out, f"{name}_index.csv"), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(INDEX_COLUMNS))
            w.writeheader()
            w.writerows(rows)
        np.save(os.path.join(out, f"{name}_X.npy"), np.stack(X).astype(np.float32))
        _limit_child_threads()
        cond = design = None
        with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("spawn"),
                                 initializer=_worker_init) as pool:
            for i, (c, d) in enumerate(pool.map(_cond_job, paths, chunksize=16)):     # results arrive in order
                if cond is None:
                    cond = np.lib.format.open_memmap(os.path.join(out, f"{name}_cond.npy"), mode="w+",
                                                     dtype=np.float16, shape=(len(paths),) + c.shape)
                    design = np.lib.format.open_memmap(os.path.join(out, f"{name}_design.npy"), mode="w+",
                                                       dtype=np.float16, shape=(len(paths),) + d.shape)
                cond[i], design[i] = c, d
        cond.flush()
        design.flush()
        del cond, design
        splits_idx = {}
        for r in rows:
            splits_idx.setdefault(r["split"], []).append(r["row"])
        with open(os.path.join(out, f"{name}_splits.json"), "w") as f:
            json.dump(splits_idx, f)
        summary[name] = {"designs": len(rows), "splits": {k: len(v) for k, v in splits_idx.items()},
                         "optimized": sum(1 for r in rows if r["kind"] == "optimized"),
                         "random": sum(1 for r in rows if r["kind"] == "random"),
                         "with_J_fine": sum(1 for r in rows if r["J_fine"] is not None),
                         "valid_fraction": float(np.mean([r["valid"] for r in rows]))}
    with open(os.path.join(out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=1)
    return summary


def smoke_config(cfg, n_per_family=1, maxit=40):
    """Small configuration for a quick end-to-end check of the whole pipeline."""
    c = copy.deepcopy(cfg)
    c["out_dir"] = os.path.join(cfg["out_dir"], "_smoke")
    c["counts"] = {s: {f: n_per_family for f in fams} for s, fams in cfg["counts"].items()}
    c["refine"]["maxit"] = maxit
    c["refine"]["min_iters"] = maxit // 2
    c["simp"]["maxit"] = 80
    c["simp_shell"]["maxit"] = 120
    return c


In [ ]:
%%writefile trimfields/checks.py
"""Numerical checks run before data generation: basis functions, analytic and benchmark solutions, design
gradients, cut-cell quadrature and STEP export. Each check returns a dict with its measured error and a pass flag."""
import math
import os
import tempfile

import numpy as np


def check_basis(seed=0):
    """Partition of unity and finite-difference derivatives of a rational host (plate with a hole)."""
    from .hosts import make_host
    host = make_host("plate", np.random.default_rng(seed)).build(12, 6)
    rng = np.random.default_rng(seed + 1)
    u, v = rng.uniform(0.05, 0.95, 200), rng.uniform(0.05, 0.95, 200)
    b = host.basis(u, v, nd=1)
    pu = float(np.abs(b["R"].sum(1) - 1).max())
    eps = 1e-6
    bp, bm = host.basis(u + eps, v, nd=0), host.basis(u - eps, v, nd=0)
    same = np.all(bp["ien"] == b["ien"], 1) & np.all(bm["ien"] == b["ien"], 1)
    fd = (bp["R"] - bm["R"]) / (2 * eps)
    d_err = float(np.abs(fd[same] - b["Ru"][same]).max())
    return {"partition_of_unity": pu, "dR_du_fd": d_err, "pass": pu < 1e-12 and d_err < 1e-6}


def check_tension():
    """Plane-stress bar in uniaxial tension: compliance F^2 L / (E H t) = 2 for L = 2, H = 1."""
    from .hosts import BEZ, Host, rectangle
    from .mechanics import Model, edge_cps, edge_load
    P, W = rectangle(2.0, 1.0)
    h = Host("rect", {}, BEZ, BEZ, P, W).build(20, 10)
    m = Model.on_grid(h, 3)
    F = edge_load(m, "u1", 0, 1, (1, 0), 1.0)
    m.set_bc(np.r_[2 * edge_cps(h, "u0", 0, 1), 1], F)
    J, U = m.solve(np.ones(m.u.size))
    err = abs(J - 2.0) / 2.0
    e_sum = abs(m.energy_density(U).sum() / J - 1.0)
    return {"J": J, "rel_err": err, "energy_sum_err": e_sum, "pass": err < 1e-9 and e_sum < 1e-9}


def check_scordelis_lo(n=24):
    """Scordelis-Lo roof (Kirchhoff-Love shell): vertical deflection at the free-edge midpoint, reference 0.3006."""
    from .basis import gauss_grid
    from .hosts import BEZ, Host, arc3
    from .mechanics import Model, edge_cps, surface_load
    R, L, t = 25.0, 50.0, 0.25
    A, w = arc3(math.radians(80), R, start=math.radians(50))
    P = [[(A[i, 0], y, A[i, 1]) for y in (0, L / 2, L)] for i in range(3)]
    W = [[w[i]] * 3 for i in range(3)]
    h = Host("cyl", {}, BEZ, BEZ, P, W).build(n, n)
    m = Model(h, *gauss_grid(h.Uu, h.Uv, 3)[:4], thickness=t, nu=0.0)
    F = surface_load(m, (0, 0, -1), 90 * h.area)
    fixed = []
    for edge in ("v0", "v1"):                                  # rigid diaphragms: u_x = u_z = 0
        cps = edge_cps(h, edge, 0, 1)
        fixed += list(3 * cps) + list(3 * cps + 2)
    fixed.append(1)                                            # axial rigid-body mode (no axial load)
    m.set_bc(np.array(fixed), F)
    _, U = m.solve(np.full(m.u.size, 4.32e8))
    b = h.basis(np.array([0.0]), np.array([0.5]), nd=0)
    wz = -float((b["R"][0] * U[3 * b["ien"][0] + 2]).sum())
    err = abs(wz - 0.3006) / 0.3006
    return {"w_mid": wz, "reference": 0.3006, "rel_err": err, "pass": err < 0.01}


def _circle_loops(centres, radii, n_cp=16):
    from .loops import fit_periodic
    t = np.linspace(0, 2 * np.pi, 200, endpoint=False)
    return [fit_periodic(np.stack([c[0] + r * np.cos(t), c[1] + r * np.sin(t)], 1), n_cp)
            for c, r in zip(centres, radii)]


def _cantilever(n_el=600):
    from .hosts import Host, make_host
    from .problems import apply_bc
    host = make_host("rect", np.random.default_rng(3))
    prob = {"family": "rect", "split": "id", "host": host.to_dict(), "vstar": 0.5, "passive": [], "thickness": None,
            "supports": [{"kind": "edge", "edge": "u0", "s0": 0.0, "s1": 1.0, "dofs": "all"}],
            "loads": [{"kind": "edge", "edge": "u1", "s0": 0.4, "s1": 0.6, "dir": [0.0, -1.0], "mag": 1.0}]}
    host = Host.from_dict(prob["host"])
    host.build(*host.mesh_for(n_el))
    return prob, host, apply_bc


def check_gradient(seed=0):
    """Design gradient of the compliance with respect to the loop control points against central differences."""
    from .indicator import Indicator
    from .mechanics import Model
    from .refine import Refiner
    from .pipeline import CONFIG
    prob, host, apply_bc = _cantilever()
    m4 = apply_bc(prob, Model.on_grid(host, 4))
    ind = Indicator(m4, prob, 16, 16)
    ref = Refiner(m4, ind, CONFIG["refine"])
    Cz, act = ref._slots(_circle_loops([(0.3, 0.5), (0.68, 0.45)], [0.12, 0.09]))
    J, _, g = ref._J_and_grad(Cz, act)
    rng = np.random.default_rng(seed)
    errs = []
    for _ in range(3):
        D = rng.normal(size=Cz.shape) * act[:, None, None]
        D /= np.linalg.norm(D)
        eps = 1e-3 * ind.h
        Jp = ref._J_and_grad(Cz + eps * D, act)[0]
        Jm = ref._J_and_grad(Cz - eps * D, act)[0]
        fd, an = (Jp - Jm) / (2 * eps), float(np.sum(g * D))
        errs.append(abs(fd - an) / max(abs(fd), 1e-12))
    return {"J": J, "max_rel_err": max(errs), "pass": max(errs) < 1e-4}


def check_cut_cell():
    """Crisp volume fraction of a two-hole design against the exact area of the sampled loops."""
    from shapely.geometry import Polygon, box
    from shapely.ops import unary_union
    from .evaluate import Evaluator
    from .loops import sample_loop
    prob, host, _ = _cantilever()
    loops = _circle_loops([(0.3, 0.5), (0.68, 0.45)], [0.12, 0.09])
    ev = Evaluator(prob, (host.nelu, host.nelv), None, 2)
    r = ev.evaluate(ev.loop_material(loops))
    holes = unary_union([Polygon(sample_loop(C, 400)) for C in loops]).intersection(box(0, 0, 1, 1))
    V_exact = 1.0 - holes.area                  # rectangle host: parametric and physical fractions agree
    err = abs(r["V"] - V_exact)
    return {"V": r["V"], "V_exact": V_exact, "abs_err": err, "cut_elements": r["cut_elements"], "pass": err < 2e-3}


def check_step():
    """STEP export with pcurves: B-rep validity after reading the file back, and face area against the design."""
    from .evaluate import Evaluator
    from .export_step import HAVE_OCP, export_design
    if not HAVE_OCP:
        return {"pass": False, "error": "OCP not importable"}
    prob, host, _ = _cantilever()
    loops = _circle_loops([(0.3, 0.5), (0.68, 0.45)], [0.12, 0.09])
    ev = Evaluator(prob, (host.nelu, host.nelv), None, 2)
    V = ev.evaluate(ev.loop_material(loops))["V"]
    with tempfile.TemporaryDirectory() as d:
        st = export_design(prob, loops, os.path.join(d, "check.step"))
    err = abs(st["area"] - V * host.area) / (V * host.area)
    return {"valid": st["valid"], "faces": st["faces"], "area_rel_err": err, "pass": st["valid"] and err < 2e-3}


def check_solvers():
    """PARDISO and SuperLU give the same compliance (only when pypardiso is installed)."""
    from . import mechanics as M
    if not M.HAVE_PARDISO:
        return {"pass": True, "note": "pypardiso not installed; SuperLU is used"}
    prob, host, apply_bc = _cantilever()
    m = apply_bc(prob, M.Model.on_grid(host, 3))
    E = np.random.default_rng(0).uniform(0.1, 1.0, m.u.size)
    use = M.USE_PARDISO
    try:
        M.USE_PARDISO = True
        J1 = m.solve(E)[0]
        M.USE_PARDISO = False
        J2 = m.solve(E)[0]
    finally:
        M.USE_PARDISO = use
    err = abs(J1 - J2) / abs(J2)
    return {"J_pardiso": J1, "J_superlu": J2, "rel_diff": err, "pass": err < 1e-8}


def check_bearing():
    """Bearing load of a bolt hole: the resultant equals magnitude * direction, the traction acts on the half of the
    hole facing the load, and the compliance equals the energy of the solution (plate with one loaded hole)."""
    from .hosts import make_host
    from .mechanics import Model
    from .problems import apply_bc, bearing_load
    host = make_host("bolted", np.random.default_rng(5))
    L, H = host.params["L"], host.params["H"]
    prob = {"family": "bolted", "split": "id", "host": host.to_dict(), "vstar": 0.5, "thickness": None,
            "passive": [{"kind": "circle", "centre": [0.5 * L, 0.5 * H], "radius": 0.1 * H, "ring": 0.08 * H}],
            "supports": [{"kind": "edge", "edge": "u0", "s0": 0.0, "s1": 1.0, "dofs": "all"}],
            "loads": [{"kind": "ring", "passive": 0, "dir": [0.6, -0.8], "mag": 2.0}]}
    host.build(*host.mesh_for(600))
    m = Model.on_grid(host, 3)
    F = bearing_load(m, prob, 0, prob["loads"][0]["dir"], 2.0)
    res = np.array([F[0::2].sum(), F[1::2].sum()])
    res_err = float(np.linalg.norm(res - 2.0 * np.array([0.6, -0.8])))
    # forces are assigned only to control points whose basis functions reach the loaded half of the hole
    P = host.P
    d = np.stack([P[:, 0] - 0.5 * L, P[:, 1] - 0.5 * H], 1)
    side = d @ np.array([0.6, -0.8]) / np.maximum(np.linalg.norm(d, axis=1), 1e-300)
    loaded = np.abs(F[0::2]) + np.abs(F[1::2]) > 1e-12
    behind = float(np.sum(loaded & (side < -0.6)) / max(loaded.sum(), 1))      # share of loaded cps behind the hole
    apply_bc(prob, m)
    J, U = m.solve(np.ones(m.u.size))
    e_err = float(abs(m.energy_density(U).sum() / J - 1.0))
    return {"resultant_err": res_err, "cps_behind_hole": behind, "J": float(J), "energy_sum_err": e_err,
            "pass": res_err < 1e-10 and behind < 0.05 and e_err < 1e-9}


def check_fields():
    """Reference fields of a design: the stress and energy of fields.stress_energy_at agree with the model's own
    quadrature-point values, the loop normals are unit vectors pointing into the holes, and the design field
    reproduces the crisp material test of the evaluator."""
    from .evaluate import Evaluator
    from .fields import design_field, loop_samples, material_mask, stress_energy_at
    from .hosts import Host
    prob, host, _ = _cantilever()
    centres, radii = [(0.3, 0.5), (0.68, 0.45)], [0.12, 0.09]
    loops = _circle_loops(centres, radii)
    ev = Evaluator(prob, (host.nelu, host.nelv), None, 2)
    r = ev.evaluate(ev.loop_material(loops), keep_model=True)
    m, U = r["model"], r["U"]
    sel = np.arange(0, m.u.size, max(1, m.u.size // 500))
    _, w = stress_energy_at(m, U, m.u[sel], m.v[sel])
    w_ref = m.energy_density(U)[sel] / m.dA[sel]
    w_err = float(np.abs(w - w_ref).max() / np.abs(w_ref).max())
    ls = loop_samples(m, U, loops, 32)
    n_err = float(np.abs(np.linalg.norm(ls["normal"], axis=2) - 1.0).max())
    L = np.array([host.Lu, host.Lv])
    inward = []
    for k, (c, _) in enumerate(zip(centres, radii)):               # rectangle host: X = (Lu u, Lv v)
        to_centre = np.asarray(c) * L - ls["x"][k]
        inward.append(np.sum(to_centre / np.linalg.norm(to_centre, axis=1, keepdims=True) * ls["normal"][k], 1))
    inward = float(np.min(np.concatenate(inward)))
    df = design_field(prob, loops)
    n = df.shape[1]
    c = (np.arange(n) + 0.5) / n
    Ug, Vg = np.meshgrid(c, c, indexing="ij")
    mat = material_mask(prob, Host.from_dict(prob["host"]).build(8, 8), Ug.ravel(), Vg.ravel(), loops)
    chi_err = int(np.sum((df[0].ravel() > 0) != mat))
    sdf_sign_err = int(np.sum((df[1].ravel() > 0) != mat))
    return {"energy_err": w_err, "normal_err": n_err, "normal_inward_min": inward, "chi_mismatch": chi_err,
            "sdf_sign_mismatch": sdf_sign_err, "vm_max": float(ls["vm"].max()),
            "pass": w_err < 1e-9 and n_err < 1e-9 and inward > 0.9 and chi_err == 0 and sdf_sign_err == 0}


def check_bodyfit_core():
    """Quadratic-triangle plane-stress solver of bodyfit.py (without gmsh) on the bar in uniaxial tension of
    check_tension: compliance 2, and the edge-segment load integration recovers the exact resultant."""
    from .bodyfit import solve_face, structured_p2
    from .hosts import BEZ, Host, rectangle
    P, W = rectangle(2.0, 1.0)
    host = Host("rect", {"L": 2.0, "H": 1.0}, BEZ, BEZ, P, W)
    prob = {"family": "rect", "split": "id", "host": host.to_dict(), "vstar": 1.0, "passive": [], "thickness": None,
            "supports": [{"kind": "edge", "edge": "u0", "s0": 0.0, "s1": 1.0, "dofs": "normal", "fixed_component": 0},
                         {"kind": "edge", "edge": "u0", "s0": 0.0, "s1": 0.001, "dofs": "all"}],
            "loads": [{"kind": "edge", "edge": "u1", "s0": 0.0, "s1": 1.0, "dir": [1.0, 0.0], "mag": 1.0}]}
    nodes, tris = structured_p2(2.0, 1.0, 8, 4)
    r = solve_face(prob, nodes, tris)
    err = abs(r["J"] - 2.0) / 2.0
    prob["loads"][0].update({"s0": 0.3, "s1": 0.7})            # partial segment, not aligned with mesh edges
    r2 = solve_face(prob, nodes, tris)
    prob["loads"][0].update({"s0": 0.0, "s1": 1.0})
    nodes_i, tris_i = structured_p2(0.4, 0.3, 2, 2)              # a detached piece inside the bar: a floating island
    nodes3 = np.concatenate([nodes, nodes_i + np.array([0.8, 0.3])])
    tris3 = np.concatenate([tris, tris_i + len(nodes)])
    r3 = solve_face(prob, nodes3, tris3)
    err3 = abs(r3["J"] - 2.0) / 2.0
    return {"J": r["J"], "rel_err": err, "V": r["V"], "J_partial": r2["J"], "islands": r3["islands"],
            "J_island": r3["J"],
            "pass": err < 1e-9 and abs(r["V"] - 1.0) < 1e-12 and r2["J"] > r["J"] and r3["islands"] == 1
            and err3 < 1e-9}


def check_load_path():
    """Load-path test of the random designs: the untrimmed host has the lowest compliance of its problem, two
    holes raise it moderately, and a loop that severs the cantilever raises it by more than the rejection factor
    of the pipeline, so that admissible() reports the broken load path."""
    from .evaluate import Evaluator
    from .loops import fit_periodic
    from .pipeline import CONFIG, admissible
    prob, host, _ = _cantilever()
    ev = Evaluator(prob, (host.nelu, host.nelv), None, 2)
    J_host = ev.evaluate(ev.loop_material([]))["J"]
    holes = _circle_loops([(0.3, 0.5), (0.68, 0.45)], [0.12, 0.09])
    r_holes = ev.evaluate(ev.loop_material(holes))
    t = np.linspace(0, 2 * np.pi, 200, endpoint=False)                 # a tall ellipse across the whole height
    cut = [fit_periodic(np.stack([0.55 + 0.04 * np.cos(t), 0.5 + 0.7 * np.sin(t)], 1), 16)]
    r_cut = ev.evaluate(ev.loop_material(cut))
    factor = CONFIG["random"]["max_compliance_host"]
    verdict = admissible(cut, r_cut, r_cut["V"], (), CONFIG["vol_tol"], factor * J_host)
    return {"J_host": J_host, "J_holes_ratio": r_holes["J"] / J_host, "J_cut_ratio": r_cut["J"] / J_host,
            "verdict": verdict,
            "pass": J_host > 0 and 1.0 < r_holes["J"] / J_host < factor and r_cut["J"] / J_host > factor
            and verdict.startswith("compliance")}


def check_uncut_host():
    """Untrimmed L-shaped host whose cut-out edges lie between the classification samples of the evaluator, so
    that no element is cut: the evaluation (used for J_host of every design on such a host) must still return a
    finite compliance and the volume of the L-shaped domain. Before 2.0.1 the material test was called at no
    points and failed, which resampled the problem."""
    from .evaluate import Evaluator
    from .hosts import Host, make_host
    host = make_host("ell", np.random.default_rng(5))
    nel = host.mesh_for(600)
    h = 1.0 / (2 * nel[0])                                     # element width of the evaluator at f = 2
    a = 0.5 + 0.05 * h                                         # cut-out corner just beyond the knot line 0.5
    prob = {"family": "ell", "split": "id", "host": host.to_dict(), "vstar": 0.4, "thickness": None,
            "passive": [{"kind": "polygon", "uv": [[a, a], [1.2, a], [1.2, 1.2], [a, 1.2]]}],
            "supports": [{"kind": "edge", "edge": "v1", "s0": 0.0, "s1": a, "dofs": "all"}],
            "loads": [{"kind": "edge", "edge": "u1", "s0": 0.2, "s1": 0.3, "dir": [0.0, -1.0], "mag": 1.0}]}
    host = Host.from_dict(prob["host"])
    ev = Evaluator(prob, nel, None, 2)
    r = ev.evaluate(ev.loop_material([]))
    return {"cut_elements": r["cut_elements"], "J_host": r["J"], "V_host": r["V"],
            "pass": r["cut_elements"] == 0 and np.isfinite(r["J"]) and r["J"] > 0 and abs(r["V"] - 0.75) < 0.01}


def check_threshold_passive():
    """Passive geometry in the thresholded-density test: on a bolted plate with a density that dips far from the
    holes and is void around them (as a converged density is), the thresholded material of the evaluator is void
    inside the holes and solid on the rings regardless of the density, the level found by threshold_for_volume
    gives the target volume, the bearing load then acts on material (compliance within a factor 50 of the untrimmed
    host, that is, an intact load path), and the three material tests agree at random points:
    Evaluator.density_material with rho = 1, Evaluator.loop_material and fields.material_mask."""
    from .evaluate import Evaluator
    from .fields import material_mask
    from .hosts import Host, make_host
    from .problems import sample_problem
    rng = np.random.default_rng(5)
    prob = sample_problem("bolted", rng)
    host = Host.from_dict(prob["host"])
    host.build(*host.mesh_for(900))
    nel = (host.nelu, host.nelv)
    ev = Evaluator(prob, nel, None, 2)
    gu, gv = host.greville_uv()                              # density: a ramp in x, void within 1.3 radii of every hole
    Gu, Gv = np.meshgrid(np.asarray(gu).ravel(), np.asarray(gv).ravel(), indexing="ij")
    Pg = host.eval(Gu.ravel(), Gv.ravel())
    L, H = prob["host"]["params"]["L"], prob["host"]["params"]["H"]
    cand = np.stack(np.meshgrid(np.linspace(0.1 * L, 0.9 * L, 40), np.linspace(0.1 * H, 0.9 * H, 20)), -1).reshape(-1, 2)
    dist = np.min([np.hypot(cand[:, 0] - ps["centre"][0], cand[:, 1] - ps["centre"][1]) for ps in prob["passive"]], 0)
    xc, yc = cand[np.argmax(dist)]                           # a smooth dip of density far from the holes
    rho = 1.0 - 0.95 * np.exp(-((Pg[:, 0] - xc) ** 2 + (Pg[:, 1] - yc) ** 2) / (0.25 * L) ** 2)
    for ps in prob["passive"]:
        rho[np.hypot(Pg[:, 0] - ps["centre"][0], Pg[:, 1] - ps["centre"][1]) < 1.3 * ps["radius"]] = 0.0
    vstar = 0.9
    th = ev.threshold_for_volume(host, rho, vstar)
    r = ev.evaluate(ev.density_material(rho, th))
    J_host = ev.evaluate(ev.loop_material([]))["J"]
    u, v = rng.uniform(0, 1, 4000), rng.uniform(0, 1, 4000)
    X = host.eval(u, v)
    d = np.min([np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1]) / ps["radius"] for ps in prob["passive"]], 0)
    rings = np.zeros(len(u), bool)
    for ps in prob["passive"]:
        dd = np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1])
        rings |= (dd >= ps["radius"]) & (dd < ps["radius"] + ps["ring"])
    m_th = ev.density_material(rho, th)(u, v)
    hole_void = int(np.sum(m_th[d < 1.0]))
    ring_solid = int(np.sum(~m_th[rings]))
    m_loop = ev.loop_material([])(u, v)
    m_one = ev.density_material(np.ones_like(rho), 0.5)(u, v)
    m_fld = material_mask(prob, host, u, v, [])
    agree = int(np.sum(m_loop != m_one) + np.sum(m_loop != m_fld))
    return {"threshold": float(th), "V_thr": r["V"], "J_thr_over_host": r["J"] / J_host, "hole_points_solid": hole_void,
            "ring_points_void": ring_solid, "tests_disagree": agree,
            "pass": abs(r["V"] - vstar) < 5e-3 and np.isfinite(r["J"]) and 1.0 < r["J"] / J_host < 50.0
            and hole_void == 0 and ring_solid == 0 and agree == 0}


ALL = {"basis": check_basis, "tension": check_tension, "scordelis_lo": check_scordelis_lo,
       "gradient": check_gradient, "cut_cell": check_cut_cell, "step": check_step, "solvers": check_solvers,
       "bearing": check_bearing, "fields": check_fields, "bodyfit_core": check_bodyfit_core,
       "load_path": check_load_path, "threshold_passive": check_threshold_passive, "uncut_host": check_uncut_host}


def run_all_checks(verbose=True):
    out = {}
    for name, fn in ALL.items():
        try:
            out[name] = fn()
        except Exception as ex:  # noqa: BLE001
            out[name] = {"pass": False, "error": f"{type(ex).__name__}: {ex}"}
        if verbose:
            r = out[name]
            vals = ", ".join(f"{k} {v:.3g}" if isinstance(v, float) else f"{k} {v}" for k, v in r.items() if k != "pass")
            print(f"[{'PASS' if r['pass'] else 'FAIL'}] {name:13s} {vals}", flush=True)
    return out


In [ ]:
%%writefile trimfields/bench.py
"""Timings used to choose the JAX backend and the worker count, and to estimate the total run time.

Run as a module in a fresh process so that JAX_PLATFORMS and the thread settings of a worker apply:
    python -m trimfields.bench indicator <n_el> <reps>
    python -m trimfields.bench design <split> <family> <idx> <out_dir> [small]
The last line of the output is 'RESULT <json>'.
"""
import copy
import json
import resource
import sys
import time

import numpy as np


def indicator_timing(n_el=2700, reps=10):
    """Time of the JAX part of one refinement step (indicator of a new design state, its gradient, volume and volume
    gradient) and of one sparse solve (the CPU part), on a production-size planar model with eight loops."""
    import jax

    from . import indicator as I
    from .checks import _cantilever, _circle_loops
    from .mechanics import USE_PARDISO, Model
    from .pipeline import CONFIG
    from .refine import Refiner
    prob, host, apply_bc = _cantilever(n_el)
    m4 = apply_bc(prob, Model.on_grid(host, 4))
    ind = I.Indicator(m4, prob, 16, 16)
    ref = Refiner(m4, ind, CONFIG["refine"])
    centres = [(0.15 + 0.23 * i, 0.3 + 0.4 * j) for i in range(4) for j in range(2)]
    Cz, act = ref._slots(_circle_loops(centres, [0.07] * 8))
    ct = np.ones(ind.m_pad)

    def step(k):                                  # one refinement step: new state, gradient, volume and gradient
        C = Cz + 1e-9 * k
        st = ind.state(C, act)
        ind.chi_vjp(st, ct)
        ind.volume(C, act)
        return st["chi"]

    t = time.perf_counter()
    step(0)
    t_compile = time.perf_counter() - t
    t = time.perf_counter()
    for k in range(reps):
        E = step(k + 1)
    t_ind = (time.perf_counter() - t) / reps
    E = I.EMIN + np.asarray(E)[: ind.m] * (1 - I.EMIN)
    m4.solve(E)                                   # warm-up (loads MKL)
    t = time.perf_counter()
    for k in range(3):
        m4.solve(E * (1 + 1e-3 * k))
    t_solve = (time.perf_counter() - t) / 3
    return {"backend": jax.default_backend(), "device": str(jax.devices()[0]), "points": int(ind.m),
            "t_indicator": t_ind, "t_solve": t_solve, "t_compile": t_compile, "pardiso": bool(USE_PARDISO),
            "max_rss_gb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20}


def design_timing(split, family, idx, out_dir, small=False):
    """Wall time of one complete design (all stages, including rejected attempts)."""
    from .pipeline import CONFIG, design_path, generate_one, smoke_config
    cfg = copy.deepcopy(CONFIG)
    cfg["out_dir"] = out_dir
    if small:                                     # smoke-mode timing: short iterations and small meshes
        cfg = smoke_config(cfg)
        cfg["out_dir"] = out_dir
        cfg["n_el_planar"], cfg["n_el_shell"] = 900, 900
    t = time.perf_counter()
    r = generate_one((split, family, idx), cfg)
    r["wall"] = time.perf_counter() - t
    r["max_rss_gb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    if r["status"] == "ok":
        m = json.loads(str(np.load(design_path(cfg, split, family, idx))["metrics"]))
        r.update({"attempt": m["attempt"], "times": m["times"], "nel": m["nel"], "ndof": m["ndof"],
                  "J_final_over_J_thr": m["J_final"] / m["J_thr"] if m.get("J_thr") else None})
    return r


if __name__ == "__main__":
    if sys.argv[1] == "indicator":
        out = indicator_timing(int(sys.argv[2]), int(sys.argv[3]))
    else:
        out = design_timing(sys.argv[2], sys.argv[3], int(sys.argv[4]), sys.argv[5],
                            small=len(sys.argv) > 6 and sys.argv[6] == "small")
    print("RESULT " + json.dumps(out, default=str), flush=True)


## 5. Configuration and numerical checks

In [ ]:
os.environ["JAX_PLATFORMS"] = "cpu"                   # the notebook process runs JAX on the CPU; workers choose
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
for k in [k for k in sys.modules if k == "trimfields" or k.startswith("trimfields.")]:
    del sys.modules[k]
importlib.invalidate_caches()
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import trimfields
from trimfields import pipeline as P
from trimfields.checks import run_all_checks

cfg = copy.deepcopy(P.CONFIG)
cfg["out_dir"] = OUT_DIR
TASKS = P.task_list(cfg)
N_PLANAR = sum(n for s in P.PLANAR_SPLITS for n in cfg["counts"].get(s, {}).values())
N_SHELL = sum(n for s in P.SHELL_SPLITS for n in cfg["counts"].get(s, {}).values())
N_RANDOM = sum(n for s in P.RANDOM_SPLITS for n in cfg["counts"].get(s, {}).values())
print(f"trimfields {trimfields.__version__}: {len(TASKS)} designs ({N_PLANAR} planar, {N_SHELL} shell; "
      f"{N_RANDOM} of them random)")
for split, fams in cfg["counts"].items():
    print(f"  {split:10s}", ", ".join(f"{f} {n}" for f, n in fams.items()))

In [ ]:
CHECKS = run_all_checks()
failed = [k for k, v in CHECKS.items() if not v["pass"]]
assert not failed, f"numerical checks failed: {failed}"

## 6. Benchmark: JAX backend, worker count and run time

Timings run in fresh processes with the settings of a worker. First, the JAX part of a refinement step (indicator,
gradient and volume) and the sparse solve; then one complete planar and one complete shell design per backend,
which also give the peak memory of a worker. With `n_g` GPU workers and `n_c` CPU workers, the throughput in
refinement steps per second is modelled as `min(n_g / (t_solve + t_gpu), n_gpus / t_gpu) + n_c / (t_solve + t_cpu)`
(each GPU is time-shared between processes). The split that maximizes it within the CPU cores and the memory of
the machine is used.

In [ ]:
def run_bench(args, backend, timeout=7200):
    env = dict(os.environ, JAX_PLATFORMS=backend, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",
               OPENBLAS_NUM_THREADS="1", XLA_PYTHON_CLIENT_PREALLOCATE="false",
               XLA_PYTHON_CLIENT_ALLOCATOR="platform")
    if backend == "cpu":
        env["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
    try:
        r = subprocess.run([sys.executable, "-m", "trimfields.bench", *map(str, args)], cwd=WORK, env=env,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"benchmark {args} on {backend}: no result within {timeout} s")
        return None
    res = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
    if r.returncode != 0 or not res:
        print(f"benchmark {args} on {backend} failed:\n", r.stdout[-1500:], r.stderr[-3000:])
        return None
    return json.loads(res[-1][7:])


N_EL = cfg["n_el_planar"] if MODE == "full" else 900
BACKENDS = ["cpu"] + (["cuda"] if JAX_GPU_OK else [])
BENCH = {b: run_bench(["indicator", N_EL, 5 if b == "cpu" else 20], b) for b in BACKENDS}
for b, r in BENCH.items():
    if r:
        print(f"{b:5s}: JAX step {r['t_indicator']*1e3:8.1f} ms | solve {r['t_solve']*1e3:6.1f} ms "
              f"(PARDISO {r['pardiso']}) | compile {r['t_compile']:.1f} s")
assert BENCH["cpu"], "the CPU benchmark failed"
if JAX_GPU_OK and not BENCH.get("cuda"):
    BACKENDS = ["cpu"]

In [ ]:
# Complete designs (all stages, including resampled attempts): wall time and peak memory per backend.
bench_dir = os.path.join(OUT_DIR, "_bench")
shutil.rmtree(bench_dir, ignore_errors=True)
small = [] if MODE == "full" else ["small"]
DESIGN_T, DESIGN_MEM = {}, {}
for backend in BACKENDS:
    for kind, (split, fam) in {"planar": ("id", "plate"), "shell": ("shell", "cyl")}.items():
        r = run_bench(["design", split, fam, 10, os.path.join(bench_dir, backend)] + small, backend)   # index 10: a test design, whose finer evaluation sets the memory peak
        if r is None or r["status"] != "ok":
            print(f"design benchmark {kind} on {backend} failed: {r}")
            continue
        DESIGN_T[(backend, kind)] = r["wall"]
        DESIGN_MEM[backend] = max(DESIGN_MEM.get(backend, 0.0), r["max_rss_gb"])
        print(f"{kind:6s} design on {backend:4s}: {r['wall']:6.1f} s, peak memory {r['max_rss_gb']:.2f} GB "
              f"(attempt {r['attempt']}, {r['nel']} elements; " + ", ".join(f"{k} {v:.1f}" for k, v in
                                                                          r["times"].items()) + " s)")
shutil.rmtree(bench_dir, ignore_errors=True)
if "cuda" in BACKENDS and not all(("cuda", k) in DESIGN_T for k in ("planar", "shell")):
    print("the GPU design benchmark failed: all workers run on the CPU")
    BACKENDS = ["cpu"]
assert all(("cpu", k) in DESIGN_T for k in ("planar", "shell")), "the CPU design benchmark failed"

In [ ]:
t_cpu, t_solve = BENCH["cpu"]["t_indicator"], BENCH["cpu"]["t_solve"]
t_rest = t_solve + 0.02                                       # solve plus the remaining NumPy work of a step
MEM = mem_gb() - 2.0                                          # keep 2 GB for the system and the notebook
m_cpu = 1.25 * DESIGN_MEM.get("cpu", BENCH["cpu"]["max_rss_gb"]) + 0.2
m_gpu = 1.25 * DESIGN_MEM.get("cuda", 2 * m_cpu) + 0.2
vis = os.environ.get("CUDA_VISIBLE_DEVICES")
VISIBLE = [d.strip() for d in vis.split(",") if d.strip()] if vis is not None else [str(i) for i in range(len(GPUS))]
N_GPUS = len(VISIBLE) if "cuda" in BACKENDS else 0           # GPUs that the workers may use
if N_GPUS > 1:                                                # GPU workers are spread over the visible GPUs
    os.environ["TRIMFIELDS_GPU_DEVICES"] = ",".join(VISIBLE)
gpu_mem = min((float(g.split(",")[1].split()[0]) / 1024 for g in GPUS), default=0.0)   # GB per GPU
max_gpu_workers = N_GPUS * int(gpu_mem / 0.8)                 # about 0.8 GB of GPU memory per worker


def total_workers(n_g):
    if WORKERS != "auto":
        return int(WORKERS)
    return max(1, min(CPUS - 1, n_g + int(max(0.0, MEM - n_g * m_gpu) / m_cpu)))


def throughput(n_g):
    n = total_workers(n_g)
    gpu = min(n_g / (t_rest + BENCH["cuda"]["t_indicator"]), N_GPUS / BENCH["cuda"]["t_indicator"]) if n_g else 0.0
    return gpu + (n - n_g) / (t_rest + t_cpu)


if N_GPUS:
    if GPU_WORKERS == "auto":
        cands = [n for n in range(0, min(CPUS - 1, max_gpu_workers) + 1) if total_workers(n) >= n
                 and n * m_gpu <= max(MEM, m_gpu)]
        N_GPU = max(cands, key=lambda n: (round(throughput(n), 6), -n))
    else:
        N_GPU = int(GPU_WORKERS)
    N_TOTAL = max(total_workers(N_GPU), N_GPU)
    print(f"refinement steps per second: all workers on the CPU {throughput(0):.1f}, chosen split "
          f"{throughput(N_GPU):.1f}")
else:
    if GPU_WORKERS not in ("auto", "0"):
        print("GPU workers requested, but JAX cannot use the GPU: all workers run on the CPU")
    N_GPU, N_TOTAL = 0, total_workers(0)
print(f"workers: {N_TOTAL} ({N_GPU} with the JAX part on the GPU); memory per worker {m_cpu:.1f} GB (CPU), "
      f"{m_gpu:.1f} GB (GPU); {MEM:.0f} GB available")


def rate(kind):                                               # designs per second
    r = 0.0
    if ("cuda", kind) in DESIGN_T:
        r += N_GPU / DESIGN_T[("cuda", kind)]
    if ("cpu", kind) in DESIGN_T:
        r += (N_TOTAL - N_GPU) / DESIGN_T[("cpu", kind)]
    return r


if MODE == "full" and rate("planar") > 0 and rate("shell") > 0:
    eta_h = (N_PLANAR / rate("planar") + N_SHELL / rate("shell")) / 3600 * 1.15    # 15 % for tail and variation
    print(f"estimated time for the full dataset: {eta_h:.0f} h ({eta_h / 24:.1f} days); designs differ in "
          "size, so treat this as a rough figure (the progress bar gives the running estimate)")
elif MODE == "smoke":
    print("smoke mode: timings use small meshes, so no run-time estimate for the full dataset is given")

## 7. Smoke run

One design per family with the production settings (smaller meshes in `smoke` mode), consolidated like the full
dataset. It checks the worker pool, the GPU assignment, STEP export and consolidation on this machine.

In [ ]:
smoke = P.smoke_config(cfg)
if MODE == "smoke":
    smoke["n_el_planar"], smoke["n_el_shell"] = 900, 900
t0 = time.monotonic()
res = P.run_all(smoke, P.task_list(smoke), N_TOTAL, n_gpu_workers=N_GPU)
bad = [r for r in res if r["status"] != "ok"]
print(f"smoke run: {len(res)} designs in {time.monotonic() - t0:.0f} s, {len(bad)} not ok")
for r in bad:
    print("  ", r)
assert not bad, "the smoke run failed; see the messages above and <out>/_smoke/failures"
SMOKE_SUMMARY = P.consolidate(smoke, min(N_TOTAL, 8))
print(json.dumps(SMOKE_SUMMARY, indent=1))

## 8. Full generation (resumable)

All designs are generated by the worker pool, longest (shell) designs first. Designs whose problems keep failing
are retried in later rounds with new seeds (the seed cursor is kept in `<out>/rounds.json`, so a rerun continues
with unused seeds). Progress is also written to `<out>/progress.log`; a design that exceeds `task_timeout` is
stopped by its watchdog and retried in the next round. The notebook stops here, before consolidation and
packaging, if designs are still missing; running it again resumes the generation.

In [ ]:
if MODE == "full":
    t0 = time.monotonic()
    ROUNDS = P.generate_until_complete(cfg, N_TOTAL, rounds=10, n_gpu_workers=N_GPU)
    missing = [t for t in TASKS if not P.is_done(cfg, t)]
    print(f"generation finished in {(time.monotonic() - t0) / 3600:.1f} h; missing designs: {len(missing)}")
    if missing:
        raise RuntimeError(f"{len(missing)} designs are still missing, e.g. {missing[:5]}. Run the notebook again: "
                           f"it resumes with new seeds for them. Reasons are in {OUT_DIR}/failures.")
else:
    ROUNDS = []
    print("smoke mode: the full generation is skipped")

## 9. Consolidation

In [ ]:
TARGET = cfg if MODE == "full" else smoke
t0 = time.monotonic()
SUMMARY = P.consolidate(TARGET, N_TOTAL)
print(f"consolidated in {time.monotonic() - t0:.0f} s")
print(json.dumps(SUMMARY, indent=1))

## 10. Quality report and gallery

In [ ]:
import pandas as pd

cdir = os.path.join(TARGET["out_dir"], "consolidated")
frames = [pd.read_csv(os.path.join(cdir, f"{g}_index.csv")) for g in ("planar", "shell")
          if os.path.exists(os.path.join(cdir, f"{g}_index.csv"))]
IDX = pd.concat(frames, ignore_index=True)
IDX["J_final_over_thr"] = IDX["J_final"] / IDX["J_thr"]            # NaN for random designs
IDX["J_final_over_simp"] = IDX["J_final"] / IDX["J_simp"]
IDX["J_f2_over_fine"] = IDX["J_final"] / IDX["J_fine"]              # production against finer evaluation (test)
QUALITY = IDX.groupby(["kind", "family"]).agg(
    designs=("row", "size"), resampled=("attempt", lambda a: float((a > 0).mean())),
    refined_final=("final_source", lambda s: float((s == "refine").mean())),
    J_final_over_thr=("J_final_over_thr", "median"), J_final_over_simp=("J_final_over_simp", "median"),
    J_f2_over_fine=("J_f2_over_fine", "median"), vm_max_loops=("vm_max_loops", "median"),
    loops=("loops_final", "median"), V=("V_final", "median"), valid=("valid", "mean"),
    step_valid=("step_valid", "mean"), t_median_s=("t_total", "median"))
QUALITY.to_csv(os.path.join(cdir, "quality.csv"))
pd.set_option("display.precision", 3)
display(QUALITY)
n_fail = len(os.listdir(os.path.join(TARGET["out_dir"], "failures"))) \
    if os.path.isdir(os.path.join(TARGET["out_dir"], "failures")) else 0
print(f"{len(IDX)} designs, {n_fail} failure records, all valid: {bool(IDX['valid'].all())}, "
      f"all STEP valid: {bool(IDX['step_valid'].all())}")

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.path import Path as MPath

from trimfields.hosts import Host
from trimfields.loops import sample_loop


def material(prob, loops, n):
    host = Host.from_dict(prob["host"]).build(8, 8)
    c = np.linspace(0, 1, n)
    U, V = np.meshgrid(c, c, indexing="ij")
    uv = np.stack([U.ravel(), V.ravel()], 1)
    X = host.eval(uv[:, 0], uv[:, 1])
    mat = np.ones(len(uv), bool)
    for C in loops:
        mat &= ~MPath(sample_loop(np.asarray(C), 20)).contains_points(uv)
    for ps in prob["passive"]:
        if ps["kind"] == "circle":
            mat &= np.hypot(X[:, 0] - ps["centre"][0], X[:, 1] - ps["centre"][1]) >= ps["radius"]
        else:
            mat &= ~MPath(np.asarray(ps["uv"])).contains_points(uv)
    return host, X.reshape(n, n, -1), mat.reshape(n, n)


def draw(ax, path, n=120):
    d = np.load(path)
    prob, loops = json.loads(str(d["problem"])), d["loops_final"]
    host, X, mat = material(prob, loops, n)
    face = np.where(mat, 1.0, np.nan)
    if X.shape[2] == 2:
        ax.pcolormesh(X[..., 0], X[..., 1], face[:-1, :-1], cmap="Greys", vmin=0, vmax=1.4, shading="flat")
        for C in loops:
            s = sample_loop(np.asarray(C), 30)
            s = s[np.all((s >= 0) & (s <= 1), 1)]
            if len(s):
                P_ = host.eval(s[:, 0], s[:, 1])
                ax.plot(P_[:, 0], P_[:, 1], ".", ms=0.6, color="#c0392b")
        ax.set_aspect("equal")
        ax.axis("off")
    else:
        fc = np.zeros(mat.shape + (4,))
        fc[mat] = (0.35, 0.45, 0.6, 1.0)
        ax.plot_surface(X[..., 0], X[..., 1], X[..., 2], facecolors=fc, rstride=1, cstride=1, linewidth=0,
                        antialiased=False, shade=False)
        ax.set_box_aspect(np.ptp(X.reshape(-1, 3), 0) + 1e-9)
        ax.axis("off")
    ax.set_title(f"{prob['split']} {prob['family']} {os.path.basename(path)[:-4]}", fontsize=11)


rng = np.random.default_rng(0)
rows = []
for kind in ("planar", "shell"):
    sub = IDX[IDX["path"].str.contains("shell") == (kind == "shell")]      # shell, shell_ood, rand_shell, rand_shell_ood
    fams = sorted(sub["family"].unique())
    n = 12 if kind == "planar" else 6
    picks, used = [], set()
    for i in range(n):                                         # cycle through the families, no repeats
        pool = sub[(sub["family"] == fams[i % len(fams)]) & ~sub["row"].isin(used)] if fams else sub.iloc[:0]
        if len(pool):
            row = pool.sample(1, random_state=int(rng.integers(1e9))).iloc[0]
            used.add(row["row"])
            picks.append(row)
    rows += [("planar", picks[:6]), ("planar", picks[6:])] if kind == "planar" else [("shell", picks)]
fig = plt.figure(figsize=(18, 9))
for r, (kind, picks) in enumerate(rows):
    for c, row in enumerate(picks):
        ax = fig.add_subplot(3, 6, 6 * r + c + 1, projection="3d" if kind == "shell" else None)
        draw(ax, os.path.join(TARGET["out_dir"], row["path"]))
fig.tight_layout()
GALLERY = os.path.join(cdir, "gallery.png")
fig.savefig(GALLERY, dpi=110)
plt.close(fig)
from IPython.display import Image, display

display(Image(GALLERY))

## 11. Manifest and archives

In [ ]:
import hashlib
import tarfile


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


root = pathlib.Path(TARGET["out_dir"])
MANIFEST = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "mode": MODE, "trimfields_version": trimfields.__version__,
            "config": TARGET, "machine": MACHINE, "benchmark": BENCH, "design_times": {f"{k[0]}/{k[1]}": v
                                                                                       for k, v in DESIGN_T.items()},
            "design_memory_gb": DESIGN_MEM,
            "workers": {"total": N_TOTAL, "gpu": N_GPU}, "checks": CHECKS, "rounds": ROUNDS, "summary": SUMMARY,
            "quality": json.loads(QUALITY.to_json(orient="index")),
            "sources": {f"trimfields/{m}.py": sha256(WORK / "trimfields" / f"{m}.py") for m in
                        [p.stem for p in sorted((WORK / "trimfields").glob("*.py"))]},
            "notebook_hours": (time.monotonic() - T_START) / 3600}
with open(root / "manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=1, default=str)
shutil.copytree(WORK / "trimfields", root / "trimfields_source", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__"))

arch = root.parent / (root.name + "_archives")
arch.mkdir(exist_ok=True)
PARTS = {"consolidated": ["consolidated", "manifest.json", "progress.log", "failures", "trimfields_source"],
         "designs": ["designs"], "step": ["step"]}
sums = []
for name, items in PARTS.items():
    out = arch / f"trimfields_{name}.tar.gz"
    t0 = time.monotonic()
    with tarfile.open(out, "w:gz", compresslevel=6) as tar:
        for it in items:
            if (root / it).exists():
                tar.add(root / it, arcname=f"{root.name}/{it}")
    sums.append(f"{sha256(out)}  {out.name}")
    print(f"{out.name}: {out.stat().st_size / 2**20:,.1f} MB in {time.monotonic() - t0:.0f} s")
(arch / "SHA256SUMS").write_text("\n".join(sums) + "\n")
print(f"archives in {arch}; total notebook time {(time.monotonic() - T_START) / 3600:.2f} h")